# Soft-collection segmentation: auto loans, DPD 1-30

**Method in 10 lines**
1. `ovrd` (monthly max DPD per contract) is turned into a DPD panel and cut into **delinquency episodes**. A new episode needs a cure in between, so partial payments (re-ageing) do not create fake entries.
2. Every episode is scored on **checkpoints** (days 1, 8, 15, 22, 29 in arrears) while it is still overdue. **Target** `bad`: the contract reaches 90+ DPD within 6 months after the checkpoint. Secondary targets: `self_cure30` and `roll31_60d`.
3. Every feature is computed **as of the checkpoint date**: only events dated strictly before it. Each data source is a plug-in *feature block* `fb_*(obs, ctx)`.
4. Sources that start late (geo, vnpt, vneshpt, PKB events) and extraction-time fields (`contr.end_date`) feed the rules and segmentation axes only, so the model does not learn time artefacts or leakage.
5. A **LightGBM PD model** with days in arrears as a feature, benchmarked against a logistic regression, is trained with a time split by episode. Out-of-time (OOT) = the episodes of the last 3 labelled entry months.
6. PD is cut into **risk grades G1-G5** at fixed cutpoints taken from the train PD of new entries (40/65/85/95% quantiles).
7. **Hard rules** take priority (deceased, bankruptcy, bankruptcy attempt, skip/abroad, FPD fraud check, gambling, collateral); **modifiers** (social exclusion lists, enforcement orders) add actions without replacing the risk segment.
8. Everything else is split by **risk x ability to pay (capacity) x willingness/reachability (engagement)** into six treatment segments, S1-S6.
9. Segments are **backtested** at every checkpoint with confidence intervals. The **current stock** (DPD 1-30 now) is scored on its **current day in arrears** and gets a segment, today's treatment and its compliance dates.
10. Outputs: parquet and Excel files, charts and the model artefacts, all in `out/` (DEMO runs: `out_demo/`).

**How to run** (pick one)
- **(a) Same kernel as your loading notebook.** Load the tables under their usual names (`contr`, `ovrd`, `ver_df_all_2024_2026`, `bankrupt`, `ocrm_call`, `vneshpt_ot`, `vnpt`, `excl`, `vneshpt`, `geo`, `dm`, `dm_rej`, optionally `hes_pkb_agg`), then run `%run -i /full/path/to/credit_segments/soft_collection_segmentation.ipynb`. Your tables are read and never modified, but **the run defines ~260 public globals with generic names** (`X`, `S`, `OB`, `CONFIG`, `log`, `psi`, `col`, `MODEL`, `fig`, `ax` ...) that overwrite your own variables of the same name; pandas display options and matplotlib settings are restored at the end. Set `CONFIG_OVERRIDES = {'MODE': 'real', ...}` in your kernel first: `MODE='real'` makes a missing table fail loudly.
- **(b) Parquet files.** Set `CONFIG['DATA_DIR']` and the file names in `CONFIG['FILES']` (globs allowed). Only the needed columns and the population's keys are read (keys are normalised before filtering, so padded CHAR keys still match).
- **(c) DEMO.** `MODE='demo'`, or `MODE='auto'` when **no** table is found at all, uses the synthetic generator `demo_data.py` (same schema, fake people). If only some tables are found, the run stops with the list of found / missing tables instead of silently switching to DEMO.

Only `contr` and `ovrd` are required. A missing optional source only switches its block off (logged in `run_summary.json`). Build the PKB aggregate (`sql/hes_pkb_monitoring_agg.sql`) **before** the run, or the PKB rules are off.

Written for Python 3.13/3.14 and pandas 3: `str` dtype, Copy-on-Write, no chained assignment.

## 1. Configuration
Every knob lives in one dict, `CONFIG`. Two ways to change a knob without editing this cell:
- define a dict `CONFIG_OVERRIDES` in your kernel before running the notebook (way (a)), e.g. `CONFIG_OVERRIDES = {'MODE': 'real', 'OUT_DIR': '/data/seg_out'}`. Nested dicts (`FILES`, `COMPLIANCE`) are merged key by key;
- `SEG_*` environment variables (used by the test runner): `SEG_MODE`, `SEG_DROP_TABLES`, `SEG_OUT_DIR`, `SEG_DATA_DIR`, `SEG_DEMO_N_CONTRACTS`, `SEG_USE_LIGHTGBM`, `SEG_STRICT_BLOCKS`, `SEG_N_JOBS`.

Relative `DATA_DIR` / `OUT_DIR` paths are resolved against the **notebook's folder** (also under `%run -i`), not the kernel's working directory.

In [ ]:
import os

CONFIG = {
    # ---- where the data comes from ---------------------------------------------------------------
    'MODE': 'auto',            # 'auto': globals() -> parquet -> DEMO only if NO table is found | 'real' | 'demo'
    'DATA_DIR': 'data',        # folder with parquet extracts (way (b)), relative to this notebook's folder
    'FILES': {                 # table -> parquet file name, glob or list  (PLACEHOLDERS: edit to your names)
        'contr': 'contr_auto042024_032026.parquet',
        'ovrd': 'ovrd.parquet',
        'ver_df_all_2024_2026': 'ver_df_all_2024_2026.parquet',
        'bankrupt': 'bankrupt.parquet',
        'ocrm_call': 'ocrm_call.parquet',
        'vneshpt_ot': 'vneshpt_ot.parquet',
        'vnpt': 'vnpt.parquet',
        'excl': 'excl.parquet',
        'vneshpt': 'vneshpt.parquet',
        'geo': 'geo.parquet',
        'dm': 'dm.parquet',
        'dm_rej': 'dm_*_rejected.parquet',      # e.g. one file per year
    },
    'PKB_AGG_FILE': 'hes_pkb_agg.parquet',      # optional: output of sql/hes_pkb_monitoring_agg.sql (in DATA_DIR)
    'DROP_TABLES': [],         # switch sources off (also used by the robustness test)
    'DEMO_N_CONTRACTS': 8000,  # synthetic portfolio size in DEMO mode (8000: ~750 OOT episodes, ~25 s, ~2 GB RAM)
    'STRICT_BLOCKS': False,    # True: a failing optional block raises; False: warn and skip it (logged in run_summary)
    'RUN_PKB_SQL': False,      # True: build hes_pkb_agg with your query() helper BEFORE loading (section 3)
    # ---- episodes, checkpoints and target --------------------------------------------------------
    'SOFT_DPD_MIN': 1, 'SOFT_DPD_MAX': 30,       # soft-collection window (current stock)
    'BAD_DPD': 90, 'HORIZON_MONTHS': 6,          # bad = reaches 90+ DPD within 6 months after obs_date
    'ROLL_DPD': 31, 'ROLL_DAYS': 60,             # fast proxy target: 31+ DPD within 60 days
    'CHECKPOINT_DAYS': [1, 8, 15, 22, 29],       # every episode is (re)scored on these days in arrears while it
                                                 # is STILL overdue (obs_date = entry + day - 1); the stock is
                                                 # scored on its current day in arrears. [1] = entry-only model;
                                                 # [6] = 'score on day 6' (replaces the old OBS_OFFSET_DAYS knob)
    'EPISODE_GAP_DAYS': 5,     # a new episode needs a cure: start_date > previous rows' last act_date + gap
    'STOCK_ONLY_AT_CUT': True, # current stock = rows still overdue on the data cut (act_date == cut)
    'PROFILE_LAG_DAYS': 0,     # ver_df snapshot must be dated < obs_date - lag (raise if snapshots land late)
    'PROFILE_MONTH_END_ONLY': 'auto',  # model features from calendar month-end snapshots only (ver_df switched
                                       # to daily snapshots in 2026); 'auto' = if >= 6 month-end dates exist
    # ---- model -----------------------------------------------------------------------------------
    'OOT_MONTHS': 3, 'VALID_SHARE': 0.2,
    'GRADE_QUANTILES': [0.40, 0.65, 0.85, 0.95],  # train-PD quantiles (first checkpoint) -> cutpoints G1..G5
    'RANDOM_STATE': 42,
    'N_JOBS': max(1, (os.cpu_count() or 2) // 2),  # threads for LightGBM / KMeans (set to the cores really free)
    'MODEL_BLOCKS': 'auto',    # 'auto' (rule in section 6) or an explicit list of block names
    'MODEL_EXCLUDE': ['pr_snapshot_age_days',     # time artefact (snapshot cadence changes over time)
                      # everything built from contr.end_date (extraction-time value, moves on restructuring):
                      'ct_term_months', 'ct_remaining_months', 'ct_installment_proxy',
                      'capacity_ratio', 'depo_cover', 'debitable_cover'],   # -> segmentation axes / overlay only
    'PSI_DROP_THRESHOLD': 0.25, 'MAX_NULL_SHARE': 0.98,
    'TOP_N_CATEGORIES': 15,
    'USE_LIGHTGBM': True,      # False or package missing -> sklearn HistGradientBoosting
    'LR_TOP_N': 25,
    'DM_ALLOW_COLUMNS': None,  # None = auto-profile dm text columns (deny-list below; printed for sign-off);
                               # a list = ONLY these dm columns (recommended on real data after review)
    'DM_DENY_PATTERN': (r'status|state|sost|stage|result|dpd|prosroch|overdue|close|zakr|modif|updat|change|'
                        r'end_?date|restruct|spis|write|balance|ostat|debt|zadol|current|actual|decision_date'),
    # ---- segmentation ----------------------------------------------------------------------------
    'KZ_BBOX': (40.5, 55.5, 46.4, 87.4),         # lat_min, lat_max, lon_min, lon_max
    'CAPACITY_RATIO_MIN': 1.5, 'DEPO_COVER_MIN': 1.0, 'CONTACT_RATE_ENGAGED': 0.5,
    'INSTALMENT_RATE': 0.18,   # annual rate for the annuity instalment proxy when MONTH_LOAN_PAY_AMOUNT is missing
    'FPD_MIN_DAYS': 8,         # R07: 1st/2nd instalment still unpaid after this many days in arrears
    'ENFORCEMENT_OPERTYPES': ['ИР3', 'ИР4'],      # R05 modifier: open orders of these types ...
    'ENFORCEMENT_MAX_AGE_DAYS': 730,              # ... accepted within this many days
    'ABROAD_NOTE_DAYS': 90,    # R06: CRM 'abroad' notes count only if younger than this
    'CHALLENGER_SHARE': 0.15,  # share of every segment routed to the challenger treatment (contract hash)
    'R02_PKB_COURT': False,    # True: PKB 'court' status (may concern ANOTHER creditor's loan) also stops contact
                               # (R02); False: court-only cases get legal_review_flag and keep their segment
    'CLUSTER_K_RANGE': (4, 8), 'CLUSTER_SAMPLE': 20000,
    # ---- regulatory constants (research memo, 2026; Legal to confirm items marked [verify]) --------
    'COMPLIANCE': {
        'NOTIFY_DAYS': 10, 'RESTRUCT_APPLY_DAYS': 30, 'RESTRUCT_REVIEW_DAYS': 15,
        'CALLS': {'weekday': (8, 21, 3), 'weekend': (9, 19, 2)},   # (from hour, to hour, max calls per day)
        'VISITS': (1, 3),                                          # max per day, per week (weekdays only)
        'PM_2026': 50851, 'DEBIT_MAX_SHARE': 0.5, 'DEBIT_PROTECTED': 2 * 50851,
        'MRP_2026': 4325, 'OOC_BANKRUPTCY_MAX': 1600 * 4325,
        'NO_NEW_LOAN_DPD': 30, 'COLLECTOR_HANDOVER_DPD': 90, 'ASSIGNMENT_BAN_UNTIL': '2027-05-01',
    },
    # ---- outputs ---------------------------------------------------------------------------------
    'OUT_DIR': 'out',          # DEMO runs write to OUT_DIR + '_demo' (e.g. out_demo/) unless OUT_DIR is overridden
}

# (1) CONFIG_OVERRIDES defined in your kernel before the run (way (a)); nested dicts are merged
_ovr = globals().get('CONFIG_OVERRIDES') or {}
for _k, _v in dict(_ovr).items():
    if _k not in CONFIG:
        print(f'WARN  CONFIG_OVERRIDES: unknown key {_k!r} ignored')
    elif isinstance(CONFIG[_k], dict) and isinstance(_v, dict):
        CONFIG[_k] = {**CONFIG[_k], **_v}
    else:
        CONFIG[_k] = _v
_OUT_DIR_SET = 'OUT_DIR' in _ovr

# (2) environment overrides (used by the test runner; harmless otherwise)
_env = os.environ
if _env.get('SEG_MODE'):
    CONFIG['MODE'] = _env['SEG_MODE'].strip().lower()
if _env.get('SEG_DROP_TABLES'):
    CONFIG['DROP_TABLES'] = [t.strip() for t in _env['SEG_DROP_TABLES'].split(',') if t.strip()]
for _k in ('OUT_DIR', 'DATA_DIR'):
    if _env.get(f'SEG_{_k}'):
        CONFIG[_k] = _env[f'SEG_{_k}']
        _OUT_DIR_SET = _OUT_DIR_SET or _k == 'OUT_DIR'
for _k in ('DEMO_N_CONTRACTS', 'N_JOBS'):
    if _env.get(f'SEG_{_k}'):
        CONFIG[_k] = int(_env[f'SEG_{_k}'])
for _k in ('USE_LIGHTGBM', 'STRICT_BLOCKS'):
    if _env.get(f'SEG_{_k}'):
        CONFIG[_k] = _env[f'SEG_{_k}'].strip().lower() in ('1', 'true', 'yes', 'y')
assert CONFIG['MODE'] in ('auto', 'demo', 'real'), CONFIG['MODE']
CONFIG['CHECKPOINT_DAYS'] = sorted({int(k) for k in CONFIG['CHECKPOINT_DAYS'] if int(k) >= 1})
assert CONFIG['CHECKPOINT_DAYS'], 'CHECKPOINT_DAYS must hold at least one day >= 1'
print({k: CONFIG[k] for k in ('MODE', 'DATA_DIR', 'OUT_DIR', 'DROP_TABLES', 'USE_LIGHTGBM', 'STRICT_BLOCKS',
                              'CHECKPOINT_DAYS', 'N_JOBS')})

## 2. Imports and helpers
The helpers are safe under pandas 3 (`str` dtype with NaN as missing, Copy-on-Write):
- `norm_iin`, `norm_contract`, `norm_ref` normalise keys. They run on the unique values only, so they are fast on tables with 10^7 rows. IINs keep or regain their leading zeros (`zfill(12)`).
- `to_dt` returns `datetime64[us]` and turns the 9999-12-31 "open" sentinel into NaT. It never casts to `[ns]`, where that date overflows.
- `flag01` reads string flags (`'1'`, `'1.0'`, `'Y'`, `'Да'`, `'TRUE'` ...) as 0/1. Section 3 reports the parse rate of every flag column, so an unknown encoding is visible.
- `load_table` finds each table in this priority order: (1) a DataFrame in `globals()`, (2) parquet with column projection and a key push-down filter, (3) DEMO. Population keys are compared **after normalisation** in both modes (CHAR padding, `.0` artefacts, lost leading zeros); rows that match only after normalisation are reported.
- `MODE='auto'` falls back to DEMO only when none of the known tables exists. DEMO outputs go to `out_demo/` and carry `run_mode='demo'`.

In [ ]:
import glob, hashlib, json, math, pickle, platform, re, sys, time, traceback
from contextlib import contextmanager, nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import sklearn
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, silhouette_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from IPython.display import Markdown, display

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:          # not installed or a broken wheel -> HistGradientBoosting fallback
    lgb, HAS_LGB = None, False

# display options for this run; your kernel's previous values are restored in the last cell
_PD_OPTIONS = {'display.width': 220, 'display.max_columns': 40, 'display.max_colwidth': 70,
               'display.float_format': lambda x: f'{x:,.3f}'}
_PD_OPTIONS_BEFORE = {k: pd.get_option(k) for k in _PD_OPTIONS}
for _k, _v in _PD_OPTIONS.items():
    pd.set_option(_k, _v)


def _notebook_dir():
    """Folder of this notebook: VS Code / `%run -i x.ipynb` set the path; plain Jupyter runs in that folder."""
    for _k in ('__vsc_ipynb_file__', '__file__'):
        _f = globals().get(_k)
        if isinstance(_f, str) and _f.lower().endswith('.ipynb') and Path(_f).exists():
            return Path(_f).resolve().parent
    return Path.cwd().resolve()


NB_DIR = _notebook_dir()


def nb_path(p):
    """Relative paths are taken relative to the notebook folder (not the kernel's working directory)."""
    p = Path(p).expanduser()
    return p if p.is_absolute() else NB_DIR / p


print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scikit-learn {sklearn.__version__} | lightgbm {lgb.__version__ if HAS_LGB else 'n/a'} | "
      f"matplotlib {matplotlib.__version__}")
if not pd.__version__.startswith('3'):
    print('NOTE: written for pandas 3.x (str dtype, Copy-on-Write).')
print(f'notebook folder: {NB_DIR}')

In [ ]:
T0 = time.perf_counter()


def log(msg):
    print(f'[{time.perf_counter() - T0:6.1f}s] {msg}', flush=True)


@contextmanager
def timer(name):
    t = time.perf_counter()
    yield
    log(f'{name}: {time.perf_counter() - t:.1f}s')


def show(df, title=None):
    """Display a compact table under a bold title."""
    if title:
        display(Markdown(f'**{title}**'))
    display(df)


def _via_uniques(s, fn):
    """Apply a vectorised string normaliser to the unique values only (fast on 10^7-row columns)."""
    s = pd.Series(s)
    codes, uniq = pd.factorize(s, use_na_sentinel=True)
    out = np.full(len(s), np.nan, dtype=object)
    if len(uniq):
        normed = fn(pd.Series(np.asarray(uniq, dtype=object), dtype='str'))
        normed = normed.to_numpy(dtype=object, na_value=np.nan)
        ok = codes >= 0
        out[ok] = normed[codes[ok]]
    return pd.Series(out, index=s.index, dtype='str')


def _clean_digits(u):
    return u.str.strip().str.replace(r'\.0+$', '', regex=True).str.replace(r'\D', '', regex=True)


def norm_iin(s):
    """IIN/BIN -> 12-digit string with leading zeros restored; anything else -> NaN."""
    def f(u):
        x = _clean_digits(u)
        x = x.where(x.str.len().between(1, 12)).str.zfill(12)
        return x.where(x != '000000000000')
    return _via_uniques(s, f)


def norm_contract(s):
    """Contract / account number -> digits-only string ('.0' float artefacts removed)."""
    def f(u):
        x = _clean_digits(u)
        return x.where(x.str.len() > 0)
    return _via_uniques(s, f)


def norm_ref(s):
    """Process reference (e.g. DGA000000000001) -> stripped upper-case string."""
    def f(u):
        x = u.str.strip().str.upper()
        return x.where(x.str.len() > 0)
    return _via_uniques(s, f)


FAR_FUTURE = pd.Timestamp('2100-01-01')
_NA_TXT = ['', 'NaT', 'nan', 'NaN', 'None', 'NULL', 'null']


def to_dt(s, max_ts=FAR_FUTURE):
    """Any date-like column -> datetime64[us]. Strings: ISO first, then mixed / day-first.
    Sentinels >= 2100 (9999-12-31 = 'still open') -> NaT. Never casts to [ns] (9999-12-31 overflows ns)."""
    s = pd.Series(s)
    if isinstance(s.dtype, pd.DatetimeTZDtype):
        s = s.dt.tz_localize(None)
    if pd.api.types.is_datetime64_dtype(s):
        out = s
    else:
        txt = s.astype('str').str.strip()
        out = pd.to_datetime(txt, errors='coerce', format='ISO8601')
        miss = out.isna() & txt.notna() & ~txt.isin(_NA_TXT)
        if miss.any():
            alt = pd.to_datetime(txt[miss], errors='coerce', format='mixed', dayfirst=True)
            out = out.where(out < max_ts).astype('datetime64[us]').fillna(alt.where(alt < max_ts).astype('datetime64[us]'))
    out = out.where(out < max_ts)
    return out.astype('datetime64[us]')


def coalesce(*series):
    out = series[0]
    for s in series[1:]:
        out = out.fillna(s)
    return out


def text_cols(df):
    """Text-like columns: object, str (pandas 3 default), string[pyarrow] or category."""
    return [c for c in df.columns
            if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
            or isinstance(df[c].dtype, pd.CategoricalDtype)]


_TRUE = {'1', '1.0', 'Y', 'YES', 'ДА', 'TRUE', 'T'}
_FALSE = {'0', '0.0', 'N', 'NO', 'НЕТ', 'FALSE', 'F'}


def flag01(s):
    """Robust 0/1 parser for string flags ('1', '1.0', 'Y', 'Да', 'TRUE' -> 1; '0', 'N', 'Нет' -> 0; else NaN)."""
    s = pd.Series(s)
    if pd.api.types.is_bool_dtype(s):
        return s.astype('float32')
    if pd.api.types.is_numeric_dtype(s):
        x = s.astype('float64')
        return x.where(x.isin([0.0, 1.0])).astype('float32')
    u = s.astype('str').str.strip().str.upper()
    out = np.where(u.isin(_TRUE), 1.0, np.where(u.isin(_FALSE), 0.0, np.nan))
    return pd.Series(out, index=s.index, dtype='float32')


def to_float32(df):
    """Memory-light downcast of numeric (and bool) columns to float32."""
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return df.astype({c: 'float32' for c in num}) if num else df


def col(df, name, default=np.nan):
    """df[name] if present, else a constant float Series (a source or block may be missing)."""
    return df[name] if name in df.columns else pd.Series(default, index=df.index, dtype='float64')


def as_cat(s, upper=True):
    """Text category: str, stripped (upper-cased), '.0' artefacts removed; NaN stays NaN."""
    x = pd.Series(s).astype('str').str.strip()
    if upper:
        x = x.str.upper()
    x = x.str.replace(r'\.0$', '', regex=True)
    return x.where(~x.isin(_NA_TXT + ['NAN', 'NONE', 'NAT']))


def psi(expected, actual, bins=10, eps=1e-4):
    """Population stability index of `actual` vs `expected` (numeric: expected-decile bins, text:
    categories). Missing values form their own bin."""
    e, a = pd.Series(expected), pd.Series(actual)
    if len(e) == 0 or len(a) == 0:
        return np.nan
    if pd.api.types.is_numeric_dtype(e) and pd.api.types.is_numeric_dtype(a):
        ev, av = e.to_numpy(dtype='float64'), a.to_numpy(dtype='float64')
        fin = ev[np.isfinite(ev)]
        edges = np.unique(np.quantile(fin, np.linspace(0, 1, bins + 1)[1:-1])) if len(fin) else np.array([])

        def dist(v):
            idx = np.where(np.isnan(v), -1, np.searchsorted(edges, v, side='right'))
            return pd.Series(idx).value_counts(normalize=True)
        pe, pa = dist(ev), dist(av)
    else:
        pe = e.astype('str').fillna('<NA>').value_counts(normalize=True)
        pa = a.astype('str').fillna('<NA>').value_counts(normalize=True)
    keys = pe.index.union(pa.index)
    pe = pe.reindex(keys, fill_value=0).to_numpy() + eps
    pa = pa.reindex(keys, fill_value=0).to_numpy() + eps
    return float(np.sum((pa - pe) * np.log(pa / pe)))


def perf(y, p):
    """n, bad rate, AUC, Gini, KS."""
    y, p = np.asarray(y, dtype='float64'), np.asarray(p, dtype='float64')
    out = dict(n=len(y), bad_rate=float(y.mean()) if len(y) else np.nan, auc=np.nan, gini=np.nan, ks=np.nan)
    if len(np.unique(y)) == 2:
        auc = roc_auc_score(y, p)
        fpr, tpr, _ = roc_curve(y, p)
        out.update(auc=auc, gini=2 * auc - 1, ks=float(np.max(tpr - fpr)))
    return out


def wilson(k, n, z=1.96):
    """Wilson 95% confidence interval for a proportion k/n."""
    k, n = np.asarray(k, dtype='float64'), np.asarray(n, dtype='float64')
    with np.errstate(divide='ignore', invalid='ignore'):
        p = k / n
        den = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / den
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return np.clip(c - h, 0, 1), np.clip(c + h, 0, 1)


def prop_test(k1, n1, k2, n2):
    """Two-sided p-value of a two-proportion z-test (k1/n1 vs k2/n2); NaN when a sample is empty."""
    if not (n1 > 0 and n2 > 0):
        return np.nan
    p = (k1 + k2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    return 0.0 if se == 0 and k1 / n1 != k2 / n2 else (1.0 if se == 0 else math.erfc(abs(k1 / n1 - k2 / n2) / se / math.sqrt(2)))


def hash_share(keys, salt='soft-collection-cc'):
    """Deterministic pseudo-random number in [0, 1) per key (champion/challenger routing)."""
    return np.array([int(hashlib.md5(f'{salt}|{k}'.encode()).hexdigest()[:12], 16) / 16 ** 12 for k in keys])

In [ ]:
REQUIRED_TABLES = ('contr', 'ovrd')
KNOWN_TABLES = list(dict.fromkeys(list(CONFIG['FILES']) + ['hes_pkb_agg']))
LOAD_LOG = []
_DEMO = {}


def _demo_tables():
    """Synthetic tables with the bank's schema, from demo_data.py next to this notebook."""
    if 'tables' not in _DEMO:
        for here in (str(NB_DIR), str(Path.cwd())):
            if here not in sys.path:
                sys.path.insert(0, here)
        try:
            from demo_data import make_demo_tables
        except ImportError as e:
            raise ImportError(f'DEMO mode needs demo_data.py in the notebook folder ({NB_DIR})') from e
        with timer(f"DEMO data generated (n_contracts={CONFIG['DEMO_N_CONTRACTS']})"):
            _DEMO['tables'] = make_demo_tables(n_contracts=CONFIG['DEMO_N_CONTRACTS'], seed=CONFIG['RANDOM_STATE'])
    return _DEMO['tables']


def _parquet_paths(name):
    spec = CONFIG['PKB_AGG_FILE'] if name == 'hes_pkb_agg' else CONFIG['FILES'].get(name)
    if not spec:
        return []
    paths = []
    for sp in ([spec] if isinstance(spec, (str, Path)) else list(spec)):
        p = Path(sp) if Path(sp).is_absolute() else nb_path(CONFIG['DATA_DIR']) / sp
        if any(ch in str(p) for ch in '*?['):
            paths += [Path(x) for x in sorted(glob.glob(str(p)))]
        elif p.exists():
            paths.append(p)
    return paths


def _user_df(name):
    obj = globals().get(name)
    return obj if isinstance(obj, pd.DataFrame) else None


def _where(name):
    return 'globals' if _user_df(name) is not None else ('parquet' if _parquet_paths(name) else None)


def resolve_mode():
    """'real' if the required tables are in globals() or on disk. MODE='auto' falls back to 'demo' ONLY when
    none of the known tables is found; a partial set (e.g. ovrd loaded under another name) is an error."""
    if CONFIG['MODE'] == 'demo':
        return 'demo'
    found = {t: _where(t) for t in KNOWN_TABLES}
    missing = [t for t in REQUIRED_TABLES if not found[t]]
    if not missing:
        return 'real'
    present = {t: v for t, v in found.items() if v}
    if CONFIG['MODE'] == 'real' or present:
        raise RuntimeError(
            f"required table(s) {missing} not found in globals() nor in DATA_DIR={str(nb_path(CONFIG['DATA_DIR']))!r}; "
            f"found: {present or 'nothing'}. Name the DataFrames {list(REQUIRED_TABLES)} (or set CONFIG['FILES']), "
            "or set MODE='demo' for the synthetic data.")
    print('NOTE: no table found in globals() or DATA_DIR -> MODE=auto falls back to the synthetic DEMO data. '
          "Set MODE='real' on real data so that a missing table fails loudly.")
    return 'demo'


def _key_variants(keys, kind):
    ks = {k for k in keys if isinstance(k, str) and k}
    out = set(ks)
    if kind == 'iin':
        out |= {k.lstrip('0') for k in ks}        # IINs that lost their leading zeros
    return out | {k + '.0' for k in out}         # float artefacts ('900101300001.0')


def _numeric_keys(keys):
    v = pd.to_numeric(pd.Series(sorted(k for k in keys if isinstance(k, str) and k), dtype='str'), errors='coerce')
    return v.dropna().unique().tolist()


_NORMALISERS = {'iin': norm_iin, 'contract': norm_contract, 'ref': norm_ref}


def _key_mask(s, keys, kind, name=''):
    """Boolean mask: the NORMALISED key value is one of the population keys (padding, '.0', lost leading zeros,
    mixed formats). Warns when rows match only after normalisation."""
    m = _NORMALISERS[kind](s).isin(keys).to_numpy()
    raw = (s.isin(_numeric_keys(keys)) if pd.api.types.is_numeric_dtype(s) else s.isin(_key_variants(keys, kind))).to_numpy()
    extra = int((m & ~raw).sum())
    if extra:
        print(f'WARN  {name}.{s.name}: {extra:,} of {int(m.sum()):,} population rows match only after key '
              'normalisation (padded / mixed-format keys) - kept')
    return m


def _arrow_key_expr(field, typ, kind, keys):
    """pyarrow dataset filter that applies the same normalisation as norm_iin / norm_contract / norm_ref, so padded
    CHAR(n) keys, '.0' artefacts and lost leading zeros are matched before the rows are read."""
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.dataset as ds
    f = ds.field(field)
    if pa.types.is_integer(typ):
        return f.isin([int(v) for v in _numeric_keys(keys)] or [-1])
    if pa.types.is_floating(typ):
        return f.isin([float(v) for v in _numeric_keys(keys)] or [-1.0])
    if pa.types.is_dictionary(typ):
        if not (pa.types.is_string(typ.value_type) or pa.types.is_large_string(typ.value_type)):
            return None
        f = f.cast(pa.string())
    elif not (pa.types.is_string(typ) or pa.types.is_large_string(typ)
              or getattr(pa.types, 'is_string_view', lambda _: False)(typ)):
        return None
    e = pc.utf8_trim_whitespace(f)
    if kind == 'ref':
        e = pc.utf8_upper(e)
    else:
        e = pc.replace_substring_regex(e, pattern=r'\.0+$', replacement='')
        e = pc.replace_substring_regex(e, pattern=r'\D', replacement='')
        if kind == 'iin':
            e = pc.utf8_lpad(e, width=12, padding='0')
    return e.isin(sorted(k for k in keys if isinstance(k, str)) or ['<none>'])


def _read_parquet(paths, name, columns=None, filters=None):
    """Read parquet file(s) with column projection and a normalised-key push-down filter (rows are re-checked
    in pandas). A file whose key column type cannot be filtered is read in full and filtered afterwards."""
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq
    parts = []
    for p in paths:
        schema = pq.read_schema(p)
        cols = [c for c in columns if c in schema.names] if columns else None
        expr = None
        for kcol, kind, keys in filters or []:
            if kcol not in schema.names:
                continue
            e = _arrow_key_expr(kcol, schema.field(kcol).type, kind, keys)
            if e is None:                         # unsupported key type: no push-down for this file
                expr = None
                print(f'NOTE  {name}: key column {kcol} has type {schema.field(kcol).type} -> read without push-down')
                break
            expr = e if expr is None else (expr | e)
        dset = ds.dataset(str(p), format='parquet')
        try:
            tbl = dset.to_table(columns=cols, filter=expr)
        except Exception as e:                    # e.g. an exotic type in the filter -> post-filter in pandas
            print(f'NOTE  {name}: push-down filter failed ({type(e).__name__}) -> full read of {p.name}')
            tbl = dset.to_table(columns=cols)
        parts.append(tbl.to_pandas())
    return parts[0] if len(parts) == 1 else pd.concat(parts, ignore_index=True)


def load_table(name, columns=None, iin_filter=None, iin_col=None, key_filters=None):
    """Load one table. Priority: (1) a DataFrame named `name` in globals() (your kernel), (2) parquet
    DATA_DIR/FILES[name], (3) the DEMO generator (only when RUN_MODE == 'demo').
    Rows are restricted to the population: `iin_filter` on `iin_col` and/or `key_filters`
    = [(column, kind, keys), ...] (OR-combined; kind in iin | contract | ref). Keys are compared after
    normalisation. Missing columns are ignored.
    Returns None (with a warning) if the table is unavailable or dropped; its block is then skipped."""
    t0 = time.perf_counter()
    filters = list(key_filters or [])
    if iin_filter is not None and iin_col:
        filters.append((iin_col, 'iin', set(iin_filter)))
    rec = dict(table=name, source=None, rows_raw=np.nan, rows_loaded=0, n_cols=0, missing_cols='', seconds=0.0)
    df = None
    if name in CONFIG['DROP_TABLES']:
        rec['source'] = 'dropped (CONFIG)'
    elif RUN_MODE == 'real':
        g = _user_df(name)
        if g is not None:
            df, rec['source'], rec['rows_raw'] = g, 'globals', len(g)
        elif (paths := _parquet_paths(name)):
            import pyarrow.parquet as pq
            rec['rows_raw'] = sum(pq.read_metadata(p).num_rows for p in paths)
            df, rec['source'] = _read_parquet(paths, name, columns, filters), f'parquet x{len(paths)}'
    else:
        demo = _demo_tables()
        if name in demo:
            df, rec['source'], rec['rows_raw'] = demo[name], 'demo', len(demo[name])
    if df is None:
        rec['source'] = rec['source'] or 'not found'
        LOAD_LOG.append(rec)
        print(f"WARN  {name}: {rec['source']} -> dependent block skipped")
        return None
    keep = list(df.columns) if columns is None else [c for c in columns if c in df.columns]
    rec['missing_cols'] = ', '.join(c for c in (columns or []) if c not in df.columns)[:120]
    active = [(c, k, v) for c, k, v in filters if c in df.columns]
    if filters and not active:
        print(f'WARN  {name}: none of the key columns {[c for c, _, _ in filters]} found -> no population filter')
    if active:
        mask = np.zeros(len(df), dtype=bool)
        for c, kind, keys in active:
            mask |= _key_mask(df[c], keys, kind, name)
        df = df.loc[mask, keep]
    else:
        df = df[keep]
    df = df.reset_index(drop=True)            # a new object: the caller's DataFrame is never modified
    rec.update(rows_loaded=len(df), n_cols=df.shape[1], seconds=round(time.perf_counter() - t0, 2))
    if active and len(df) == 0 and rec['rows_raw'] and rec['rows_raw'] > 0:
        print(f"WARN  {name}: {int(rec['rows_raw']):,} rows but NONE matches the population keys "
              f"{[c for c, _, _ in active]} -> check the key format / column")
    LOAD_LOG.append(rec)
    return df


RUN_MODE = resolve_mode()
_out = CONFIG['OUT_DIR']
if RUN_MODE == 'demo' and not _OUT_DIR_SET and str(_out) == 'out':
    _out = 'out_demo'                         # synthetic results never land next to real ones
OUT = nb_path(_out)
for _d in (OUT, OUT / 'charts', OUT / 'model'):
    _d.mkdir(parents=True, exist_ok=True)
log(f"RUN_MODE = {RUN_MODE!r} (CONFIG['MODE'] = {CONFIG['MODE']!r}); dropped tables: {CONFIG['DROP_TABLES'] or 'none'}; "
    f'outputs -> {OUT}')

## 3. Data load, normalisation and data-quality report
1. **Required tables.** `contr` is normalised and de-duplicated: the same contract can appear twice with different `end_date`, so the row with the latest `open_date`, then the latest `end_date`, is kept. `ovrd` is restricted to contracts in `contr` with DPD > 0.
2. **Population** = contracts that ever appear in `ovrd` with DPD > 0, plus their clients and process refs. Every optional table is loaded **only for these keys and only with the needed columns**. At real scale this is what keeps `geo` (37M rows) and `vneshpt` (33M rows) manageable.
3. **Data-quality report**: rows, key nulls, duplicates and date ranges for each table.

`close_date`, `status_name`, `restructing_fl` and `end_date` in `contr` are extraction-time values (`end_date` moves on restructuring). They are never model features; the term and instalment proxy derived from `end_date` feed only the segmentation axes.

In [ ]:
contr_raw = load_table('contr')
if contr_raw is None:
    raise RuntimeError('contr is required (globals / parquet / demo)')
_c = contr_raw.assign(contract_number=norm_contract(contr_raw['contract_number']),
                      iin=norm_iin(contr_raw['iin']),
                      process_ref=norm_ref(col(contr_raw, 'process_ref')))
for _dcol in ('open_date', 'end_date', 'close_date'):
    if _dcol in _c.columns:
        _c[_dcol] = to_dt(_c[_dcol])
_n_raw = len(_c)
contracts = (_c.dropna(subset=['contract_number'])
               .sort_values(['contract_number', 'open_date', 'end_date'], na_position='first')
               .drop_duplicates('contract_number', keep='last').reset_index(drop=True))
if 'end_date' in _c.columns:   # earliest end_date of a duplicated contract = closest to the original schedule
    contracts['end_date_min'] = contracts['contract_number'].map(_c.groupby('contract_number')['end_date'].min())
_raw_iin = contr_raw['iin'].astype('str').str.strip().str.replace(r'\.0+$', '', regex=True)
contr_summary = pd.Series({
    'contr rows (raw)': _n_raw,
    'duplicate contract rows dropped': _n_raw - len(contracts),
    'contracts': len(contracts),
    'clients (IIN)': contracts['iin'].nunique(),
    'invalid / missing IIN': int(contracts['iin'].isna().sum()),
    'IINs that needed leading zeros restored': int((_raw_iin.str.len() < 12).sum()),
    'process_ref shared by >1 contract': int(contracts['process_ref'].dropna().duplicated().sum()),
    'open_date range': f"{contracts['open_date'].min():%Y-%m-%d} .. {contracts['open_date'].max():%Y-%m-%d}",
}, name='value', dtype=object)

LEAKAGE_COLS = ('close_date', 'status_name', 'restructing_fl', 'end_date')
if 'status_name' in contracts.columns:
    _st = contracts['status_name'].value_counts(dropna=False).rename('contracts').to_frame()
    _st['share'] = _st['contracts'] / len(contracts)
    show(_st, 'contr.status_name (status at extraction time)')
    if set(contracts['status_name'].dropna().unique()) <= {'Актуален'}:
        print('WARNING: contr holds only currently active (Актуален) contracts -> closed / written-off / sold '
              'contracts are missing: survivorship bias (the worst outcomes are under-represented). '
              'Please re-extract contr including closed contracts of the same cohorts.')
    else:
        _na = (contracts['status_name'] != 'Актуален').mean()
        print(f'contr includes non-active statuses ({_na:.1%} of contracts). Check that ALL closed / written-off '
              'contracts of the cohorts are there; otherwise the history suffers from survivorship bias.')
print('Leakage guard: close_date / status_name / restructing_fl / end_date are extraction-time values -> never model '
      'features (term, remaining term and the instalment proxy derived from end_date feed only the segmentation axes).')

In [ ]:
ovrd_raw = load_table('ovrd')
if ovrd_raw is None:
    raise RuntimeError('ovrd is required (it is the target source)')
_o = ovrd_raw.assign(contract_number=norm_contract(ovrd_raw['contract_number']),
                     act_date=to_dt(ovrd_raw['act_date']),
                     dpd=pd.to_numeric(ovrd_raw['ovd_days_cnt'], errors='coerce'))
_rm = to_dt(col(ovrd_raw, 'rep_month')).fillna(_o['act_date'])
_o['rep_month'] = (_rm - pd.to_timedelta(_rm.dt.day - 1, unit='D')).dt.normalize().astype('datetime64[us]')
_known = _o['contract_number'].isin(set(contracts['contract_number']))
ovrd_stats = pd.Series({
    'ovrd rows (raw)': len(_o),
    'rows of contracts not in contr (other products, dropped)': int((~_known).sum()),
    'rows with DPD <= 0 / missing (dropped)': int((~(_o['dpd'] > 0)).sum()),
    'rows with unparseable act_date (dropped)': int(_o['act_date'].isna().sum()),
}, name='value', dtype=object)
ovrd_clean = _o[_known & (_o['dpd'] > 0) & _o['act_date'].notna()].reset_index(drop=True)

POP_CONTRACTS = set(ovrd_clean['contract_number'])
_pop = contracts[contracts['contract_number'].isin(POP_CONTRACTS)]
POP_IINS, POP_REFS = set(_pop['iin'].dropna()), set(_pop['process_ref'].dropna())
POP_KEYS = {'iin': POP_IINS, 'contract': POP_CONTRACTS, 'ref': POP_REFS}
ovrd_stats['population: ever-delinquent contracts'] = len(POP_CONTRACTS)
ovrd_stats['population: clients'] = len(POP_IINS)
ovrd_stats['share of portfolio ever delinquent'] = f'{len(POP_CONTRACTS) / len(contracts):.1%}'

# Survivorship estimate: ovrd contracts missing from contr whose number looks like a portfolio contract
# (same length and digits 3-6 as >= 1% of contr) are probably auto loans that left contr (closed, written off,
# sold) - exactly the outcomes a contr of active contracts misses.
_sig = lambda s: s.str.len().astype('str') + ':' + s.str.slice(2, 6)
_csig = _sig(contracts['contract_number']).value_counts(normalize=True)
_port_sig = set(_csig[_csig >= 0.01].index)
_gone = _o[~_known & _o['contract_number'].notna()]
_gone = _gone.assign(auto_like=_sig(_gone['contract_number']).isin(_port_sig))
_gmax = _gone.groupby(['contract_number', 'auto_like'])['dpd'].max().reset_index()
_al = _gmax[_gmax['auto_like']]
_in90 = ovrd_clean.groupby('contract_number')['dpd'].max().ge(CONFIG['BAD_DPD']).mean() if len(ovrd_clean) else np.nan
ovrd_stats['ovrd contracts not in contr that look like portfolio contracts'] = len(_al)
ovrd_stats['... their share reaching 90+ DPD (vs. contracts in contr)'] = (
    f"{_al['dpd'].ge(CONFIG['BAD_DPD']).mean():.1%} vs {_in90:.1%}" if len(_al) else 'n/a')
if len(_al) >= max(10, 0.02 * len(POP_CONTRACTS)):
    print(f"WARNING survivorship: {len(_al):,} contracts in ovrd look like portfolio contracts but are missing from contr "
          f"({_al['dpd'].ge(CONFIG['BAD_DPD']).mean():.0%} of them reached 90+ DPD). They are dropped, so the bad rate "
          'is biased low. Re-extract contr including closed / written-off / sold contracts.')
show(pd.concat([contr_summary, ovrd_stats]).to_frame(), 'Required tables')

In [ ]:
# ---- what to read from each optional source: columns and population keys (OR-combined) ----------
TABLE_COLUMNS = {
    'ver_df_all_2024_2026': None,     # all 203 columns (171 numeric are features)
    'bankrupt': ['iin', 'processcreate', 'date_value', 'date_change', 'bankruptcytype', 'statementstatus',
                 'request_amount'],
    'ocrm_call': ['callid', 'num_dbz', 'createdon_date', 'date_change1', 'date_value1', 'talktime', 'result_predl',
                  'call_comment', 'tnumb_to', 'tsptp_date', 'activityid', 'act_creat_date', 'fact_zvr_date',
                  'date_value', 'activ_sost', 'activ_result', 'activ_result_dtl', 'company_sc', 'ocrm_tsdebt_id'],
    'vneshpt_ot': ['req_id', 'iin_bin', 'recip_client_iin', 'is_actual', 'date_change', 'accept_date', 'close_date',
                   'amount_sum', 'opertype_name', 'descr', 'k2_requirement_group_id', 'k2_req_status_id'],
    'vnpt': ['req_id', 'dogovor', 'doc_num', 'iin_bin', 'recip_client_iin', 'is_actual', 'date_change',
             'accept_date', 'close_date', 'amount_sum', 'k2_req_status_id'],
    'excl': ['iin_bin', 'is_active_fl', 'exclude_title', 'is_actual', 'updated_date'],
    'vneshpt': ['РЕФЕРЕНС', 'ИИН', 'ext_dea_code', 'longname', 'status_code', 'status_name', 'Причина',
                'created_date', 'date_change', 'debt'],
    'geo': ['client_iin', 'event_datetime', 'date_value', 'action', 'latitude', 'longitude', 'device_id',
            'device_name', 'ip_address', 'app_version_name'],
    'dm': None, 'dm_rej': None,       # 159 columns, auto-profiled
    'hes_pkb_agg': None,
}
TABLE_KEYS = {
    'ver_df_all_2024_2026': [('IIN', 'iin'), ('iin', 'iin')],
    'bankrupt': [('iin', 'iin')],
    'ocrm_call': [('num_dbz', 'contract')],
    'vneshpt_ot': [('iin_bin', 'iin'), ('recip_client_iin', 'iin')],
    'vnpt': [('dogovor', 'contract'), ('iin_bin', 'iin')],
    'excl': [('iin_bin', 'iin')],
    'vneshpt': [('ИИН', 'iin'), ('ext_dea_code', 'contract')],
    'geo': [('client_iin', 'iin')],
    'dm': [('process_ref', 'ref')],
    'dm_rej': [('iin', 'iin'), ('IIN', 'iin'), ('client_iin', 'iin'), ('iin_bin', 'iin')],
    'hes_pkb_agg': [('iin', 'iin')],
}

# ---- per-source preparation (keys normalised, dates parsed, duplicates removed) -------------------
DQ_NOTES = {}
# ---- ver_df column groups (used by prep_ver and fb_profile) --------------------------------------
PROFILE_FLAGS = ['FLG_DEATH', 'FLG_BLACK_LIST', 'FLG_LOAN_OVERDUE_MORE_1', 'MAU_FIREBASE', 'MAU_AUTORIZATION',
                 'HALYK_EMPLOYEE']
PROFILE_CATS = ['GENDER', 'BRANCH', 'CLIENT_SEGMENT', 'DIGITAL_SEGMENT', 'EDUCATION', 'FAMILY_STATUS',
                'SALARY_CATEGORY_BY_MODEL', 'TRANSACTION_CLIENT_SEGMENT', 'MOBILE_OS', 'CITIZENSHIP']
TREND_VARS = ['DEPO_SUM', 'PKB_SUM_UNPAY_ACT', 'SUM_PKB_OVRD_ACT', 'VOLUME_INBOX_TRANSF_ANOT_BANK',
              'VOLUME_INBOX_TRANSF_KASPI', 'VOLUME_INBOX_TRANSF_HALYK', 'CNT_SESSIONS', 'CRED_REMAIN_LOAN_SUM',
              'PKB_REQ_30D', 'AMOUNT_LOAN_OVERDUE', 'CNT_PAYMENT', 'MONTH_LOAN_PAY_AMOUNT']
INCOME_VARS = ['VOLUME_INBOX_TRANSF_ANOT_BANK', 'VOLUME_INBOX_TRANSF_KASPI', 'VOLUME_INBOX_TRANSF_HALYK',
               'AMOUNT_SOC_PAYOUT', 'AMOUNT_PENSION']
CAPACITY_INCOME_VARS = [c for c in INCOME_VARS if c not in ('AMOUNT_SOC_PAYOUT', 'AMOUNT_PENSION')]  # protected inflows out
HOME_COUNTRY_RE = r'КАЗАХ|ҚАЗАҚ|KAZAK|KAZAH|QAZAQ|^KZ$|^KAZ$|^РК$'


def is_abroad_country(cty):
    """1 = a country other than Kazakhstan (any spelling), 0 = Kazakhstan, NaN = unknown. Input: as_cat() output."""
    home = cty.str.contains(HOME_COUNTRY_RE, case=False, regex=True, na=False)
    return pd.Series(np.where(cty.isna(), np.nan, (~home).astype(float)), index=cty.index)




def prep_ver(v):
    v = v.rename(columns={'IIN': 'iin'})
    v = v.assign(iin=norm_iin(v['iin']), DATE_VALUE=to_dt(v['DATE_VALUE']))
    if 'DOC_EXPIRY_DATE' in v.columns:
        v['DOC_EXPIRY_DATE'] = to_dt(v['DOC_EXPIRY_DATE'])
    n0 = len(v)
    v = v.dropna(subset=['iin', 'DATE_VALUE']).drop_duplicates(['iin', 'DATE_VALUE'], keep='last')
    notes = [f'{n0 - len(v):,} rows dropped (missing key / duplicate iin x DATE_VALUE)']
    # string flags: raw vocabulary and parse rate (an unknown encoding would silently switch R01 / engagement off)
    for c in [c for c in PROFILE_FLAGS if c in v.columns and not pd.api.types.is_numeric_dtype(v[c])]:
        nn = int(v[c].notna().sum())
        ok = int(flag01(v[c]).notna().sum())
        top = v[c].astype('str').value_counts().head(4).to_dict()
        notes.append(f'{c}: parsed {ok / max(nn, 1):.0%} of {nn:,} non-null {top}')
        if nn and ok < 0.99 * nn:
            print(f'WARN  ver_df.{c}: only {ok / nn:.1%} of non-null values parse as 0/1; raw values {top} -> extend flag01')
    if 'LAST_EVENT_COUNTRY' in v.columns:
        cty = as_cat(v['LAST_EVENT_COUNTRY'])
        ab = is_abroad_country(cty)
        top = cty.value_counts().head(5).to_dict()
        notes.append(f'LAST_EVENT_COUNTRY top {top}; abroad share {ab.mean():.1%}')
        if ab.notna().any() and ab.mean() > 0.05:
            print(f'WARN  ver_df.LAST_EVENT_COUNTRY: {ab.mean():.1%} of snapshots read as abroad; top values {top} -> '
                  'check HOME_COUNTRY_RE')
    snap = v['DATE_VALUE'].drop_duplicates()
    notes.append(f'{snap.dt.is_month_end.sum()} month-end + {(~snap.dt.is_month_end).sum()} other snapshot dates')
    DQ_NOTES['ver_df_all_2024_2026'] = '; '.join(notes)
    return to_float32(v).reset_index(drop=True)


def prep_bankrupt(b):
    b = b.assign(iin=norm_iin(b['iin']),
                 event_date=coalesce(to_dt(col(b, 'processcreate')), to_dt(col(b, 'date_value')),
                                     to_dt(col(b, 'date_change'))),
                 refused=(col(b, 'statementstatus').astype('str').str.contains('otkaz|отказ', case=False, na=False)
                          if 'statementstatus' in b.columns else pd.Series(True, index=b.index)).astype('float32'))
    types = b['bankruptcytype'].value_counts().to_dict() if 'bankruptcytype' in b.columns else {}
    DQ_NOTES['bankrupt'] = f'types {types}; refused share {b["refused"].mean():.0%}'
    return b.dropna(subset=['iin', 'event_date']).reset_index(drop=True)


OCRM_KEEP = {'num_dbz', 'contract_number', 'callid', 'createdon_date', 'date_change1', 'date_value1', 'talktime',
             'result_predl', 'tnumb_to', 'tsptp_date', 'call_comment', 'activityid', 'act_creat_date', 'fact_zvr_date',
             'date_value', 'activ_sost', 'activ_result', 'activ_result_dtl', 'ocrm_tsdebt_id', 'company_sc'}


def prep_ocrm(o):
    o = o.assign(contract_number=norm_contract(o['num_dbz']))
    empty = [c for c in o.columns if c not in OCRM_KEEP and o[c].isna().all()]
    o = o.drop(columns=empty)                 # all-null technical columns (key / used columns are always kept)
    calls = acts = None                       # None only when the key column itself is absent
    if 'callid' in o.columns:
        calls = o.dropna(subset=['callid']).drop_duplicates('callid')
        calls = calls.assign(call_ts=coalesce(to_dt(col(calls, 'createdon_date')), to_dt(col(calls, 'date_change1')),
                                              to_dt(col(calls, 'date_value1'))),
                             talktime=pd.to_numeric(col(calls, 'talktime'), errors='coerce'),
                             tsptp_date=to_dt(col(calls, 'tsptp_date')))
        keep = ['contract_number', 'callid', 'call_ts', 'talktime', 'result_predl', 'tnumb_to', 'tsptp_date',
                'call_comment']
        calls = calls[[c for c in keep if c in calls.columns]].dropna(subset=['call_ts', 'contract_number'])
    if 'activityid' in o.columns:
        acts = o.dropna(subset=['activityid']).drop_duplicates('activityid')
        acts = acts.assign(act_ts=coalesce(to_dt(col(acts, 'act_creat_date')), to_dt(col(acts, 'fact_zvr_date')),
                                           to_dt(col(acts, 'date_value'))))
        keep = ['contract_number', 'activityid', 'act_ts', 'activ_sost', 'activ_result', 'activ_result_dtl',
                'ocrm_tsdebt_id', 'company_sc']
        acts = acts[[c for c in keep if c in acts.columns]].dropna(subset=['act_ts', 'contract_number'])
    n_c = 0 if calls is None else len(calls)
    n_a = 0 if acts is None else len(acts)
    DQ_NOTES['ocrm_call'] = (f'calls x activities join: {len(o):,} rows -> {n_c:,} calls + {n_a:,} activities '
                             f'({len(o) / max(n_c, 1):.2f} rows per call); all-null columns dropped: {len(empty)}')
    return calls, acts


def prep_k2(t, name, contract_key=False):
    """K2 card-index (vneshpt_ot = third-party collection orders ИР3/ИР4, vnpt = the bank's own ВнПТ)."""
    n0 = len(t)
    if 'is_actual' in t.columns:
        t = t[t['is_actual'].astype('str').str.strip().str.upper().eq('A')]
    iin = norm_iin(col(t, 'iin_bin'))
    if 'recip_client_iin' in t.columns:
        iin = iin.fillna(norm_iin(t['recip_client_iin']))
    t = t.assign(iin=iin, accept_date=to_dt(t['accept_date']), close_date=to_dt(col(t, 'close_date')),
                 date_change=to_dt(col(t, 'date_change')),
                 amount_sum=pd.to_numeric(col(t, 'amount_sum'), errors='coerce'))
    if contract_key:
        cn = norm_contract(col(t, 'dogovor'))
        if 'doc_num' in t.columns:
            cn = cn.fillna(norm_contract(t['doc_num']))
        t = t.assign(contract_number=cn)
    if 'req_id' in t.columns:                 # SCD2: one (latest) version per demand
        t = t.sort_values('date_change', na_position='first').drop_duplicates('req_id', keep='last')
    t = t.dropna(subset=['accept_date']).reset_index(drop=True)
    ops = t['opertype_name'].astype('str').str.strip().value_counts().head(6).to_dict() if 'opertype_name' in t.columns else {}
    DQ_NOTES[name] = (f'{n0 - len(t):,} rows dropped (not actual / old SCD versions); open (close_date NaT): '
                      f'{t["close_date"].isna().mean():.0%}; opertype_name {ops}')
    return t


EXCL_CATEGORIES = [('bankruptcy', r'банкрот'), ('socially_vulnerable', r'СУСН'), ('uvek_letter_2020', r'УВЕК'),
                   ('simplified', r'упращ|упрощ'), ('bizbirgemiz', r'БизБиргемиз|Бiз|Біз')]
EXCL_ALL = [k for k, _ in EXCL_CATEGORIES] + ['other']
EXCL_SOCIAL = ['socially_vulnerable', 'uvek_letter_2020', 'simplified', 'bizbirgemiz']   # R04; 'other' -> review flag


def prep_excl(e):
    if 'is_actual' in e.columns:
        e = e[e['is_actual'].astype('str').str.strip().str.upper().eq('A')]
    title = e['exclude_title'].astype('str').str.strip()
    cat = np.select([title.str.contains(p, case=False, regex=True, na=False).to_numpy() for _, p in EXCL_CATEGORIES],
                    [k for k, _ in EXCL_CATEGORIES], default='other')
    e = e.assign(iin=norm_iin(e['iin_bin']), excl_category=pd.Series(cat, index=e.index, dtype='str'),
                 title_norm=title.str.upper(), is_active_fl=pd.to_numeric(col(e, 'is_active_fl'), errors='coerce'),
                 updated_date=to_dt(col(e, 'updated_date')))
    n0 = len(e)
    e = (e.dropna(subset=['iin']).sort_values('updated_date', na_position='first')
          .drop_duplicates(['iin', 'excl_category', 'title_norm'], keep='last').reset_index(drop=True))
    DQ_NOTES['excl'] = f'{n0 - len(e):,} duplicate rows dropped; categories {e["excl_category"].value_counts().to_dict()}'
    return e


def prep_vneshpt(x):
    x = x.rename(columns={'РЕФЕРЕНС': 'reference', 'ИИН': 'iin_raw', 'Тип бизнеса': 'business_type', 'Причина': 'reason'})
    n0 = len(x)
    x = x.drop_duplicates([c for c in ('reference', 'longname', 'status_code', 'date_change', 'debt') if c in x.columns])
    n_dup = n0 - len(x)
    x = x.assign(iin=norm_iin(col(x, 'iin_raw')), contract_number=norm_contract(col(x, 'ext_dea_code')),
                 created_date=to_dt(col(x, 'created_date')), date_change=to_dt(col(x, 'date_change')),
                 debt=pd.to_numeric(col(x, 'debt'), errors='coerce'))
    x = x.assign(ev_date=coalesce(x['date_change'], x['created_date']),
                 created_date=coalesce(x['created_date'], x['date_change']))
    DQ_NOTES['vneshpt'] = f'exact duplicates dropped FIRST: {n_dup:,} ({n_dup / max(n0, 1):.0%} of rows)'
    return x.dropna(subset=['iin', 'ev_date']).reset_index(drop=True)


def prep_geo(g):
    """App logins -> daily aggregates per client (+ distinct device / IP per day)."""
    lat_min, lat_max, lon_min, lon_max = CONFIG['KZ_BBOX']
    g = g.assign(iin=norm_iin(g['client_iin']),
                 ts=coalesce(to_dt(col(g, 'event_datetime')), to_dt(col(g, 'date_value')))).dropna(subset=['iin', 'ts'])
    lat = pd.to_numeric(col(g, 'latitude'), errors='coerce')
    lon = pd.to_numeric(col(g, 'longitude'), errors='coerce')
    valid = lat.notna() & lon.notna() & ~((lat == 0) & (lon == 0))        # 0/0 = missing geolocation
    inside = lat.between(lat_min, lat_max) & lon.between(lon_min, lon_max)
    act = col(g, 'action').astype('str').str.strip().str.lower()
    g = g.assign(day=g['ts'].dt.normalize(),
                 bio=act.isin(['face_id', 'touch_id']).astype('int32'),
                 night=(g['ts'].dt.hour < 6).astype('int32'),
                 valid=valid.astype('int32'), abroad=(valid & ~inside).astype('int32'),
                 premium=col(g, 'device_name').astype('str').str.contains(r'\b(?:Pro|Max|Ultra)\b', case=False,
                                                                         regex=True, na=False).astype('int32'),
                 app_major=pd.to_numeric(col(g, 'app_version_name').astype('str').str.extract(r'^\s*(\d+)', expand=False),
                                         errors='coerce'))
    daily = (g.groupby(['iin', 'day'], sort=False)
              .agg(n=('ts', 'size'), n_bio=('bio', 'sum'), n_night=('night', 'sum'), n_valid=('valid', 'sum'),
                   n_abroad=('abroad', 'sum'), premium=('premium', 'max'), app_major=('app_major', 'max'))
              .reset_index())
    dev = g[['iin', 'day', 'device_id']].dropna().drop_duplicates() if 'device_id' in g.columns else None
    ip = g[['iin', 'day', 'ip_address']].dropna().drop_duplicates() if 'ip_address' in g.columns else None
    DQ_NOTES['geo'] = (f'{len(g):,} logins -> {len(daily):,} client-days; coordinates 0/0 (missing): '
                       f'{1 - valid.mean():.0%}; abroad logins: {(valid & ~inside).mean():.2%}')
    return daily, dev, ip


DATE_CANDIDATES = ['processcreate', 'date_value', 'date_change', 'lastmodificationdate', 'createdon', 'created_date']
IIN_CANDIDATES = ['iin', 'IIN', 'client_iin', 'iin_bin']
ID_LIKE = re.compile(r'(^|_)(id|iin|bin|inn|rnn|fio|name|mobile|phone|tel|email|ref|orderid|clientid|processid)(_|$)'
                     r'|_id$|^id', re.I)


def detect_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)


def detect_date_col(df, candidates=DATE_CANDIDATES, min_share=0.8):
    """First candidate column with >= 80% parseable dates."""
    for c in candidates:
        if c in df.columns and to_dt(df[c]).notna().mean() >= min_share:
            return c
    return None


def _looks_like_id(v):
    v = v.dropna()
    if len(v) == 0:
        return False
    integral = np.mean(np.isclose(v.to_numpy() % 1, 0)) > 0.99
    return bool(integral and v.abs().median() >= 1e8 and v.nunique() / len(v) > 0.3)


def prep_dm(d, max_num=40, max_card=30):
    """Approved applications: profile the 159 (mostly text) columns once -> numeric + small categoricals.
    Leakage guard: only fields fixed at application time may enter the model. dm is a current extract (not
    versioned), so lifecycle fields (status, DPD, close / modification dates ...) would leak and cannot be caught by
    the as-of test. On real data set CONFIG['DM_ALLOW_COLUMNS'] to a reviewed list; until then the auto-selected
    columns are filtered by DM_DENY_PATTERN and printed for sign-off."""
    dcol = detect_date_col(d)
    d = d.copy()                           # consolidate a fragmented 159-column frame before adding columns
    d = d.assign(process_ref=norm_ref(col(d, 'process_ref')),
                 app_date=to_dt(d[dcol]) if dcol else pd.Series(pd.NaT, index=d.index, dtype='datetime64[us]'))
    base = [c for c in ('request_amount', 'amount_duration', 'response_result') if c in d.columns]
    skip = set(base) | {dcol, 'process_ref', 'app_date'} | set(DATE_CANDIDATES) | set(IIN_CANDIDATES)
    allow = CONFIG['DM_ALLOW_COLUMNS']
    deny = re.compile(CONFIG['DM_DENY_PATTERN'], re.I)
    num, cats, stats, denied = {}, [], [], []
    for c in text_cols(d):
        if c in skip or ID_LIKE.search(c):
            continue
        if allow is not None and c not in allow:
            continue
        if allow is None and deny.search(c):
            denied.append(c)
            continue
        s = d[c]
        nn = int(s.notna().sum())
        if nn < 0.2 * len(d):
            continue
        txt = s.astype('str').str.strip()
        v = pd.to_numeric(txt.str.replace(',', '.', regex=False), errors='coerce')
        if v.notna().sum() >= 0.9 * nn:                              # >= 90% numeric-parsable text
            if not _looks_like_id(v):
                num[c] = v.astype('float32')
                stats.append((c, nn, v.nunique()))
            continue
        smp = txt.dropna().head(2000)
        if len(smp) and pd.to_datetime(smp, errors='coerce', format='ISO8601').notna().mean() >= 0.8:
            continue                                                 # date-like text
        if 2 <= s.nunique() <= max_card:
            cats.append(c)
    keep_num = [c for c, _, _ in sorted(stats, key=lambda t: (-t[1], -t[2]))[:max_num]]
    res = pd.concat([d[['process_ref', 'app_date']],
                     pd.DataFrame({c: pd.to_numeric(d[c], errors='coerce') for c in base}, index=d.index).astype('float32'),
                     pd.DataFrame({c: num[c] for c in keep_num}, index=d.index),
                     pd.DataFrame({f'cat_{c}': as_cat(d[c]) for c in cats}, index=d.index)], axis=1)
    res = (res.dropna(subset=['process_ref']).sort_values('app_date', na_position='first')
              .drop_duplicates('process_ref', keep='last').reset_index(drop=True))
    DQ_NOTES['dm'] = (f'date col {dcol!r}; {len(text_cols(d))} text cols -> {len(keep_num)} numeric '
                      f'(>=90% parsable, cap {max_num}) + {len(cats)} categoricals; '
                      f"{'allow-list' if allow is not None else f'deny-pattern dropped {len(denied)}'}")
    DM_SELECTED.update(numeric=keep_num, categorical=cats, denied=denied, mode='allow-list' if allow is not None else 'auto')
    return res


DM_SELECTED = {}


def prep_rej(r):
    if len(r) == 0:                        # no rejected application in the population: counts are simply 0
        DQ_NOTES['dm_rej'] = '0 population rows'
        return pd.DataFrame({'iin': pd.Series(dtype='str'), 'app_date': pd.Series(dtype='datetime64[us]'),
                             'request_amount': pd.Series(dtype='float64')})
    icol, dcol = detect_col(r, IIN_CANDIDATES), detect_date_col(r)
    if icol is None or dcol is None:
        raise ValueError(f'dm_rej: iin column {icol!r} / date column {dcol!r} not detected')
    out = pd.DataFrame({'iin': norm_iin(r[icol]), 'app_date': to_dt(r[dcol]),
                        'request_amount': pd.to_numeric(col(r, 'request_amount'), errors='coerce')})
    DQ_NOTES['dm_rej'] = f'iin col {icol!r}, date col {dcol!r}'
    return out.dropna(subset=['iin', 'app_date']).reset_index(drop=True)


# PKB monitoring rule names -> codes (case-insensitive Python regex on the unique names, first match wins; names
# may be truncated with '...'). 'court' is anchored: 'Суд' must not be preceded by a letter ('ссуда' = loan).
PKB_RULE_CODES = [
    ('unsubscribe', r'Отписка'), ('comparison', r'сравнени'), ('ludoman', r'Лудоман'),
    ('stop_credit', r'Стоп.?Кредит'), ('conscript', r'Призывник'), ('bankruptcy', r'Банкрот'),
    ('court', r'(?<![а-яёa-z])Суд'), ('assignment_sold', r'переуступ'), ('closed_repaid', r'закрытие займа'),
    ('collateral_change', r'залог'), ('partial_overdue', r'Частичная просрочка'),
    ('overdue_change', r'сумма просрочки'), ('dpd_change', r'дней просрочки'), ('refinance', r'Рефинанс'),
    ('restructuring', r'Реструктур'), ('prolongation', r'Пролонгац'), ('settled', r'урегулирован'),
    ('credit_app', r'кредитной заявки'), ('report_request', r'запрос[а-яё]* на кредитный отч'),
    ('inquiry', r'искал информацию|Любой поставщик'), ('new_loan', r'нового займа'),
    ('new_contract', r'новый контракт'), ('payment_change', r'ежемесячного платежа'),
    ('principal_change', r'основного долга'),
]
PKB_CODES = [k for k, _ in PKB_RULE_CODES if k != 'unsubscribe'] + ['other']
PKB_KEY_CODES = ['ludoman', 'stop_credit', 'bankruptcy', 'court', 'collateral_change', 'assignment_sold',
                 'partial_overdue', 'overdue_change', 'dpd_change']


_PKB_RX = [(k, re.compile(p, re.I)) for k, p in PKB_RULE_CODES]


def pkb_code(names):
    """rule_name -> code. Python `re` on the unique names: Unicode-aware (pandas 3 pyarrow strings use RE2, where
    \\w is ASCII-only and look-behind is unsupported)."""
    names = pd.Series(names).astype('str')
    u = names.dropna().unique()
    m = {n: next((k for k, rx in _PKB_RX if rx.search(str(n))), 'other') for n in u}
    return names.map(m).astype('str')


PKB_RULE_MAP = pd.DataFrame(columns=['rule_name', 'code', 'rows'])


def prep_pkb(p):
    global PKB_RULE_MAP
    p = p.assign(iin=norm_iin(p['iin']), code=pkb_code(p['rule_name']),
                 first_ts=to_dt(col(p, 'first_ts')), last_ts=to_dt(col(p, 'last_ts')))
    p = p.assign(**{c: pd.to_numeric(col(p, c), errors='coerce').fillna(0) for c in ('cnt_7d', 'cnt_30d', 'cnt_60d', 'cnt_all')})
    snap = to_dt(p['snapshot_date']).max() if 'snapshot_date' in p.columns else p['last_ts'].max()
    PKB_RULE_MAP = (p.groupby(['rule_name', 'code'], observed=True).size().rename('rows').reset_index()
                     .sort_values('code').reset_index(drop=True))
    unmapped = PKB_RULE_MAP.loc[PKB_RULE_MAP['code'] == 'other', 'rule_name'].tolist()
    DQ_NOTES['hes_pkb_agg'] = f'snapshot {snap}; {len(PKB_RULE_MAP)} rule names -> {p["code"].nunique()} codes; unmapped: {len(unmapped)}'
    if unmapped:
        print('WARN  hes_pkb_agg: rule names mapped to no code (-> pkb_other):\n  ' + '\n  '.join(map(str, unmapped)))
    return p[p['code'] != 'unsubscribe'].dropna(subset=['iin']).reset_index(drop=True), snap

In [ ]:
TBL = {}                       # prepared optional sources (keys normalised, population only)


def _prepare(name, fn):
    raw = load_table(name, columns=TABLE_COLUMNS.get(name),
                     key_filters=[(c, kind, POP_KEYS[kind]) for c, kind in TABLE_KEYS.get(name, [])])
    if raw is None:
        return
    try:
        fn(raw)
    except Exception as e:     # a malformed optional source must not stop the run
        if CONFIG['STRICT_BLOCKS']:
            raise
        print(f'WARN  {name}: preparation failed ({type(e).__name__}: {e}) -> source skipped')


def _put(names, frames):
    """Store only real DataFrames (a part that could not be built is simply absent -> its features are skipped)."""
    TBL.update({k: v for k, v in zip(names, frames) if isinstance(v, pd.DataFrame)})


def _put_ocrm(raw):
    _put(('ocrm_calls', 'ocrm_acts'), prep_ocrm(raw))


def _put_geo(raw):
    _put(('geo_daily', 'geo_dev', 'geo_ip'), prep_geo(raw))


def _put_pkb(raw):
    TBL['pkb'], TBL['pkb_snapshot'] = prep_pkb(raw)


# Optional: build the PKB monitoring aggregate now (needs your Vertica query() helper and the SQL placeholders
# replaced; the SQL text is in sql/hes_pkb_monitoring_agg.sql and in the appendix)
if CONFIG['RUN_PKB_SQL'] and _user_df('hes_pkb_agg') is None:
    _sql = (NB_DIR / 'sql' / 'hes_pkb_monitoring_agg.sql').read_text(encoding='utf-8')
    _sql = _sql.replace(':snapshot_date', f"'{pd.Timestamp.today().normalize():%Y-%m-%d}'")
    assert '__PLACEHOLDER' not in _sql, 'replace the placeholder column names in sql/hes_pkb_monitoring_agg.sql first'
    hes_pkb_agg = query(_sql)                                     # noqa: F821 - your existing helper -> DataFrame
    _pkb_path = nb_path(CONFIG['DATA_DIR']) / CONFIG['PKB_AGG_FILE']
    _pkb_path.parent.mkdir(parents=True, exist_ok=True)
    hes_pkb_agg.to_parquet(_pkb_path, index=False)
    log(f'PKB aggregate built: {hes_pkb_agg.shape} -> {_pkb_path}')

with timer('optional sources loaded (population keys only) and normalised'):
    _prepare('ver_df_all_2024_2026', lambda d: TBL.update(ver=prep_ver(d)))
    _prepare('bankrupt', lambda d: TBL.update(bankrupt=prep_bankrupt(d)))
    _prepare('ocrm_call', _put_ocrm)
    _prepare('vneshpt_ot', lambda d: TBL.update(vneshpt_ot=prep_k2(d, 'vneshpt_ot')))
    _prepare('vnpt', lambda d: TBL.update(vnpt=prep_k2(d, 'vnpt', contract_key=True)))
    _prepare('excl', lambda d: TBL.update(excl=prep_excl(d)))
    _prepare('vneshpt', lambda d: TBL.update(vneshpt=prep_vneshpt(d)))
    _prepare('geo', _put_geo)
    _prepare('dm', lambda d: TBL.update(dm=prep_dm(d)))
    _prepare('dm_rej', lambda d: TBL.update(dm_rej=prep_rej(d)))
    _prepare('hes_pkb_agg', _put_pkb)


def _rng(s):
    s = pd.Series(s).dropna()
    return (s.quantile(0.01), s.max()) if len(s) else (pd.NaT, pd.NaT)


# Coverage of each source in time: (start = 1% quantile of its event dates, end). Used for the as-of
# masking of blocks and for the MODEL_BLOCKS='auto' rule.
_SRC_DATE = {  # source table -> (prepared table key, date column)
    'ver_df_all_2024_2026': ('ver', 'DATE_VALUE'), 'bankrupt': ('bankrupt', 'event_date'),
    'ocrm_call': ('ocrm_calls', 'call_ts'), 'vneshpt_ot': ('vneshpt_ot', 'accept_date'),
    'vnpt': ('vnpt', 'date_change'), 'vneshpt': ('vneshpt', 'ev_date'), 'geo': ('geo_daily', 'day'),
    'dm': ('dm', 'app_date'), 'dm_rej': ('dm_rej', 'app_date'), 'hes_pkb_agg': ('pkb', 'first_ts'),
}
SRC_RANGE = {'contr': _rng(contracts['open_date']), 'ovrd': _rng(ovrd_clean['act_date'])}
for _src, (_key, _dc) in _SRC_DATE.items():
    _t = TBL.get(_key)
    if isinstance(_t, pd.DataFrame) and _dc in _t.columns:
        _s = _t[_dc]
        if _src == 'vnpt' and _s.isna().all() and 'accept_date' in _t.columns:
            _s = _t['accept_date']
        SRC_RANGE[_src] = _rng(_s)
if isinstance(TBL.get('excl'), pd.DataFrame):   # a snapshot list: no event history
    SRC_RANGE['excl'] = (pd.NaT, TBL['excl']['updated_date'].max())
SRC_START = {k: v[0] for k, v in SRC_RANGE.items()}
if 'pkb' not in TBL:
    print('NOTICE PKB monitoring aggregate (hes_pkb_agg) not loaded -> R08 gambling, R09 collateral alert, the PKB part '
          'of R02 and the no_credit_offer flag from «Стоп-Кредит» are OFF. Run sql/hes_pkb_monitoring_agg.sql first '
          "(or set CONFIG['RUN_PKB_SQL']=True in the Vertica kernel).")

In [ ]:
# ---- data-quality report: one row per source ------------------------------------------------------
_DQ_SPEC = [  # (source, prepared frame, key column, natural-key columns, date column)
    ('contr', contracts, 'contract_number', ['contract_number'], 'open_date'),
    ('ovrd', ovrd_clean, 'contract_number', ['contract_number', 'rep_month'], 'act_date'),
    ('ver_df_all_2024_2026', TBL.get('ver'), 'iin', ['iin', 'DATE_VALUE'], 'DATE_VALUE'),
    ('bankrupt', TBL.get('bankrupt'), 'iin', ['iin', 'event_date'], 'event_date'),
    ('ocrm_call', TBL.get('ocrm_calls'), 'contract_number', ['callid'], 'call_ts'),
    ('vneshpt_ot', TBL.get('vneshpt_ot'), 'iin', ['iin', 'accept_date', 'amount_sum'], 'accept_date'),
    ('vnpt', TBL.get('vnpt'), 'contract_number', ['contract_number', 'accept_date', 'amount_sum'], 'accept_date'),
    ('excl', TBL.get('excl'), 'iin', ['iin', 'excl_category'], 'updated_date'),
    ('vneshpt', TBL.get('vneshpt'), 'iin', ['reference', 'status_code', 'date_change'], 'ev_date'),
    ('geo', TBL.get('geo_daily'), 'iin', ['iin', 'day'], 'day'),
    ('dm', TBL.get('dm'), 'process_ref', ['process_ref'], 'app_date'),
    ('dm_rej', TBL.get('dm_rej'), 'iin', ['iin', 'app_date'], 'app_date'),
    ('hes_pkb_agg', TBL.get('pkb'), 'iin', ['iin', 'code'], 'last_ts'),
]
_loaded = {r['table']: r for r in LOAD_LOG}
_rows = []
for _name, _df, _key, _nk, _dc in _DQ_SPEC:
    _r = _loaded.get(_name, {})
    _row = dict(table=_name, source=_r.get('source', 'loaded'), rows_raw=_r.get('rows_raw', np.nan),
                rows_population=_r.get('rows_loaded', np.nan), rows_prepared=np.nan, key=_key,
                key_null_pct=np.nan, n_keys=np.nan, dup_natural_key=np.nan, date_min=pd.NaT, date_max=pd.NaT,
                notes=DQ_NOTES.get(_name, ''))
    if _df is not None and _key in _df.columns:
        _nk = [c for c in _nk if c in _df.columns]
        _d = _df[_dc].dropna() if _dc in _df.columns else pd.Series(dtype='datetime64[us]')
        _row.update(rows_prepared=len(_df), key_null_pct=round(100 * _df[_key].isna().mean(), 2),
                    n_keys=_df[_key].nunique(), dup_natural_key=int(_df.duplicated(_nk).sum()) if _nk else np.nan,
                    date_min=_d.min() if len(_d) else pd.NaT, date_max=_d.max() if len(_d) else pd.NaT)
    _rows.append(_row)
DQ_REPORT = pd.DataFrame(_rows).set_index('table')
DQ_REPORT = DQ_REPORT.astype({c: 'Int64' for c in ('rows_raw', 'rows_population', 'rows_prepared', 'n_keys', 'dup_natural_key')})
show(DQ_REPORT, 'Data-quality report (population-filtered extracts)')
if len(PKB_RULE_MAP):
    show(PKB_RULE_MAP, 'PKB rule_name -> code mapping (check once; unmapped names go to pkb_other)')
if DM_SELECTED:
    print(f"dm columns admitted to the model candidates ({DM_SELECTED['mode']}) - SIGN-OFF NEEDED on real data: only "
          'application-time fields may stay; put the reviewed list into CONFIG[\'DM_ALLOW_COLUMNS\']')
    print('  numeric:     ', DM_SELECTED['numeric'])
    print('  categorical: ', DM_SELECTED['categorical'])
    if DM_SELECTED['denied']:
        print('  denied by DM_DENY_PATTERN:', DM_SELECTED['denied'])

## 4. DPD panel, delinquency episodes and scoring checkpoints
- **Panel.** One row per (contract, `rep_month`): the row with the maximum DPD, with its `act_date`. The implied **start of the overdue** is `start_date = act_date - (dpd - 1)`.
- **Episodes need a cure.** Rows are sorted by contract and `start_date`. A row starts a new episode only if its `start_date` lies more than `EPISODE_GAP_DAYS` after the latest `act_date` seen so far for that contract, i.e. the client was observed current in between. A partial payment that moves the oldest unpaid instalment forward while the client stays overdue (**re-ageing**) therefore does *not* create a fake new entry: the row is merged into the running episode and the episode is flagged `reaged`. Each episode spans `[entry_date, last_act_date]` continuously overdue.
- **Checkpoints.** Every episode is scored on the days in arrears listed in `CHECKPOINT_DAYS` (default 1, 8, 15, 22, 29) **as long as it is still overdue on that day** (`last_act_date >= entry_date + day - 1`). An observation has `obs_date = entry_date + day - 1` and the feature `dh_days_in_arrears = day`. Features use **only events dated before `obs_date`**; the DPD history uses only *previous* episodes (rows dated before `entry_date`). The model therefore learns the risk **conditional on having survived k days without a cure**, which is exactly the situation of the current stock.
- **Targets** (per observation):
  - `bad = 1` if a contract row reaches 90 DPD on a date `start_date + 89` inside `(obs_date, obs_date + 6 months]`.
  - `roll31_60d` is the same test at 31+ DPD within 60 days.
  - `self_cure30` means the episode's maximum DPD is at most 30.
- **Labelled** observations have a full 6-month window before the data cut (the maximum `act_date`).
- **Current stock.** Contracts whose row in the latest `rep_month` has DPD 1-30 and is still overdue on the cut. Each is scored on **its current day in arrears** (`obs_date` = the cut, `dh_days_in_arrears` = days since entry), with the same feature definitions as the training checkpoints. Its rule overlays use the latest information at `score_date = cut + 1 day`.

In [ ]:
_p = ovrd_clean.sort_values(['contract_number', 'rep_month', 'dpd', 'act_date'])
panel = _p.drop_duplicates(['contract_number', 'rep_month'], keep='last')
_n_multi = len(_p) - len(panel)
panel = panel.assign(start_date=(panel['act_date'].dt.normalize()
                                 - pd.to_timedelta(panel['dpd'] - 1, unit='D')).astype('datetime64[us]'))
panel = panel.sort_values(['contract_number', 'start_date', 'act_date']).reset_index(drop=True)
_gap = pd.Timedelta(days=CONFIG['EPISODE_GAP_DAYS'])
_cn = panel['contract_number']
_prev_end = panel['act_date'].dt.normalize().groupby(_cn).cummax().groupby(_cn).shift()   # latest overdue day so far
panel['new_ep'] = _prev_end.isna() | (panel['start_date'] > _prev_end + _gap)               # a cure in between
panel['episode_id'] = panel['new_ep'].cumsum().astype('int64') - 1
# re-ageing: the implied start jumps forward while the client is still overdue (partial payment) -> merged
panel['reage'] = ~panel['new_ep'] & ((panel['start_date'] - panel.groupby(_cn)['start_date'].shift()) > _gap)

EPISODES = panel.groupby('episode_id').agg(contract_number=('contract_number', 'first'),
                                           entry_date=('start_date', 'min'), max_dpd=('dpd', 'max'),
                                           last_act_date=('act_date', 'max'), n_months=('rep_month', 'size'),
                                           reaged=('reage', 'max'))
EPISODES = EPISODES.assign(last_act_date=EPISODES['last_act_date'].dt.normalize().astype('datetime64[us]'),
                           reaged=EPISODES['reaged'].astype('int8'),
                           self_cure30=(EPISODES['max_dpd'] <= 30).astype('int8'))
DATA_CUT = panel['act_date'].max().normalize()
SCORE_DATE = DATA_CUT + pd.Timedelta(days=1)
log(f'panel: {len(panel):,} contract-months ({_n_multi} duplicate contract-month rows collapsed to the max DPD); '
    f'{len(EPISODES):,} episodes on {EPISODES["contract_number"].nunique():,} contracts '
    f'({int(EPISODES["reaged"].sum()):,} with re-ageing merged in); data cut {DATA_CUT:%Y-%m-%d}')

In [ ]:
CHECKPOINTS = CONFIG['CHECKPOINT_DAYS']
CP0 = CHECKPOINTS[0]                     # first checkpoint (1 = the entry day)

# ---- history observations: episode x checkpoint, while still overdue ------------------------------
_e = EPISODES.reset_index()
_parts = []
for _k in CHECKPOINTS:
    _od = (_e['entry_date'] + pd.Timedelta(days=_k - 1)).astype('datetime64[us]')
    _alive = (_e['last_act_date'] >= _od).to_numpy()
    _parts.append(_e.loc[_alive, ['episode_id', 'contract_number', 'entry_date']]
                    .assign(checkpoint=_k, days_in_arrears=_k, obs_date=_od[_alive], is_stock=False))

# ---- current stock: one observation per contract, on its current day in arrears -------------------
_last_m = panel['rep_month'].max()
_cur = panel[(panel['rep_month'] == _last_m) & panel['dpd'].between(CONFIG['SOFT_DPD_MIN'], CONFIG['SOFT_DPD_MAX'])]
_n_soft = len(_cur)
if CONFIG['STOCK_ONLY_AT_CUT']:
    _cur = _cur[_cur['act_date'].dt.normalize() >= DATA_CUT]
_st = _cur[['episode_id', 'contract_number']].merge(EPISODES[['entry_date']], left_on='episode_id', right_index=True)
_st = _st.assign(obs_date=_cur['act_date'].dt.normalize().astype('datetime64[us]').to_numpy(),
                 dpd_now=_cur['dpd'].to_numpy(), dpd_asof=_cur['act_date'].to_numpy(), is_stock=True)
_st['days_in_arrears'] = (_st['obs_date'] - _st['entry_date']).dt.days + 1
_st['checkpoint'] = 0                     # 0 = stock (scored on its own day in arrears)

OB = pd.concat(_parts + [_st], ignore_index=True).rename_axis('obs_id')
OB = OB.astype({'checkpoint': 'int64', 'days_in_arrears': 'int64', 'obs_date': 'datetime64[us]'})
OB = OB.join(EPISODES[['max_dpd', 'self_cure30', 'reaged', 'last_act_date']], on='episode_id')
print(f'latest rep_month {_last_m:%Y-%m} (cut {DATA_CUT:%Y-%m-%d}): {_n_soft} rows with DPD '
      f"{CONFIG['SOFT_DPD_MIN']}-{CONFIG['SOFT_DPD_MAX']}, {len(_st)} still overdue on the cut -> current stock "
      f"(days in arrears median {_st['days_in_arrears'].median() if len(_st) else float('nan'):.0f}); "
      f'score_date {SCORE_DATE:%Y-%m-%d}')


def _hits(thr, end_col):
    """Observations whose contract has a row reaching `thr` DPD on a date in (obs_date, end]."""
    h = panel.loc[panel['dpd'] >= thr, ['contract_number', 'start_date']]
    h = h.assign(d=h['start_date'] + pd.Timedelta(days=thr - 1)).drop_duplicates(['contract_number', 'd'])
    m = OB[['contract_number', 'obs_date', end_col]].reset_index().merge(h[['contract_number', 'd']], on='contract_number')
    m = m[(m['d'] > m['obs_date']) & (m['d'] <= m[end_col])]
    return OB.index.isin(m['obs_id'])


OB['horizon_end'] = (OB['obs_date'] + pd.DateOffset(months=CONFIG['HORIZON_MONTHS'])).astype('datetime64[us]')
OB['roll_end'] = (OB['obs_date'] + pd.Timedelta(days=CONFIG['ROLL_DAYS'])).astype('datetime64[us]')
OB['bad'] = _hits(CONFIG['BAD_DPD'], 'horizon_end').astype('int8')
OB['roll31_60d'] = _hits(CONFIG['ROLL_DPD'], 'roll_end').astype('int8')
OB['labeled'] = (OB['horizon_end'] <= DATA_CUT) & ~OB['is_stock']
OB['labeled_roll'] = (OB['roll_end'] <= DATA_CUT) & ~OB['is_stock']
OB['obs_month'] = OB['obs_date'].dt.to_period('M')
OB['entry_month'] = OB['entry_date'].dt.to_period('M')
FIRST = OB.index[(OB['checkpoint'] == CP0)]          # one observation per episode (the first checkpoint)

In [ ]:
_f = OB.loc[FIRST]
_lab = _f[_f['labeled']]
MONTHLY = _f.groupby('entry_month').agg(entries=('bad', 'size'), labeled=('labeled', 'sum'), reaged=('reaged', 'sum'))
MONTHLY['bad_rate'] = _lab.groupby('entry_month')['bad'].mean()
MONTHLY['self_cure_rate'] = _lab.groupby('entry_month')['self_cure30'].mean()
MONTHLY['roll31_rate'] = _f[_f['labeled_roll']].groupby('entry_month')['roll31_60d'].mean()
MONTHLY['bad_rate_reaged'] = _lab[_lab['reaged'] == 1].groupby('entry_month')['bad'].mean()
show(MONTHLY, f'Entries into delinquency by month (first checkpoint = day {CP0}; bad / self-cure on labelled episodes)')

_hl = OB[OB['labeled']]
CHECKPOINT_TABLE = _hl.groupby('checkpoint').agg(n=('bad', 'size'), bad_rate=('bad', 'mean'),
                                                 self_cure=('self_cure30', 'mean'), roll31_60d=('roll31_60d', 'mean'))
CHECKPOINT_TABLE['share_of_episodes'] = CHECKPOINT_TABLE['n'] / max(len(_lab), 1)
show(CHECKPOINT_TABLE, 'Labelled observations by checkpoint: the risk of an episode that is still overdue on day k')
print(f"episodes {len(_f):,} | labelled {len(_lab):,} | bad rate at day {CP0} {_lab['bad'].mean():.1%} | "
      f"self-cure {_lab['self_cure30'].mean():.1%} | observations (all checkpoints) {int((~OB['is_stock']).sum()):,}, "
      f"labelled {int(OB['labeled'].sum()):,} | re-aged episodes {int(_f['reaged'].sum()):,} "
      f"(labelled bad rate {_lab.loc[_lab['reaged'] == 1, 'bad'].mean() if (_lab['reaged'] == 1).any() else float('nan'):.1%})")

# observation frames for the feature blocks: every observation at its obs_date, and the stock overlay
OBS = (OB.reset_index()[['obs_id', 'contract_number', 'obs_date', 'entry_date', 'days_in_arrears']]
         .merge(contracts[['contract_number', 'process_ref', 'iin']], on='contract_number', how='left'))
STOCK_IDS = OB.index[OB['is_stock']]
OBS_NOW = (OBS[OBS['obs_id'].isin(STOCK_IDS)].assign(obs_date=SCORE_DATE, overlay=True)
           .astype({'obs_date': 'datetime64[us]'}).reset_index(drop=True))

## 5. Feature blocks
Each block is a function `fb_x(obs, ctx) -> DataFrame` indexed by `obs_id`, with a block prefix on every column. `obs` has the columns `[obs_id, contract_number, process_ref, iin, obs_date, entry_date, days_in_arrears]` (plus `overlay=True` for the stock overlay, where blocks may use the latest snapshot instead of the model cadence).

Every block follows the same pattern:
1. Filter the source to the population keys.
2. Merge it with `obs` on the key.
3. Keep events dated **strictly before** `obs_date`, using 30/90/180/365-day windows.
4. Aggregate with `groupby(obs_id)`.

There are no row-wise Python loops and no merges without a key filter. Count features are 0 when nothing happened. All features of a source are NaN before that source starts, so a missing history never reads as zero activity.

To add a source: write `fb_new(obs, ctx)`, register it in `BLOCKS` and add its table to `TABLE_COLUMNS` / `TABLE_KEYS`.

| block | source | prefix | content |
|---|---|---|---|
| a contract | contr | `ct_` | amount, term, MOB, remaining term, dealer, instalment proxy |
| b dpd_history | ovrd | `dh_` | days in arrears now (checkpoint); previous episodes: count, max DPD, months in arrears 3/6/12m; instalment number / FPD |
| c profile | ver_df | `pr_` | latest snapshot before obs (171 numbers, flags, categories), 90-day trends, income proxy |
| d internal_debits | vnpt | `vn_` | the bank's own payment demands: counts, open, days to close |
| e enforcement | vneshpt_ot | `en_` | third-party collection orders ИР3/ИР4 (bailiffs, tax) |
| f ext_debits | vneshpt | `ex_` | direct debits sent to other banks, reasons (e.g. insufficient funds) |
| g calls | ocrm_call | `cl_` | calls / activities: contact rate, PTP, broken PTP, third party, abroad notes |
| h app_geo | geo | `geo_` | Homebank logins, devices, biometrics, night, abroad share |
| i exclusions | excl | `exc_` | active exclusion lists (bankruptcy, СУСН, УВЕК, ...) |
| j bankruptcy | bankrupt | `bk_` | refused out-of-court bankruptcy applications |
| k applications | dm / dm_rej | `app_` / `rej_` | application data of the contract; rejected applications before obs |
| l pkb_monitoring | PKB agg | `pkb_` | bureau events (overlay only: history starts 2026-09-07) |

In [ ]:
def events_before(obs, ev, key, date_col, cols=()):
    """obs x events joined on `key`, keeping events dated strictly before obs_date; adds age_d (days)."""
    cols = [c for c in dict.fromkeys(cols) if c in ev.columns and c not in (key, date_col)]
    e = ev.loc[ev[key].isin(obs[key].dropna().unique()), [key, date_col, *cols]]
    m = obs[['obs_id', key, 'obs_date']].merge(e, on=key, how='inner')
    m = m[m[date_col] < m['obs_date']]
    return m.assign(age_d=(m['obs_date'] - m[date_col]) / pd.Timedelta(days=1))


def n_within(m, days):
    return m.loc[m['age_d'] <= days].groupby('obs_id').size()


def finish(obs, F, prefix, counts=(), src_start=None):
    """Align a block to obs ids, float32, counts -> 0, everything NaN where obs_date <= source start."""
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id'))
    F = F.astype({c: 'float32' for c in F.columns if pd.api.types.is_numeric_dtype(F[c])})
    for c in counts:
        if c in F.columns:
            F[c] = F[c].fillna(0)
    if src_start is not None and pd.notna(src_start):
        early = (obs['obs_date'] <= src_start).to_numpy()
        if early.any():
            F.loc[early, :] = np.nan
    F.columns = [prefix + c for c in F.columns]
    return F

In [ ]:
def annuity(amount, term_months, annual_rate):
    """Monthly annuity payment; falls back to amount / term when the rate is 0."""
    r = annual_rate / 12
    n = term_months.where(term_months > 0)
    return amount / n if r == 0 else amount * r / (1 - (1 + r) ** (-n))


def fb_contract(obs, ctx):
    """a) contract terms at obs_date. term / remaining term / instalment come from contr.end_date, an
    extraction-time value (moves on restructuring): they are in MODEL_EXCLUDE and feed only the capacity axis."""
    c = ctx['contr'].set_index('contract_number')
    o = obs.set_index('obs_id')[['contract_number', 'obs_date']].join(
        c[[x for x in ('open_date', 'end_date', 'end_date_min', 'contract_amount', 'partner_name', 'product_id',
                       'repayment_kind', 'repayment_day') if x in c.columns]], on='contract_number')
    amt = pd.to_numeric(col(o, 'contract_amount'), errors='coerce')
    end0 = o['end_date_min'] if 'end_date_min' in o.columns else o['end_date']   # earliest schedule seen
    term = ((end0 - o['open_date']).dt.days / 30.4375).round()
    F = pd.DataFrame({
        'amount': amt,
        'log_amount': np.log1p(amt.clip(lower=0)),
        'term_months': term,
        'mob_at_obs': (o['obs_date'] - o['open_date']).dt.days / 30.4375,
        'remaining_months': (o['end_date'] - o['obs_date']).dt.days / 30.4375,
        'installment_proxy': annuity(amt, term, CONFIG['INSTALMENT_RATE']),
        'repayment_day': pd.to_numeric(col(o, 'repayment_day'), errors='coerce'),
        'repayment_kind': pd.to_numeric(col(o, 'repayment_kind'), errors='coerce'),
        'partner': as_cat(col(o, 'partner_name'), upper=False),
        'product_id': as_cat(col(o, 'product_id')),
    }, index=o.index)
    return finish(obs, F, 'ct_')


def fb_dpd_history(obs, ctx):
    """b) delinquency history BEFORE the current episode (ovrd rows with act_date < entry_date; with the cure rule
    of section 4 these are exactly the previous episodes) + days in arrears now + instalment number of the entry.
    The current episode's own rows are excluded: whether they exist depends on how the episode continues."""
    o = obs[['obs_id', 'contract_number', 'entry_date']]
    p = ctx['panel']
    p = p.loc[p['contract_number'].isin(o['contract_number'].unique()),
              ['contract_number', 'act_date', 'dpd', 'start_date', 'episode_id']]
    m = o.merge(p, on='contract_number', how='inner')
    m = m[m['act_date'] < m['entry_date']]
    day = pd.Timedelta(days=1)
    m = m.assign(age_d=(m['entry_date'] - m['act_date']) / day, start_age=(m['entry_date'] - m['start_date']) / day)
    g = m.groupby('obs_id')
    F = pd.DataFrame({
        'n_prior_episodes': g['episode_id'].nunique(),
        'max_dpd_ever': g['dpd'].max(),
        'months_dpd_3m': n_within(m, 92), 'months_dpd_6m': n_within(m, 183), 'months_dpd_12m': n_within(m, 366),
        'max_dpd_6m': m[m['age_d'] <= 183].groupby('obs_id')['dpd'].max(),
        'max_dpd_12m': m[m['age_d'] <= 366].groupby('obs_id')['dpd'].max(),
        'days_since_prev_entry': g['start_age'].min(),
        'days_since_last_dpd': g['age_d'].min(),
        'prior_ep_mean_max_dpd': m.groupby(['obs_id', 'episode_id'])['dpd'].max().groupby(level='obs_id').mean(),
    })
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id'))
    for t in (30, 60, 90):
        F[f'ever{t}'] = (F['max_dpd_ever'] >= t).astype('float32')
    F['days_in_arrears'] = obs['days_in_arrears'].to_numpy(dtype='float64')
    # instalment number of the missed due date (entry_date - 1): months from the opening month (1st instalment is
    # due in the month after opening). 1 or 2 = first / second payment default.
    od = obs['contract_number'].map(ctx['contr'].set_index('contract_number')['open_date'])
    due = obs['entry_date'] - pd.Timedelta(days=1)
    inst = (due.dt.year - od.dt.year) * 12 + (due.dt.month - od.dt.month)
    F['instalment_no'] = inst.to_numpy(dtype='float64')
    F['fpd_flag'] = np.where(np.isnan(F['instalment_no']), np.nan, (F['instalment_no'] <= 2).astype('float64'))
    return finish(obs, F, 'dh_', counts=['n_prior_episodes', 'max_dpd_ever', 'months_dpd_3m', 'months_dpd_6m',
                                         'months_dpd_12m'])

In [ ]:
_ME_CACHE = {}


def month_end_snapshots(v):
    """Model features use calendar month-end snapshots only, so the snapshot cadence (month-end until 2026, then
    daily) is the same in training and at scoring. Cached per source frame."""
    mode = CONFIG['PROFILE_MONTH_END_ONLY']
    if mode is False or mode is None:
        return v
    key = id(v)
    if key not in _ME_CACHE or _ME_CACHE[key][0] is not v:
        me = v['DATE_VALUE'].dt.is_month_end.to_numpy()
        use = mode is True or v.loc[me, 'DATE_VALUE'].nunique() >= 6
        _ME_CACHE[key] = (v, v[me] if use else v)
    return _ME_CACHE[key][1]


def _asof_snapshot(obs, v, cols, shift_days=0):
    """Latest ver_df snapshot with DATE_VALUE < obs_date - shift_days, per IIN (merge_asof, [us] on both sides)."""
    left = obs[['obs_id', 'iin', 'obs_date']].dropna(subset=['iin'])
    left = left.assign(_k=(left['obs_date'] - pd.Timedelta(days=shift_days)).astype('datetime64[us]')).sort_values('_k')
    right = v.loc[v['iin'].isin(left['iin'].unique()), ['iin', 'DATE_VALUE', *cols]].sort_values('DATE_VALUE')
    m = pd.merge_asof(left, right, left_on='_k', right_on='DATE_VALUE', by='iin', direction='backward',
                      allow_exact_matches=False)
    return m.set_index('obs_id')


def norm_branch(s):
    x = as_cat(s).str.replace(r'\s+', ' ', regex=True)
    return x.str.replace('ОБЛАСТНОЙ ФИЛИАЛ', 'ОФ', regex=False).str.replace('ГОРОДСКОЙ ФИЛИАЛ', 'ГФ', regex=False)


def fb_profile(obs, ctx):
    """c) client profile: latest snapshot strictly before obs_date (+ 90-day trends, derived income)."""
    v = ctx['tables'].get('ver')
    if v is None:
        return None
    lag = CONFIG['PROFILE_LAG_DAYS']
    overlay = bool(col(obs, 'overlay', 0).astype(bool).any())       # stock overlay: the latest snapshot
    vm = v if overlay else month_end_snapshots(v)                    # model: month-end cadence only
    num = [c for c in v.columns if c not in ('iin', 'DATE_VALUE') and pd.api.types.is_numeric_dtype(v[c])]
    flags = [c for c in PROFILE_FLAGS if c in v.columns and not pd.api.types.is_numeric_dtype(v[c])]
    cats = [c for c in PROFILE_CATS if c in v.columns]
    extra = [c for c in ('DOC_EXPIRY_DATE', 'LAST_EVENT_COUNTRY') if c in v.columns]
    m = _asof_snapshot(obs, vm, num + flags + cats + extra, lag)
    new = {f'flag_{c}': flag01(m[c]) for c in flags}
    new.update({f'cat_{c}': (norm_branch(m[c]) if c == 'BRANCH' else as_cat(m[c])) for c in cats})
    new['snapshot_age_days'] = (m['obs_date'] - m['DATE_VALUE']).dt.days
    if 'DOC_EXPIRY_DATE' in m.columns:
        new['doc_days_to_expiry'] = (m['DOC_EXPIRY_DATE'] - m['obs_date']).dt.days
    if 'LAST_EVENT_COUNTRY' in m.columns:
        new['last_event_abroad'] = is_abroad_country(as_cat(m['LAST_EVENT_COUNTRY']))
    inc = [c for c in INCOME_VARS if c in num]
    new['income_proxy'] = m[inc].astype('float64').sum(axis=1, min_count=1) if inc else np.nan
    inc_c = [c for c in CAPACITY_INCOME_VARS if c in num]
    new['capacity_income'] = m[inc_c].astype('float64').sum(axis=1, min_count=1) if inc_c else np.nan
    tv = [c for c in TREND_VARS if c in num]
    if tv and vm is not v:                  # month-end cadence: the snapshot exactly 3 month-ends earlier
        tgt = (m['DATE_VALUE'].dt.to_period('M') - 3).dt.to_timestamp(how='end').dt.normalize().astype('datetime64[us]')
        r = vm.loc[vm['iin'].isin(m['iin'].dropna().unique()), ['iin', 'DATE_VALUE', *tv]]
        r = r.assign(_d=r['DATE_VALUE'].dt.normalize().astype('datetime64[us]')).drop(columns='DATE_VALUE')
        r = r.drop_duplicates(['iin', '_d'], keep='last')
        m90 = (m[['iin']].assign(_d=tgt).reset_index().merge(r, on=['iin', '_d'], how='left')
                .set_index('obs_id').reindex(m.index))
    elif tv:                                # as-of merge at obs_date - 90 days
        m90 = _asof_snapshot(obs, v, tv, lag + 90).reindex(m.index)
    if tv:                                  # deltas / ratios
        for c in tv:
            now, then = m[c].astype('float64'), m90[c].astype('float64')
            new[f'd90_{c}'] = now - then
            new[f'r90_{c}'] = now / then.where(then.abs() > 0)
    F = pd.concat([m[num], pd.DataFrame(new, index=m.index)], axis=1)
    return finish(obs, F, 'pr_')

In [ ]:
def fb_internal_debits(obs, ctx):
    """d) vnpt: the bank's own payment demands (ВнПТ) - payment discipline. Key: dogovor -> contract; demands on
    accounts that are not a portfolio contract fall back to the client's IIN (decided against the portfolio, not
    against the obs set, so a demand is attributed the same way whatever is being scored)."""
    t = ctx['tables'].get('vnpt')
    if t is None:
        return None
    cols = ['accept_date', 'close_date', 'amount_sum']
    in_c = t['contract_number'].isin(ctx['portfolio_contracts'])
    m = pd.concat([events_before(obs, t[in_c], 'contract_number', 'accept_date', cols),
                   events_before(obs, t[~in_c & t['iin'].notna()], 'iin', 'accept_date', cols)], ignore_index=True)
    m = m.assign(is_open=m['close_date'].isna() | (m['close_date'] >= m['obs_date']))
    closed = m[~m['is_open']]
    closed = closed.assign(ttc=(closed['close_date'] - closed['accept_date']) / pd.Timedelta(days=1),
                           same_day=(closed['close_date'].dt.normalize() == closed['accept_date'].dt.normalize()).astype('float32'))
    closed = closed.assign(within3=(closed['ttc'] <= 3).astype('float32'))
    g, gc, op = m.groupby('obs_id'), closed.groupby('obs_id'), m[m['is_open']].groupby('obs_id')
    r90 = m[m['age_d'] <= 90]
    per_day = r90.groupby([r90['obs_id'], r90['accept_date'].dt.normalize()]).size()
    F = pd.DataFrame({
        'n_30d': n_within(m, 30), 'n_90d': n_within(m, 90), 'n_180d': n_within(m, 180),
        'n_open': op.size(), 'open_amount': op['amount_sum'].sum(),
        'amount_90d': r90.groupby('obs_id')['amount_sum'].sum(),
        'ttc_mean': gc['ttc'].mean(), 'ttc_max': gc['ttc'].max(),
        'ttc_last': closed.sort_values('accept_date').groupby('obs_id')['ttc'].last(),
        'share_same_day': gc['same_day'].mean(), 'share_within_3d': gc['within3'].mean(),
        'days_since_last': g['age_d'].min(),
        'multi_same_day_90d': (per_day >= 2).groupby(level=0).sum(),          # partial-payment proxy
    })
    return finish(obs, F, 'vn_', counts=['n_30d', 'n_90d', 'n_180d', 'n_open', 'open_amount', 'amount_90d',
                                         'multi_same_day_90d'], src_start=SRC_START.get('vnpt'))


def fb_enforcement(obs, ctx):
    """e) vneshpt_ot: third-party collection orders ИР3/ИР4 on the client's accounts (enforcement)."""
    t = ctx['tables'].get('vneshpt_ot')
    if t is None:
        return None
    m = events_before(obs, t, 'iin', 'accept_date',
                      ['close_date', 'amount_sum', 'opertype_name', 'descr', 'k2_requirement_group_id'])
    ops = as_cat(col(m, 'opertype_name'))
    rule_types = {str(t).strip().upper() for t in CONFIG['ENFORCEMENT_OPERTYPES']}
    m = m.assign(is_open=m['close_date'].isna() | (m['close_date'] >= m['obs_date']),
                 ir4=ops.eq('ИР4').astype('float32'),
                 rule_type=ops.isin(rule_types) if 'opertype_name' in m.columns else True,
                 ispoln=col(m, 'descr').astype('str').str.contains('исполнительн', case=False, na=False).astype('float32'),
                 grp=col(m, 'k2_requirement_group_id'))
    g, op = m.groupby('obs_id'), m[m['is_open']].groupby('obs_id')
    rule = m[m['is_open'] & m['rule_type'] & (m['age_d'] <= CONFIG['ENFORCEMENT_MAX_AGE_DAYS'])]
    F = pd.DataFrame({
        'n_ever': g.size(), 'n_365d': n_within(m, 365), 'n_90d': n_within(m, 90),
        'n_open': op.size(), 'open_amount': op['amount_sum'].sum(), 'max_amount': g['amount_sum'].max(),
        'days_since_last': g['age_d'].min(), 'n_groups': g['grp'].nunique(),
        'share_ir4': g['ir4'].mean(), 'share_enforcement_descr': g['ispoln'].mean(),
        'n_open_rule': rule.groupby('obs_id').size(),     # R05: open ИР3/ИР4 orders accepted within the window
    })
    return finish(obs, F, 'en_', counts=['n_ever', 'n_365d', 'n_90d', 'n_open', 'open_amount', 'n_groups', 'n_open_rule'],
                  src_start=SRC_START.get('vneshpt_ot'))


def fb_ext_debits(obs, ctx):
    """f) vneshpt: direct-debit demands sent to the client's OTHER banks (status events before obs)."""
    t = ctx['tables'].get('vneshpt')
    if t is None:
        return None
    m = events_before(obs, t, 'iin', 'ev_date', ['reference', 'longname', 'reason', 'created_date', 'debt'])
    m = m.assign(c_age=(m['obs_date'] - m['created_date']) / pd.Timedelta(days=1))
    w90 = m[m['age_d'] <= 90]
    debt_ref = w90.groupby(['obs_id', 'reference'])['debt'].max()
    cols = {
        'n_demands_30d': m[m['c_age'] <= 30].groupby('obs_id')['reference'].nunique(),
        'n_demands_90d': m[m['c_age'] <= 90].groupby('obs_id')['reference'].nunique(),
        'n_refs_ever': m.groupby('obs_id')['reference'].nunique(),
        'n_banks_90d': w90.groupby('obs_id')['longname'].nunique(),
        'n_banks_ever': m.groupby('obs_id')['longname'].nunique(),
        'debt_sum_90d': debt_ref.groupby(level='obs_id').sum(),
        'debt_max_90d': debt_ref.groupby(level='obs_id').max(),
        'days_since_last': m.groupby('obs_id')['age_d'].min(),
    }
    for reason, label in ctx['ext_reasons'].items():
        cols[f'share_{label}_90d'] = (w90['reason'] == reason).astype('float32').groupby(w90['obs_id']).mean()
    return finish(obs, pd.DataFrame(cols), 'ex_',
                  counts=['n_demands_30d', 'n_demands_90d', 'n_refs_ever', 'n_banks_90d', 'n_banks_ever', 'debt_sum_90d'],
                  src_start=SRC_START.get('vneshpt'))

In [ ]:
# 'went abroad' in CRM notes. Bare 'уехал' is NOT enough (work trip, village...): it needs a foreign destination.
# No \\w / look-behind: pandas 3 pyarrow strings use RE2 (ASCII \\w).
ABROAD_RE = (r'загран|за границ|за рубеж|выехал[аи]? из (?:РК|страны|Казахстан)'
             r'|уехал[аи]? (?:в|на) (?:Росси|РФ|Москв|Турци|Кита|Коре|ОАЭ|Дуба|Узбекистан|Кыргыз|Киргиз|Европ|США|'
             r'Америк|Германи|Польш|Чехи|Израил|Грузи|Армени|Таиланд)')


def fb_calls(obs, ctx):
    """g) ocrm_call split into CALLS (dedup callid) and ACTIVITIES (dedup activityid), key num_dbz -> contract."""
    calls, acts = ctx['tables'].get('ocrm_calls'), ctx['tables'].get('ocrm_acts')
    if calls is None and acts is None:
        return None
    cols = {}
    third = pd.Series(dtype='float64')
    abroad = pd.DataFrame(columns=['obs_id', 'age_d'])
    if calls is not None:
        mc = events_before(obs, calls, 'contract_number', 'call_ts',
                           ['callid', 'talktime', 'result_predl', 'tnumb_to', 'tsptp_date', 'call_comment'])
        res = col(mc, 'result_predl').astype('str').str.strip()
        mc = mc.assign(contacted=pd.to_numeric(col(mc, 'talktime'), errors='coerce').gt(0).astype('float32'),
                       noans=res.eq('Нет ответа').astype('float32'),
                       ptp=res.str.upper().eq('PTP').astype('float32'),
                       third=res.str.contains('трет|3 лиц', case=False, na=False),
                       abroad=col(mc, 'call_comment').astype('str').str.contains(ABROAD_RE, case=False, na=False))
        y = mc[mc['age_d'] <= 365]
        gy = y.groupby('obs_id')
        cols.update({
            'n_calls_30d': n_within(mc, 30), 'n_calls_90d': n_within(mc, 90), 'n_calls_365d': gy.size(),
            'n_contacted_365d': gy['contacted'].sum(), 'contact_rate_365d': gy['contacted'].mean(),
            'n_noanswer_365d': gy['noans'].sum(), 'n_ptp_calls_365d': gy['ptp'].sum(),
            'n_numbers_dialled': mc.groupby('obs_id')['tnumb_to'].nunique() if 'tnumb_to' in mc.columns else None,
            'days_since_call': mc.groupby('obs_id')['age_d'].min(),
            'days_since_contact': mc[mc['contacted'] > 0].groupby('obs_id')['age_d'].min(),
        })
        third = y[y['third']].groupby('obs_id').size()
        abroad = mc.loc[mc['abroad'], ['obs_id', 'age_d']]
        if 'tsptp_date' in mc.columns:       # broken promise: still overdue after the promised date
            pt = mc.loc[mc['tsptp_date'].notna() & (mc['tsptp_date'] < mc['obs_date']),
                        ['obs_id', 'contract_number', 'callid', 'tsptp_date', 'obs_date']].drop_duplicates(['obs_id', 'callid'])
            x = pt.merge(ctx['panel'][['contract_number', 'start_date', 'act_date']], on='contract_number')
            x = x[(x['act_date'] < x['obs_date']) & (x['start_date'] <= x['tsptp_date']) & (x['act_date'] > x['tsptp_date'])]
            cols['n_ptp_due'] = pt.groupby('obs_id').size()
            cols['n_ptp_broken'] = x.drop_duplicates(['obs_id', 'callid']).groupby('obs_id').size()
    if acts is not None:
        ma = events_before(obs, acts, 'contract_number', 'act_ts',
                           ['activ_sost', 'activ_result', 'activ_result_dtl', 'ocrm_tsdebt_id'])
        ares = col(ma, 'activ_result').astype('str')
        ma = ma.assign(canceled=col(ma, 'activ_sost').astype('str').str.strip().str.lower().eq('canceled').astype('float32'),
                       ptp=ares.str.contains('Обещание', case=False, na=False).astype('float32'),
                       third=ares.str.contains('3 лиц', case=False, na=False),
                       abroad=col(ma, 'activ_result_dtl').astype('str').str.contains(ABROAD_RE, case=False, na=False))
        ya = ma[ma['age_d'] <= 365]
        ga = ya.groupby('obs_id')
        cols.update({'n_acts_365d': ga.size(), 'share_canceled_365d': ga['canceled'].mean(),
                     'n_ptp_acts_365d': ga['ptp'].sum(),
                     'n_debt_episodes': ma.groupby('obs_id')['ocrm_tsdebt_id'].nunique() if 'ocrm_tsdebt_id' in ma.columns else None})
        third = third.add(ya[ya['third']].groupby('obs_id').size(), fill_value=0)
        abroad = pd.concat([abroad, ma.loc[ma['abroad'], ['obs_id', 'age_d']]], ignore_index=True)
    cols['n_third_party_365d'] = third
    cols['abroad_kw'] = pd.Series(dtype='float32')
    if len(abroad):                          # only recent notes count (an old episode's note is stale)
        cols['days_since_abroad_note'] = abroad.astype({'age_d': 'float64'}).groupby('obs_id')['age_d'].min()
        cols['abroad_kw'] = (cols['days_since_abroad_note'] <= CONFIG['ABROAD_NOTE_DAYS']).astype('float32')
    F = pd.DataFrame({k: v for k, v in cols.items() if v is not None})
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id'))
    counts = [c for c in F.columns if c.startswith('n_')] + ['abroad_kw']
    F[[c for c in counts if c in F.columns]] = F[[c for c in counts if c in F.columns]].fillna(0)
    F['n_ptp_365d'] = F.get('n_ptp_calls_365d', 0) + F.get('n_ptp_acts_365d', 0)
    if 'n_ptp_due' in F.columns:
        F['ptp_broken_share'] = F['n_ptp_broken'] / F['n_ptp_due'].where(F['n_ptp_due'] > 0)
    return finish(obs, F, 'cl_', src_start=SRC_START.get('ocrm_call'))


def fb_app_geo(obs, ctx):
    """h) Homebank logins (pre-aggregated per client-day): activity, trend, devices, biometrics, abroad."""
    d = ctx['tables'].get('geo_daily')
    if d is None:
        return None
    m = events_before(obs, d, 'iin', 'day', ['n', 'n_bio', 'n_night', 'n_valid', 'n_abroad', 'premium', 'app_major'])
    w30, w90 = m[m['age_d'] <= 30], m[m['age_d'] <= 90]
    s30 = w30.groupby('obs_id')[['n', 'n_valid', 'n_abroad']].sum()
    s90 = w90.groupby('obs_id')[['n', 'n_bio', 'n_night', 'n_valid', 'n_abroad']].sum()
    cols = {
        'logins_7d': m[m['age_d'] <= 7].groupby('obs_id')['n'].sum(), 'logins_30d': s30['n'], 'logins_90d': s90['n'],
        'logins_31_90d': w90[w90['age_d'] > 30].groupby('obs_id')['n'].sum(),
        'active_days_30d': w30.groupby('obs_id').size(),
        'days_since_login': m.groupby('obs_id')['age_d'].min(),
        'share_bio_90d': s90['n_bio'] / s90['n'], 'share_night_90d': s90['n_night'] / s90['n'],
        'share_valid_coords_90d': s90['n_valid'] / s90['n'],
        'share_abroad_30d': s30['n_abroad'] / s30['n_valid'].where(s30['n_valid'] > 0),
        'share_abroad_90d': s90['n_abroad'] / s90['n_valid'].where(s90['n_valid'] > 0),
        'premium_device_90d': w90.groupby('obs_id')['premium'].max(),
        'app_major_last': m.sort_values('day').groupby('obs_id')['app_major'].last(),
    }
    for name, key in (('n_devices_90d', 'geo_dev'), ('n_ips_90d', 'geo_ip')):
        t = ctx['tables'].get(key)
        if t is not None:
            k = t.columns[-1]
            x = events_before(obs, t, 'iin', 'day', [k])
            cols[name] = x[x['age_d'] <= 90].groupby('obs_id')[k].nunique()
    F = pd.DataFrame(cols).reindex(pd.Index(obs['obs_id'], name='obs_id'))
    counts = ['logins_7d', 'logins_30d', 'logins_90d', 'logins_31_90d', 'active_days_30d', 'n_devices_90d', 'n_ips_90d']
    F[[c for c in counts if c in F.columns]] = F[[c for c in counts if c in F.columns]].fillna(0)
    F['trend_30_vs_prev60'] = (F['logins_30d'] + 1) / (F['logins_31_90d'] / 2 + 1)      # smoothed ratio
    return finish(obs, F, 'geo_', src_start=SRC_START.get('geo'))

In [ ]:
def fb_exclusions(obs, ctx):
    """i) exclusion lists active at obs_date (is_active_fl == 1 and not updated after obs)."""
    e = ctx['tables'].get('excl')
    if e is None:
        return None
    overlay = bool(col(obs, 'overlay', 0).astype(bool).any())
    m = obs[['obs_id', 'iin', 'obs_date']].dropna(subset=['iin']).merge(
        e[['iin', 'excl_category', 'is_active_fl', 'updated_date']], on='iin')
    # excl is a CURRENT snapshot. An entry without updated_date is known today (stock overlay) but not at a past
    # obs_date: using it in history would be look-ahead (clients may be listed BECAUSE they defaulted).
    known = (m['updated_date'] < m['obs_date']) | (m['updated_date'].isna() & overlay)
    act = m[known & m['is_active_fl'].eq(1)]
    piv = (act.groupby(['obs_id', 'excl_category']).size().unstack(fill_value=0).clip(upper=1)
              .reindex(columns=EXCL_ALL, fill_value=0).add_prefix('active_'))
    F = pd.concat([piv, m[known].groupby('obs_id').size().clip(upper=1).rename('listed_ever')], axis=1)
    F = F.reindex(pd.Index(obs['obs_id'], name='obs_id')).fillna(0)
    F['active_any'] = F[[f'active_{c}' for c in EXCL_ALL]].max(axis=1)
    F['active_social_list'] = F[[f'active_{c}' for c in EXCL_SOCIAL]].max(axis=1)   # R04 (named lists only)
    return finish(obs, F, 'exc_')


def fb_bankruptcy(obs, ctx):
    """j) refused out-of-court bankruptcy applications before obs_date."""
    b = ctx['tables'].get('bankrupt')
    if b is None:
        return None
    m = events_before(obs, b, 'iin', 'event_date', ['refused'])
    g = m.groupby('obs_id')
    F = pd.DataFrame({'n_apps': g.size(), 'n_refused': g['refused'].sum(), 'days_since_last': g['age_d'].min()})
    F['flag_365d'] = (F['days_since_last'] <= 365).astype('float32')
    return finish(obs, F, 'bk_', counts=['n_apps', 'n_refused', 'flag_365d'], src_start=SRC_START.get('bankrupt'))


def fb_applications(obs, ctx):
    """k1) dm: the approved application of this contract (process_ref), dated before obs_date."""
    a = ctx['tables'].get('dm')
    if a is None:
        return None
    x = obs[['obs_id', 'process_ref', 'obs_date']].dropna(subset=['process_ref']).merge(a, on='process_ref')
    x = x[x['app_date'].isna() | (x['app_date'] < x['obs_date'])]
    x = x.sort_values('app_date', na_position='first').drop_duplicates('obs_id', keep='last').set_index('obs_id')
    feats = [c for c in a.columns if c not in ('process_ref', 'app_date')]
    F = x[feats].assign(days_app_to_obs=(x['obs_date'] - x['app_date']).dt.days)
    return finish(obs, F, 'app_')


def fb_rejections(obs, ctx):
    """k2) dm_rej: rejected applications of the client before obs_date (credit hunger)."""
    r = ctx['tables'].get('dm_rej')
    if r is None:
        return None
    m = events_before(obs, r, 'iin', 'app_date', ['request_amount'])
    F = pd.DataFrame({'n_30d': n_within(m, 30), 'n_90d': n_within(m, 90), 'n_180d': n_within(m, 180),
                      'n_ever': m.groupby('obs_id').size(), 'days_since_last': m.groupby('obs_id')['age_d'].min(),
                      'sum_requested_180d': m[m['age_d'] <= 180].groupby('obs_id')['request_amount'].sum()})
    return finish(obs, F, 'rej_', counts=['n_30d', 'n_90d', 'n_180d', 'n_ever', 'sum_requested_180d'],
                  src_start=SRC_START.get('dm_rej'))


def fb_pkb_monitoring(obs, ctx):
    """l) PKB monitoring aggregate (long format from the SQL). Only for obs on/after the aggregate's snapshot or
    for the stock overlay; NaN otherwise (history starts 2026-09-07 -> overlay / rules only)."""
    p = ctx['tables'].get('pkb')
    if p is None:
        return None
    avail = (obs['obs_date'] >= ctx['pkb_available_from']).to_numpy() | col(obs, 'overlay', 0).astype(bool).to_numpy()
    agg = p.groupby(['iin', 'code']).agg(cnt_7d=('cnt_7d', 'sum'), cnt_30d=('cnt_30d', 'sum'), cnt_60d=('cnt_60d', 'sum'),
                                         cnt_all=('cnt_all', 'sum'), last_ts=('last_ts', 'max'))
    wide = {}
    for w in ('7d', '30d', '60d', 'all'):
        u = agg[f'cnt_{w}'].unstack('code').reindex(columns=PKB_CODES)
        wide.update({f'{c}_{w}': u[c] for c in PKB_CODES})
    W = pd.DataFrame(wide)
    last = agg['last_ts'].unstack('code')
    o = obs.loc[avail, ['obs_id', 'iin', 'obs_date']]
    counts = o[['iin']].join(W, on='iin').drop(columns='iin').fillna(0)
    days = {}
    for c in PKB_KEY_CODES:                 # a code without any event stays NaN (never a float - datetime clash)
        lt = pd.to_datetime(o['iin'].map(last[c]), errors='coerce') if c in last.columns else pd.Series(pd.NaT, index=o.index)
        days[f'days_since_{c}'] = (o['obs_date'] - lt.astype('datetime64[us]')).dt.days
    days = pd.DataFrame(days, index=o.index)
    F = pd.concat([counts, days], axis=1).set_index(o['obs_id'])
    return finish(obs, F, 'pkb_')

In [ ]:
BLOCKS = [  # name, function, source table, role: 'model' = always, 'auto' = by source start, 'overlay' = rules only
    dict(name='contract', fn=fb_contract, source='contr', role='model'),
    dict(name='dpd_history', fn=fb_dpd_history, source='ovrd', role='model'),
    dict(name='profile', fn=fb_profile, source='ver_df_all_2024_2026', role='auto'),
    dict(name='internal_debits', fn=fb_internal_debits, source='vnpt', role='auto'),
    dict(name='enforcement', fn=fb_enforcement, source='vneshpt_ot', role='auto'),
    dict(name='ext_debits', fn=fb_ext_debits, source='vneshpt', role='auto'),
    dict(name='calls', fn=fb_calls, source='ocrm_call', role='auto'),
    dict(name='app_geo', fn=fb_app_geo, source='geo', role='auto'),
    dict(name='exclusions', fn=fb_exclusions, source='excl', role='overlay'),     # snapshot list, no history
    dict(name='bankruptcy', fn=fb_bankruptcy, source='bankrupt', role='auto'),
    dict(name='applications', fn=fb_applications, source='dm', role='auto'),
    dict(name='rejections', fn=fb_rejections, source='dm_rej', role='auto'),
    dict(name='pkb_monitoring', fn=fb_pkb_monitoring, source='hes_pkb_agg', role='overlay'),
]

# context shared by the blocks
_ext_reasons = {}
if 'vneshpt' in TBL and 'reason' in TBL['vneshpt'].columns:
    for _i, _r in enumerate(TBL['vneshpt']['reason'].value_counts().head(5).index):
        _mm = re.match(r'\s*(\d+)', str(_r))
        _lbl = f'reason_{_mm.group(1)}' if _mm else f'reason_k{_i}'
        _ext_reasons[_r] = _lbl if _lbl not in _ext_reasons.values() else f'{_lbl}_{_i}'
CTX = dict(contr=contracts, panel=panel, tables=TBL, ext_reasons=_ext_reasons,
           portfolio_contracts=set(contracts['contract_number']),
           pkb_available_from=TBL.get('pkb_snapshot', pd.NaT))


BLOCK_FAILURES = []           # reported in run_summary.json


def run_blocks(obs, label=''):
    """Run every registered block on an observation frame -> ({block: DataFrame}, log table)."""
    out, rows = {}, []
    for b in BLOCKS:
        t = time.perf_counter()
        try:
            F = b['fn'](obs, CTX) if len(obs) else None
            status = 'ok' if F is not None else ('source missing' if len(obs) else 'no observations')
        except Exception as e:
            if CONFIG['STRICT_BLOCKS'] or b['source'] in REQUIRED_TABLES:
                raise
            F, status = None, f'FAILED: {type(e).__name__}: {e}'
            BLOCK_FAILURES.append(dict(block=b['name'], run=label, error=f'{type(e).__name__}: {e}'[:300]))
            msg = f"WARN  block {b['name']} ({label}) FAILED -> skipped; dependent rules are OFF\n{traceback.format_exc(limit=3)}"
            print(msg)
            print(msg.splitlines()[0], file=sys.stderr)
        if F is not None:
            out[b['name']] = F
        start, end = SRC_RANGE.get(b['source'], (pd.NaT, pd.NaT))
        rows.append(dict(block=b['name'], source=b['source'], role=b['role'], status=status,
                         n_features=0 if F is None else F.shape[1],
                         coverage=np.nan if F is None or F.shape[1] == 0 else round(float(F.notna().any(axis=1).mean()), 3),
                         src_start=start, src_end=end, seconds=round(time.perf_counter() - t, 2)))
    return out, pd.DataFrame(rows).set_index('block')


with timer(f'feature blocks at obs_date ({len(OBS):,} observations: every checkpoint + the stock on the cut)'):
    XB, BLOCK_LOG = run_blocks(OBS, 'observations')
with timer('feature blocks at score date (current stock overlay, latest info)'):
    XN, _ = run_blocks(OBS_NOW, 'stock overlay')
show(BLOCK_LOG, 'Feature blocks as of obs_date (coverage = share of observations with any non-null feature)')

### 5.1 As-of self-test (leakage check)
This test rebuilds the data "as the DWH looked at the start of day T": every event dated on or after T is removed, and later closing dates become unknown. It then recomputes every block for the observations with `obs_date = T` (two entry dates and two later-checkpoint dates) and requires **identical** features. Any difference means a block reads information from after `obs_date`.

What this test cannot see:
- `excl` is a snapshot list with no history (rules only; history uses only entries whose `updated_date` is before `obs_date`). PKB is a pre-aggregated snapshot used as an overlay only.
- **Static extracts are not versioned:** `contr` (its `end_date` moves on restructuring, hence `MODEL_EXCLUDE`) and `dm` (application fields; hence the dm allow / deny list in section 3). A lifecycle field in them would pass this test.
- `ovrd` keeps one monthly maximum, so a month whose maximum is reached after T is excluded entirely, although part of it was known at T. That loses some information but never leaks.

In [ ]:
def truncate_tables(T, ob):
    """The prepared sources as they looked at the start of day T (events >= T removed, later closes unknown),
    restricted first to the keys of the tested episodes (cheap at full scale)."""
    cut = {'ver': 'DATE_VALUE', 'vneshpt': 'ev_date', 'ocrm_calls': 'call_ts', 'ocrm_acts': 'act_ts',
           'geo_daily': 'day', 'geo_dev': 'day', 'geo_ip': 'day', 'dm_rej': 'app_date', 'bankrupt': 'event_date'}
    keys = {'iin': set(ob['iin'].dropna()), 'contract_number': set(ob['contract_number']),
            'process_ref': set(ob['process_ref'].dropna())}
    out = {}
    for k, v in TBL.items():
        if not isinstance(v, pd.DataFrame):     # e.g. the PKB snapshot date
            out[k] = v
            continue
        if k in cut or k in ('vnpt', 'vneshpt_ot', 'dm'):
            m = np.zeros(len(v), dtype=bool)
            for kc, ks in keys.items():
                if kc in v.columns:
                    m |= v[kc].isin(ks).to_numpy()
            v = v[m]
        if k in cut and cut[k] in v.columns:
            out[k] = v[v[cut[k]] < T]
        elif k in ('vnpt', 'vneshpt_ot'):
            x = v[v['accept_date'] < T]
            out[k] = x.assign(close_date=x['close_date'].where(x['close_date'] < T))
        elif k == 'dm':
            out[k] = v[v['app_date'].isna() | (v['app_date'] < T)]
        else:                                     # excl (snapshot list) / pkb (overlay only): not testable
            out[k] = v
    return out


def _same(a, b):
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        return np.isclose(a.to_numpy(dtype='float64'), b.to_numpy(dtype='float64'), equal_nan=True, rtol=1e-6)
    return ((a.astype('str') == b.astype('str')) | (a.isna() & b.isna())).to_numpy()


_l = OB[OB['labeled']]
_dates = list(_l.loc[_l['checkpoint'] == CP0, 'obs_date'].value_counts().head(2).index)
_dates += [d for d in _l.loc[_l['checkpoint'] > CP0, 'obs_date'].value_counts().index if d not in _dates][:2]
_rows = []
for _T in _dates:
    _ob = OBS[OBS['obs_date'] == _T]
    _ctx = dict(CTX, tables=truncate_tables(_T, _ob), panel=panel[panel['act_date'] < _T])
    for _b in BLOCKS:
        if _b['name'] in ('exclusions', 'pkb_monitoring') or _b['name'] not in XB:
            continue
        _A = XB[_b['name']].loc[_ob['obs_id']]
        _B = _b['fn'](_ob, _ctx).reindex(index=_A.index, columns=_A.columns)
        _bad = [c for c in _A.columns if not _same(_A[c], _B[c]).all()]
        _rows.append(dict(obs_date=_T, block=_b['name'], n_obs=len(_ob), n_features=_A.shape[1],
                          n_mismatch=len(_bad), mismatching=', '.join(_bad[:5])))
ASOF_TEST = pd.DataFrame(_rows)
ASOF_OK = bool(len(ASOF_TEST)) and int(ASOF_TEST['n_mismatch'].sum()) == 0
show(ASOF_TEST.groupby('block').agg(n_obs=('n_obs', 'sum'), n_features=('n_features', 'max'),
                                    n_mismatch=('n_mismatch', 'sum'), mismatching=('mismatching', 'max')),
     f"As-of self-test on {len(_dates)} entry dates: {'PASS - no look-ahead' if ASOF_OK else 'FAIL - see mismatching features'}")
if not ASOF_OK:
    print('WARNING: a block uses information from after obs_date - fix it before trusting the model.')

## 6. Feature matrix, time split and feature selection
- **Assembly.** All blocks plus cross-block derived features. The monthly payment is `pay = max(MONTH_LOAN_PAY_AMOUNT, annuity instalment)`, where the annuity uses the contract amount, the term from the earliest `end_date` and `INSTALMENT_RATE` (an amount / term proxy would ignore interest and understate the payment by 1.2-1.9x).
  - `capacity_ratio` = capacity income / pay. Capacity income = inflows from other banks, Kaspi and Halyk transfers; protected social and pension inflows are left out because they cannot be debited.
  - `depo_cover` = `DEPO_SUM` / pay, `debitable_cover` = `debitable_funds` / pay.
  - `debitable_funds` = max(0, min(50% of deposits, deposits - 2 x the subsistence minimum, ПМ)). This is the legal debit limit from 21.10.2026.
  - Because the term comes from `contr.end_date` (an extraction-time value), the instalment proxy and the three ratios built on it are in `MODEL_EXCLUDE`: they drive only the capacity axis of the segmentation.
- **Split.** By **episode**: OOT = the episodes whose entry month is one of the last `OOT_MONTHS` labelled entry months. The other episodes are split into train and valid, stratified by `bad` and **grouped by contract**, so all checkpoints of an episode, and all episodes of a contract, sit on one side. A later checkpoint whose 6-month window is not complete yet is `unlabeled`.
- **`MODEL_BLOCKS='auto'`.** A block feeds the model only if its source starts on or before the **10th percentile of the train `obs_date`**. Late sources (geo since 2025-09, vnpt since 2026-01, vneshpt since 2026-03, PKB since 2026-09) would otherwise teach the model "source present = recent period". They stay in the overlay rules.
- **Selection.** Features are dropped if they are:
  - more than 98% null or constant on train,
  - unstable, with PSI(train -> OOT) above `PSI_DROP_THRESHOLD`,
  - leakage (`close_date`, `status_name`, `restructing_fl`) or listed in `MODEL_EXCLUDE`.

In [ ]:
DERIVED = ['capacity_ratio', 'depo_cover', 'debitable_funds', 'debitable_cover']
_C = CONFIG['COMPLIANCE']


def assemble(blocks, obs):
    idx = pd.Index(obs['obs_id'], name='obs_id')
    parts = [F for F in blocks.values() if F is not None and F.shape[1]]
    X = pd.concat(parts, axis=1).reindex(idx) if parts else pd.DataFrame(index=idx)
    inst = col(X, 'ct_installment_proxy').astype('float64')
    pay = pd.concat([inst, col(X, 'pr_MONTH_LOAN_PAY_AMOUNT').astype('float64')], axis=1).max(axis=1)
    pay = pay.where(pay > 0)
    depo = col(X, 'pr_DEPO_SUM').astype('float64')
    d = {'capacity_ratio': col(X, 'pr_capacity_income').astype('float64') / pay,
         'depo_cover': depo / pay,
         'debitable_funds': pd.concat([_C['DEBIT_MAX_SHARE'] * depo, depo - _C['DEBIT_PROTECTED']], axis=1)
                              .min(axis=1, skipna=False).clip(lower=0)}
    d['debitable_cover'] = d['debitable_funds'] / pay
    return pd.concat([X, pd.DataFrame(d, index=X.index).astype('float32')], axis=1)


X = assemble(XB, OBS)
XNOW = assemble(XN, OBS_NOW)
FEATURE_BLOCK = {c: b for b, F in XB.items() for c in F.columns} | {c: 'derived' for c in DERIVED}

# ---- time split by episode (all checkpoints of an episode on the same side) -----------------------
_f = OB.loc[FIRST]
_lab_months = sorted(_f.loc[_f['labeled'], 'entry_month'].unique())
OOT_MONTHS = set(_lab_months[-CONFIG['OOT_MONTHS']:])
_ep_oot = _f.loc[_f['labeled'] & _f['entry_month'].isin(OOT_MONTHS), 'episode_id']
_dev = _f[_f['labeled'] & ~_f['entry_month'].isin(OOT_MONTHS)]
_sgk = StratifiedGroupKFold(n_splits=max(2, round(1 / CONFIG['VALID_SHARE'])), shuffle=True,
                            random_state=CONFIG['RANDOM_STATE'])
_tr, _va = next(_sgk.split(np.zeros(len(_dev)), _dev['bad'], groups=_dev['contract_number']))
_ep_sample = pd.concat([pd.Series('train', index=_dev['episode_id'].iloc[_tr]),
                        pd.Series('valid', index=_dev['episode_id'].iloc[_va]),
                        pd.Series('oot', index=_ep_oot)])
_sample = OB['episode_id'].map(_ep_sample).fillna('unlabeled').astype('str')
_sample[~OB['labeled']] = 'unlabeled'
_sample[OB['is_stock']] = 'stock'
OB['sample'] = _sample
IDS = {s: OB.index[OB['sample'] == s] for s in ('train', 'valid', 'oot')}
IDS_FIRST = {s: v.intersection(FIRST) for s, v in IDS.items()}       # first checkpoint only (one row per episode)
TRAIN_P10 = OB.loc[IDS['train'], 'obs_date'].quantile(0.10)

_split = OB.assign(_bad=OB['bad'].where(OB['labeled'])).groupby('sample').agg(
                                  n_obs=('bad', 'size'), n_episodes=('episode_id', 'nunique'), bad_rate=('_bad', 'mean'),
                                  first_entry_month=('entry_month', 'min'), last_entry_month=('entry_month', 'max'))
show(_split.reindex(['train', 'valid', 'oot', 'unlabeled', 'stock']),
     'Time split by episode (valid is grouped by contract; n_obs counts every checkpoint)')

In [ ]:
def block_used(b):
    if isinstance(CONFIG['MODEL_BLOCKS'], (list, tuple, set)):
        return b['name'] in CONFIG['MODEL_BLOCKS'], 'listed in MODEL_BLOCKS'
    if b['role'] == 'model':
        return True, 'core block'
    if b['role'] == 'overlay':
        return False, 'overlay only (snapshot list / short history)'
    s = SRC_START.get(b['source'])
    if s is None or pd.isna(s):
        return False, 'no source dates'
    ok = s <= TRAIN_P10
    return ok, f"source starts {s:%Y-%m-%d} {'<=' if ok else '>'} train P10 obs_date {TRAIN_P10:%Y-%m-%d}"


_rows = []
for _b in BLOCKS:
    _F = XB.get(_b['name'])
    _used, _why = block_used(_b) if _F is not None else (False, 'source missing / block skipped')
    _cov = lambda F, ids: round(float(F.loc[ids].notna().any(axis=1).mean()), 3) if F is not None and F.shape[1] and len(ids) else np.nan
    _rows.append(dict(block=_b['name'], source=_b['source'], src_start=SRC_START.get(_b['source'], pd.NaT),
                      n_features=0 if _F is None else _F.shape[1], coverage_train=_cov(_F, IDS['train']),
                      coverage_oot=_cov(_F, IDS['oot']),
                      coverage_stock=_cov(_F, STOCK_IDS),
                      coverage_stock_overlay=_cov(XN.get(_b['name']), XN[_b['name']].index) if _b['name'] in XN else np.nan,
                      used_in_model=_used, reason=_why))
BLOCK_COVERAGE = pd.DataFrame(_rows).set_index('block')
MODEL_BLOCKS_USED = BLOCK_COVERAGE.index[BLOCK_COVERAGE['used_in_model']].tolist()
show(BLOCK_COVERAGE, f"Block coverage and model usage (MODEL_BLOCKS={CONFIG['MODEL_BLOCKS']!r})")

In [ ]:
LEAK_RE = re.compile('|'.join(LEAKAGE_COLS) + r'|^(bad|self_cure30|roll31_60d|labeled|max_dpd)$', re.I)
_cand = [c for b in MODEL_BLOCKS_USED for c in XB[b].columns]
if 'profile' in MODEL_BLOCKS_USED:
    _cand += DERIVED
_drop = [dict(feature=c, reason='excluded (CONFIG / leakage guard)', value=np.nan)
         for c in _cand if c in CONFIG['MODEL_EXCLUDE'] or LEAK_RE.search(c)]
_cand = [c for c in _cand if c not in CONFIG['MODEL_EXCLUDE'] and not LEAK_RE.search(c)]

_Xtr = X.loc[IDS['train'], _cand]
_null = _Xtr.isna().mean()
_nun = _Xtr.nunique(dropna=True)
_drop += [dict(feature=c, reason=f"null share > {CONFIG['MAX_NULL_SHARE']}", value=_null[c])
          for c in _cand if _null[c] > CONFIG['MAX_NULL_SHARE']]
_drop += [dict(feature=c, reason='constant on train', value=float(_nun[c]))
          for c in _cand if _nun[c] <= 1 and _null[c] <= CONFIG['MAX_NULL_SHARE']]
_keep = [c for c in _cand if _null[c] <= CONFIG['MAX_NULL_SHARE'] and _nun[c] > 1]
PSI_FEATURES = pd.Series({c: psi(X.loc[IDS['train'], c], X.loc[IDS['oot'], c]) for c in _keep}, dtype='float64')
_unstable = PSI_FEATURES[PSI_FEATURES > CONFIG['PSI_DROP_THRESHOLD']].sort_values(ascending=False)
_drop += [dict(feature=c, reason=f"PSI train->OOT > {CONFIG['PSI_DROP_THRESHOLD']}", value=v) for c, v in _unstable.items()]
FEATURES = [c for c in _keep if c not in _unstable.index]
DROPPED = pd.DataFrame(_drop, columns=['feature', 'reason', 'value'])
DROPPED['block'] = DROPPED['feature'].map(FEATURE_BLOCK)
CAT_FEATURES = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(X[c])]
NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

# categoricals: train top-N levels + 'OTHER' (fixed categories -> identical encoding everywhere)
CAT_LEVELS = {}
for _c in CAT_FEATURES:
    _vc = X.loc[IDS['train'], _c].value_counts()
    CAT_LEVELS[_c] = [v for v in _vc.index[:CONFIG['TOP_N_CATEGORIES']] if v != 'OTHER'] + ['OTHER']


def encode(Xs):
    """Model matrix: numeric float32, categoricals as pandas category with the train levels."""
    out = {}
    for c in FEATURES:
        if c in CAT_LEVELS:
            v = Xs[c].astype('str')
            v = v.where(v.isna() | v.isin(CAT_LEVELS[c]), 'OTHER')
            out[c] = pd.Categorical(v, categories=CAT_LEVELS[c])
        else:
            out[c] = Xs[c].astype('float32')
    return pd.DataFrame(out, index=Xs.index)


print(f'candidates {len(_cand) + int((DROPPED["reason"] == "excluded (CONFIG / leakage guard)").sum())} -> model features '
      f'{len(FEATURES)} ({len(NUM_FEATURES)} numeric, {len(CAT_FEATURES)} categorical); dropped: '
      + ', '.join(f'{k}: {v}' for k, v in DROPPED['reason'].value_counts().items()))
show(DROPPED[DROPPED['reason'].str.startswith('PSI')].head(20), 'Unstable features dropped (PSI train -> OOT)')

## 7. PD model: LightGBM, with a logistic-regression benchmark
- **One model for every checkpoint.** It is trained on all checkpoint observations, with `dh_days_in_arrears` as a feature, so it gives the PD of an episode that is still overdue on day k. The stock is scored on its own current day in arrears.
- **LightGBM**: learning rate 0.03, 31 leaves, `min_child_samples = max(100, 2% of train observations)` (larger than for one row per episode, because an episode contributes up to five correlated checkpoint rows), feature and bagging fraction 0.8, L2 = 5, `CONFIG['N_JOBS']` threads (set it to the cores that are really free on a shared server). It stops early after 100 rounds without a valid-AUC gain, with at most 3000 rounds. Categoricals are native pandas `category`. If LightGBM is unavailable, sklearn `HistGradientBoostingClassifier` is used instead.
- **Benchmark**: a logistic regression on the top-25 numeric features by gain, with 1/99% clipping (train bounds), median imputation and standardisation.
- **Out-of-fold PD for train observations.** In-sample PDs of a boosted model are over-fitted. For every train observation, `pd` is therefore replaced by a **5-fold out-of-fold** prediction (folds grouped by contract, same settings and number of rounds). The grade cutpoints, the train grade table, the segment backtest and the monitoring baseline are then all out-of-sample. The final model, trained on all train observations, scores valid, OOT and the stock.
- **Reported**: AUC, Gini and KS on train (in-sample and out-of-fold) / valid / OOT over all checkpoints, Gini **within** each checkpoint (the ranking that days in arrears alone cannot explain), the score PSI from train to OOT, gain importance, and the feature PSI from train to the **current stock**. Exact TreeSHAP contributions come from LightGBM's `pred_contrib`, so no extra package is needed.

In [ ]:
XE = encode(X)                          # every observation (all checkpoints + the stock on its current day)
Y = OB['bad']
Xtr, ytr = XE.loc[IDS['train']], Y.loc[IDS['train']]
Xva, yva = XE.loc[IDS['valid']], Y.loc[IDS['valid']]
USE_LGB = bool(CONFIG['USE_LIGHTGBM'] and HAS_LGB)
MIN_LEAF = max(100, int(0.02 * len(Xtr)))   # one episode appears at up to len(CHECKPOINTS) checkpoints -> bigger leaves

with timer('PD model trained'):
    if USE_LGB:
        MODEL = lgb.LGBMClassifier(objective='binary', metric='auc', learning_rate=0.03, num_leaves=31,
                                   min_child_samples=MIN_LEAF, colsample_bytree=0.8,
                                   subsample=0.8, subsample_freq=1, reg_lambda=5.0, n_estimators=3000,
                                   random_state=CONFIG['RANDOM_STATE'], n_jobs=CONFIG['N_JOBS'], verbose=-1)
        import inspect
        _eval = (dict(eval_X=(Xva,), eval_y=(yva,)) if 'eval_X' in inspect.signature(MODEL.fit).parameters
                 else dict(eval_set=[(Xva, yva)]))              # lightgbm >= 4.7 renamed eval_set
        MODEL.fit(Xtr, ytr, callbacks=[lgb.early_stopping(100, verbose=False)], **_eval)
        N_ROUNDS = int(MODEL.best_iteration_ or MODEL.n_estimators)
        MODEL_KIND = f'LightGBM ({N_ROUNDS} trees)'
        _fixed = dict(n_estimators=N_ROUNDS)
    else:
        MODEL = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, max_leaf_nodes=31,
                                               min_samples_leaf=MIN_LEAF, l2_regularization=5.0,
                                               categorical_features='from_dtype', early_stopping=True,
                                               validation_fraction=0.2, n_iter_no_change=50,
                                               random_state=CONFIG['RANDOM_STATE'])
        MODEL.fit(Xtr, ytr)
        N_ROUNDS = int(MODEL.n_iter_)
        MODEL_KIND = f'HistGradientBoosting ({N_ROUNDS} iterations)'
        _fixed = dict(max_iter=N_ROUNDS, early_stopping=False)


def predict_pd(Xe):
    return MODEL.predict_proba(Xe)[:, 1] if len(Xe) else np.array([])


# PD of every episode from the final model; for TRAIN episodes it is replaced by an out-of-fold PD
# (5 folds grouped by contract, same settings and number of rounds). In-sample boosted PDs are over-fitted,
# so grade cutpoints, train grade tables, the backtest and the monitoring baseline would otherwise be biased.
PD_IN_SAMPLE = pd.Series(predict_pd(XE), index=OB.index)
OB['pd'] = PD_IN_SAMPLE
_oof = pd.Series(np.nan, index=IDS['train'], dtype='float64')
_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CONFIG['RANDOM_STATE'])
with timer('out-of-fold PD for train episodes (5 folds)'):
    for _a, _b in _folds.split(np.zeros(len(Xtr)), ytr, groups=OB.loc[IDS['train'], 'contract_number']):
        _m = clone(MODEL).set_params(**_fixed).fit(Xtr.iloc[_a], ytr.iloc[_a])
        _oof.iloc[_b] = _m.predict_proba(Xtr.iloc[_b])[:, 1]
OB.loc[IDS['train'], 'pd'] = _oof
log(f'{MODEL_KIND}: PD scored for {len(OB):,} observations (train observations out-of-fold)')

In [ ]:
if USE_LGB:
    IMP = pd.Series(MODEL.booster_.feature_importance('gain'), index=FEATURES, dtype='float64')
else:
    from sklearn.inspection import permutation_importance
    _pi = permutation_importance(MODEL, Xva, yva, scoring='roc_auc', n_repeats=3, random_state=CONFIG['RANDOM_STATE'])
    IMP = pd.Series(_pi.importances_mean, index=FEATURES, dtype='float64').clip(lower=0)
IMP = IMP.sort_values(ascending=False)

LR_FEATURES = [c for c in IMP.index if c in NUM_FEATURES and IMP[c] > 0][:CONFIG['LR_TOP_N']] or NUM_FEATURES[:CONFIG['LR_TOP_N']]
LR, LR_CLIP = None, None
if LR_FEATURES:                 # benchmark: 1/99% clip (train bounds) -> median impute -> standardise -> logit
    LR_CLIP = (Xtr[LR_FEATURES].astype('float64').quantile(0.01), Xtr[LR_FEATURES].astype('float64').quantile(0.99))

    def lr_input(Xe):
        return Xe[LR_FEATURES].astype('float64').clip(lower=LR_CLIP[0], upper=LR_CLIP[1], axis=1)

    LR = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=2000))
    LR.fit(lr_input(Xtr), ytr)
    OB['pd_lr'] = LR.predict_proba(lr_input(XE))[:, 1]

MAIN = MODEL_KIND.split(' ')[0]
_rows = [dict(model=MAIN, sample='train_in_sample', **perf(Y.loc[IDS['train']], PD_IN_SAMPLE.loc[IDS['train']]))]
for _name, _col in ((MAIN, 'pd'), ('LogReg', 'pd_lr')):
    if _col not in OB.columns:
        continue
    for _s in ('train', 'valid', 'oot'):
        _lbl = 'train_oof' if (_s == 'train' and _name == MAIN) else _s
        _rows.append(dict(model=_name, sample=_lbl, **perf(Y.loc[IDS[_s]], OB.loc[IDS[_s], _col])))
METRICS = pd.DataFrame(_rows)
GINI = METRICS.set_index(['model', 'sample'])['gini']
SCORE_PSI = psi(OB.loc[IDS['train'], 'pd'], OB.loc[IDS['oot'], 'pd'])
show(METRICS.set_index(['model', 'sample']), 'Discrimination by sample, all checkpoints (train_oof = honest train performance)')
# per checkpoint: the ranking WITHIN a given day in arrears (days in arrears alone cannot inflate it)
_rows = []
for _name, _col in ((MAIN, 'pd'), ('LogReg', 'pd_lr')):
    if _col not in OB.columns:
        continue
    for _s in ('valid', 'oot'):
        for _k, _ids in OB.loc[IDS[_s]].groupby('checkpoint').groups.items():
            _rows.append(dict(model=_name, sample=_s, checkpoint=_k, **perf(Y.loc[_ids], OB.loc[_ids, _col])))
METRICS_CP = pd.DataFrame(_rows)
if len(METRICS_CP):
    show(METRICS_CP.pivot_table(index=['sample', 'checkpoint'], columns='model', values='gini'),
         'Gini within each checkpoint (valid / OOT)')
print(f"OOT Gini: {MODEL_KIND} {GINI.get((MAIN, 'oot'), np.nan):.3f} | LogReg {GINI.get(('LogReg', 'oot'), np.nan):.3f} | "
      f"score PSI train->OOT {SCORE_PSI:.3f} ({'stable' if SCORE_PSI < 0.1 else 'investigate' if SCORE_PSI < 0.25 else 'UNSTABLE'})")

In [ ]:
IMP_TABLE = (IMP / IMP.sum()).rename('gain_share').to_frame()
IMP_TABLE['block'] = IMP_TABLE.index.map(FEATURE_BLOCK)
show(IMP_TABLE.head(30), 'Top-30 features by gain share')

# Drift train -> current stock (the stock is scored in today's data regime; PSI train -> OOT cannot see that).
# The reference is the train sample re-weighted to the stock's mix of days in arrears (nearest checkpoint),
# so features that grow with time in arrears (calls in the episode, overdue amounts) are compared like for like.
_EXPECTED_DRIFT = {'dh_days_in_arrears', 'ct_mob_at_obs'}
_ref_ids = IDS['train']
if len(STOCK_IDS):
    _cps = np.asarray(CHECKPOINTS)
    _near = _cps[np.abs(OB.loc[STOCK_IDS, 'days_in_arrears'].to_numpy()[:, None] - _cps[None, :]).argmin(axis=1)]
    _mix = pd.Series(_near).value_counts(normalize=True)
    _trc = OB.loc[IDS['train'], 'checkpoint']
    _rs = np.random.default_rng(CONFIG['RANDOM_STATE'])
    _ref_ids = pd.Index(np.concatenate([_rs.choice(_trc.index[_trc == k], size=int(round(w * len(_trc))), replace=True)
                                        for k, w in _mix.items() if (_trc == k).any()]))
STOCK_PSI = pd.DataFrame({'gain_share': IMP_TABLE['gain_share'],
                          'psi_train_oot': PSI_FEATURES.reindex(IMP_TABLE.index),
                          'psi_train_stock': [psi(X.loc[_ref_ids, c], X.loc[STOCK_IDS, c]) if len(STOCK_IDS) else np.nan
                                              for c in IMP_TABLE.index]}, index=IMP_TABLE.index)
STOCK_PSI['block'] = STOCK_PSI.index.map(FEATURE_BLOCK)
_drift = STOCK_PSI[(STOCK_PSI['psi_train_stock'] > CONFIG['PSI_DROP_THRESHOLD']) & ~STOCK_PSI.index.isin(_EXPECTED_DRIFT)]
_top30 = [c for c in IMP_TABLE.index[:30] if c in _drift.index]
_POP_PREFIX = ('dh_', 'ct_')           # loan age / delinquency history: the stock is simply older than the training mix
STOCK_DRIFT = dict(n_features_psi_gt_threshold=int(len(_drift)),
                   gain_share_of_drifting=round(float(_drift['gain_share'].sum()), 4),
                   top30_population_shift=[c for c in _top30 if c.startswith(_POP_PREFIX)],
                   top30_source_drift=[c for c in _top30 if not c.startswith(_POP_PREFIX)])
STOCK_PSI['drift'] = np.where(~STOCK_PSI.index.isin(_drift.index), '',
                              np.where(STOCK_PSI.index.str.startswith(_POP_PREFIX), 'population shift', 'source drift?'))
show(STOCK_PSI.head(30), 'Top-30 features: PSI train -> OOT and train (same days-in-arrears mix) -> current stock')
if STOCK_DRIFT['top30_population_shift']:
    print(f"NOTE population shift (expected: the stock has older loans and longer histories than the training mix): "
          f"{STOCK_DRIFT['top30_population_shift'][:8]}")
if STOCK_DRIFT['top30_source_drift'] and len(STOCK_IDS) < 100:
    print(f'NOTE drift: only {len(STOCK_IDS)} stock contracts - PSI is too noisy to judge; listed for information: '
          f"{STOCK_DRIFT['top30_source_drift'][:8]}")
elif STOCK_DRIFT['top30_source_drift']:
    print(f"WARNING drift: {len(STOCK_DRIFT['top30_source_drift'])} of the top-30 non-history features have PSI(train -> "
          f"stock) > {CONFIG['PSI_DROP_THRESHOLD']}: {STOCK_DRIFT['top30_source_drift'][:8]}. Check whether a source changed "
          'its cadence / definition (or the population moved) before acting on the stock PDs.')
print(f"train -> stock: {STOCK_DRIFT['n_features_psi_gt_threshold']} model features with PSI > "
      f"{CONFIG['PSI_DROP_THRESHOLD']} ({STOCK_DRIFT['gain_share_of_drifting']:.1%} of the gain)")

SHAP_OK = USE_LGB
if SHAP_OK and len(IDS['oot']):
    _contrib = MODEL.predict(XE.loc[IDS['oot']], pred_contrib=True)[:, :-1]
    SHAP_OOT = pd.Series(np.abs(_contrib).mean(axis=0), index=FEATURES).sort_values(ascending=False)
    show(SHAP_OOT.head(15).rename('mean |SHAP| (log-odds), OOT').to_frame(), 'TreeSHAP (LightGBM pred_contrib) on OOT')


BLOCK_LABEL = {'ct_': 'contract', 'dh_': 'DPD history', 'pr_': 'client profile', 'vn_': 'own debits',
               'en_': 'enforcement', 'ex_': 'other-bank debits', 'cl_': 'calls', 'geo_': 'app logins',
               'exc_': 'exclusion list', 'bk_': 'bankruptcy appl.', 'app_': 'application', 'rej_': 'rejected appl.',
               'pkb_': 'PKB'}


def feature_label(c):
    """'pr_PKB_REQ_360D' -> 'client profile: PKB_REQ_360D' (business-readable driver names)."""
    for p, lbl in BLOCK_LABEL.items():
        if c.startswith(p):
            return f'{lbl}: {c[len(p):]}'
    return c


def top_drivers(Xe, k=3):
    """Per row: the k features pushing PD up the most (TreeSHAP, log-odds)."""
    if not SHAP_OK or not len(Xe):
        return pd.Series('', index=Xe.index, dtype='str')
    cb = MODEL.predict(Xe, pred_contrib=True)[:, :-1]
    names = np.asarray([feature_label(c) for c in FEATURES])
    order = np.argsort(-cb, axis=1)[:, :k]
    return pd.Series(['; '.join(f'{names[j]} (+{cb[i, j]:.2f})' for j in order[i] if cb[i, j] > 0)
                      for i in range(len(cb))], index=Xe.index, dtype='str')


MODEL_DIR = OUT / 'model'
if USE_LGB:
    MODEL.booster_.save_model(str(MODEL_DIR / 'lgbm_model.txt'))
_ = (MODEL_DIR / 'features.json').write_text(json.dumps({'features': FEATURES, 'categorical_levels': CAT_LEVELS,
                                                      'lr_features': LR_FEATURES}, ensure_ascii=False, indent=1),
                                          encoding='utf-8')
_ = (MODEL_DIR / 'config.json').write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1, default=str), encoding='utf-8')

## 8. Risk grades G1-G5
The PD cutpoints are the quantiles of the **train** PD of new entries (the first checkpoint) at `GRADE_QUANTILES` (40/65/85/95%). The train PD here is the out-of-fold PD, so the cutpoints are not stretched by over-fitting. The cutpoints are then **fixed** for every checkpoint, OOT and the stock: a grade is a PD range, so an episode that is still overdue on day 15 usually sits one or two grades higher than on day 1. Grade shares may drift while the meaning of a grade does not.

Each sample shows n, average PD, the actual bad rate and the self-cure rate per grade, followed by a monotonicity check. An inversion between neighbouring grades is tested with a one-sided two-proportion z-test. Small G4/G5 samples can invert by chance; only a *significant* inversion means the ranking is broken.

In [ ]:
GRADES = ['G1', 'G2', 'G3', 'G4', 'G5']
CUTS = np.quantile(OB.loc[IDS_FIRST['train'], 'pd'], CONFIG['GRADE_QUANTILES'])   # anchored on new entries
if len(np.unique(CUTS)) < len(CUTS):
    print('WARN: tied PD cutpoints (few distinct scores) -> some grades will be empty')


def to_grade(p):
    idx = np.searchsorted(CUTS, np.asarray(p, dtype='float64'), side='right')
    return np.asarray(GRADES, dtype=object)[np.clip(idx, 0, len(GRADES) - 1)]


OB['risk_grade'] = pd.Series(to_grade(OB['pd']), index=OB.index, dtype='str')


def grade_table(df):
    t = df.groupby('risk_grade').agg(n=('bad', 'size'), avg_pd=('pd', 'mean'), bad_rate=('bad', 'mean'),
                                     self_cure=('self_cure30', 'mean'))
    t['share'] = t['n'] / t['n'].sum()
    return t.reindex(GRADES)[['n', 'share', 'avg_pd', 'bad_rate', 'self_cure']]


GRADE_TABLES = {s: grade_table(OB.loc[IDS[s]]) for s in ('train', 'valid', 'oot')}
GRADES_TBL = pd.concat(GRADE_TABLES, names=['sample'])
_cut_txt = ' | '.join(f'{g} <= {c:.3f}' for g, c in zip(GRADES, CUTS)) + f' | G5 > {CUTS[-1]:.3f}'
show(GRADES_TBL, f'Risk grades, all checkpoints (PD cutpoints: {_cut_txt})')
_lo = OB.loc[IDS['valid'].union(IDS['oot'])]
GRADE_BY_CP = _lo.pivot_table(index='risk_grade', columns='checkpoint', values='bad', aggfunc='mean', observed=True).reindex(GRADES)
GRADE_BY_CP.columns = [f'bad90 day {c}' for c in GRADE_BY_CP.columns]
GRADE_BY_CP['pd_mean'] = _lo.groupby('risk_grade')['pd'].mean()
show(GRADE_BY_CP, 'Actual bad rate by grade and checkpoint (valid + OOT): a grade means the same risk on any day')
def monotonicity(t, alpha=0.05):
    """Strict check of bad rate G1 <= ... <= G5, plus a one-sided two-proportion z-test for every inversion
    (an inversion that is not significant is sampling noise of small grades, not a broken ranking)."""
    r = t.dropna(subset=['bad_rate'])
    inv = []
    for (g1, a), (g2, b) in zip(r.iterrows(), list(r.iterrows())[1:]):
        if b['bad_rate'] < a['bad_rate']:
            p = (a['bad_rate'] * a['n'] + b['bad_rate'] * b['n']) / (a['n'] + b['n'])
            se = np.sqrt(p * (1 - p) * (1 / a['n'] + 1 / b['n']))
            pval = 0.5 * math.erfc((a['bad_rate'] - b['bad_rate']) / se / math.sqrt(2)) if se > 0 else 0.0
            inv.append((g1, g2, pval))
    strict = not inv
    signif = [x for x in inv if x[2] < alpha]
    status = ('monotone' if strict else 'SIGNIFICANT inversion' if signif else
              'inversion within sampling noise (' + ', '.join(f'{a}>{b} p={p:.2f}' for a, b, p in inv) + ')')
    return dict(strict=strict, significant_inversion=bool(signif), status=status)


MONOTONE = {}
for _s, _t in GRADE_TABLES.items():
    MONOTONE[_s] = monotonicity(_t)
    _r = _t['bad_rate'].dropna().to_numpy()
    print(f"{_s:5s}: bad rate by grade {' -> '.join(f'{x:.0%}' for x in _r)}  -> {MONOTONE[_s]['status']}"
          + ('  WARN' if MONOTONE[_s]['significant_inversion'] else ''))

with open(MODEL_DIR / 'model_bundle.pkl', 'wb') as _fh:
    pickle.dump(dict(model=MODEL, model_kind=MODEL_KIND, features=FEATURES, cat_levels=CAT_LEVELS, lr=LR,
                     lr_features=LR_FEATURES, lr_clip=None if LR_CLIP is None else (LR_CLIP[0].to_dict(), LR_CLIP[1].to_dict()),
                     grade_cutpoints=CUTS.tolist(), grades=GRADES, config=CONFIG, checkpoint_days=CHECKPOINTS,
                     data_cut=str(DATA_CUT.date()), run_mode=RUN_MODE), _fh)
_ = (MODEL_DIR / 'cutpoints.json').write_text(json.dumps({'grades': GRADES, 'pd_cutpoints': CUTS.tolist(),
                                                       'quantiles': CONFIG['GRADE_QUANTILES']}, indent=1), encoding='utf-8')

## 9. Segmentation
### 9.0 Regulatory guardrails (Kazakhstan, 2026; from the research memo. Legal must confirm items marked [verify])
- **Banking law.** Law 2444 (art. 36) was replaced by **Law 258-VIII of 16.01.2026**, in force from 19.03.2026. Borrower settlement is now **art. 61** [verify transition].
- **Notice.** The borrower must be notified **within 10 calendar days** of the overdue, in writing **and** via digital channels. This applies to **every segment and every challenger arm**. The output carries `notify_due_date` and `notify_overdue` (the deadline has passed: check that the notice went out).
- **Change of terms.** The borrower may apply **within 30 days**, and the bank answers in writing **within 15 days** (`restructuring_window_end`). New consumer loans are banned above **30 DPD** (art. 58) [verify]. Hardship relief must therefore be a **change of terms**, never a new loan.
- **Contact limits** (Rules No. 136), carried into the stock file as `contact_caps` for every segment with calls or visits:
  - calls: weekdays 08:00-21:00 with **at most 3 a day**; weekends and holidays 09:00-19:00 with **at most 2 a day**;
  - only from identifiable bank numbers;
  - visits on weekdays only, **at most 1 a day and 3 a week**.
- **Third parties.** Banking secrecy applies: the debt is never disclosed to relatives or employers. `third_party_contact_flag` marks CRM "3 лицу" outcomes for compliance review.
- **Debiting.** A bank payment demand may take **at most 50%** of a balance and must leave **2 x ПМ = 101,702 KZT a month**, from 21.10.2026 (`debitable_funds`). Social, pension and alimony accounts are protected, so these inflows are left out of the capacity axis. Debiting at other banks needs contract consent [uncertain].
- **Collectors.** Hand-over only after **90 DPD**, under an agency model. Sale of retail debt to collectors is **banned until 01.05.2027**.
- **Bankruptcy** (Law 178-VII). Out-of-court bankruptcy requires debt of at most 1,600 MRP (6.92M KZT), **no property** and 12 months without payments. Car owners fail the property test, which plausibly explains why every application in `bankrupt` was refused (R03). A lawsuit by **another** creditor does not create a moratorium, so a PKB "court" status alone sets `legal_review_flag` and does not stop contact (`R02_PKB_COURT=False`).
- **«Лудоман»** (gambling-restriction list) is a vulnerability flag: no credit offers and a monitored restructuring. **«Стоп-кредит»** has 5.1M users and is revocable. It is a *weak* signal, so it sets `no_credit_offer` but is **not** a hard rule; this is why R08 is gambling only.
- **Soft-stage collateral triggers mean inspection, not repossession.** Out-of-court sale of the car is reportedly barred below 3 months overdue [verify].
- The remaining `CONFIG['COMPLIANCE']` constants (restructuring review days, ПМ / MRP, out-of-court bankruptcy limit, collector hand-over DPD, assignment ban date) are documentation for Legal; the code does not enforce them.

### Segment playbook
1. **Hard rules** apply first, in priority order; the first match is the segment. The treatments of every other rule hit are not dropped: they are listed in `additional_actions`.
2. **Modifiers** (R04 social exclusion list, R05 enforcement) do **not** replace the risk segment, because risk inside them still spans G1-G5. They add actions (no auto-debit retry, early restructuring offer) on top of it, in `modifiers` / `additional_actions`.
3. Everyone else is placed on the risk x capacity x engagement matrix S1-S6.

| segment | kind | definition | treatment |
|---|---|---|---|
| R01 deceased | hard | `FLG_DEATH` = 1 | stop contact; legal / insurance / heirs |
| R02 bankruptcy | hard | active bankruptcy exclusion, or PKB bankruptcy status (PKB court only if `R02_PKB_COURT`) | stop contact; legal process |
| R03 bankruptcy attempt | hard | refused out-of-court application within 365 days | senior agent; documented art. 61 offer; car check |
| R06 abroad / skip | hard | CRM note "went abroad" (last 90 days), or abroad logins >= 50% (or last app event abroad with abroad logins) and no contact for 90 days, or unreachable (no logins 90 days, 3+ calls, 0% contact) | skip-trace; field check of the car |
| R07 FPD / fraud | hard | 1st or 2nd instalment unpaid for `FPD_MIN_DAYS`+ days | fraud verification; dealer review |
| R08 gambling | hard | PKB «Лудоман» | vulnerability script; no credit |
| R09 collateral alert | hard | PKB pledge change / assignment | car inspection within 3 days; legal review |
| R04 social exclusion | modifier | active СУСН / УВЕК / упрощенка / БизБиргемиз list (other titles -> `exclusion_review_flag`) | no auto-debit reliance; soft reminders; early restructuring call |
| R05 enforcement | modifier | open ИР3/ИР4 orders accepted within `ENFORCEMENT_MAX_AGE_DAYS` | no auto-debit retry (bailiff orders); restructuring offer |
| S1 self-cure | model | G1-G2, engaged and has capacity | push + SMS + auto-debit retry; no operator before day 7 |
| S2 low-risk remind | model | other G1-G2 | robot / IVR + SMS on days 1-5; operator from day 6 |
| S3 standard | model | G3 (not hardship) | SC 4-30 operator queue; PTP tracking |
| S4 hardship | model | G3-G5, engaged, no capacity | proactive art. 61 restructuring / deferral |
| S5 high-risk contactable | model | G4-G5, engaged (capacity yes or unknown) | senior collectors from day 1; daily PTP control |
| S6 high-risk unreachable | model | G4-G5, not engaged | skip-trace; early field visit; car check |

- **Capacity** = `capacity_ratio >= 1.5` or `depo_cover >= 1` (both against `pay = max(MONTH_LOAN_PAY_AMOUNT, annuity)`). It is *unknown* when both are missing.
- **Engagement** = logins in the last 30 days, or MAU flags, or `CNT_SESSIONS > 0`, or a past call contact rate of at least 50%.
- **Day in the playbook.** The stock is graded on its current day in arrears. `operator_due_now` says whether the segment's first operator day has passed; for S1 / S2 that means the digital-only window is over and `treatment_now` switches to the next stage.
- **Validation.** Segments reached only through PKB events (R08, R09, the PKB part of R02) have no labelled history yet (PKB monitoring starts 2026-09-07): `validation = 'unvalidated (PKB overlay)'`.

**Treatment contamination.** Past outcomes reflect the current SC 4-30 calling strategy, so the backtest shows *risk under today's treatment*, not uplift. Uplift needs randomised challengers: 15% of each segment that has a challenger treatment, by contract hash (`arm`; `none` where the segment has no challenger). The first test is S1 with *no operator until day 10*.

In [ ]:
SEGMENT_TREATMENT = {   # edit here: kind, action, channels, first operator day, challenger idea, next stage
    'R01_DECEASED': dict(kind='hard', action='Stop collection contact; verify the death record; legal / insurance / heirs process',
                         channels='none (legal)', operator_from_day=None, challenger=None),
    'R02_BANKRUPTCY': dict(kind='hard', action='Stop collection contact; legal process (bankruptcy)',
                           channels='legal', operator_from_day=None, challenger=None),
    'R03_BANKRUPTCY_ATTEMPT': dict(kind='hard', action='Senior agent; document an art. 61 change-of-terms offer; car (collateral) check',
                                   channels='senior agent, letter', operator_from_day=1, challenger='early voluntary-sale offer'),
    'R06_ABROAD_SKIPTRACE': dict(kind='hard', action='Skip-trace; field check of the car; contact the borrower / obligated parties only',
                                 channels='skip-trace, field', operator_from_day=1, challenger=None),
    'R07_FPD_FRAUD_CHECK': dict(kind='hard', action='Fraud verification (identity, income, pledge); dealer review',
                                channels='fraud unit, call', operator_from_day=1, challenger=None),
    'R08_GAMBLING': dict(kind='hard', action='Vulnerability script; monitored restructuring; no credit offers',
                         channels='trained agent', operator_from_day=3, challenger=None),
    'R09_COLLATERAL_ALERT': dict(kind='hard', action='Car inspection within 3 days; legal review',
                                 channels='field, legal', operator_from_day=1, challenger=None),
    'R04_SOCIAL_EXCLUSION': dict(kind='modifier', action='Social exclusion list: do not rely on auto-debit; soft reminders; early restructuring call',
                                 channels='push, SMS, call', operator_from_day=None, challenger=None),
    'R05_ENFORCEMENT': dict(kind='modifier', action='Enforcement orders on accounts: no auto-debit retry; restructuring / payment-holiday offer',
                            channels='call, letter', operator_from_day=None, challenger=None),
    'S1_SELF_CURE': dict(kind='model', action='Homebank push + SMS, auto-debit retry within the 50% / 2xПМ limits; no operator before day 7',
                         channels='push, SMS, auto-debit', operator_from_day=7, challenger='no operator until day 10',
                         next_stage='Digital window over: operator reminder call; keep the auto-debit retry'),
    'S2_LOW_RISK_REMIND': dict(kind='model', action='Robot / IVR + SMS on days 1-5; operator from day 6',
                               channels='IVR, SMS', operator_from_day=6, challenger='push instead of IVR',
                               next_stage='Operator queue (SC 4-30); PTP tracking'),
    'S3_STANDARD': dict(kind='model', action='SC 4-30 operator queue; PTP tracking',
                        channels='operator', operator_from_day=4, challenger='operator from day 2'),
    'S4_HARDSHIP': dict(kind='model', action='Proactive art. 61 restructuring / deferral (change of terms, not a new loan)',
                        channels='operator, Homebank offer', operator_from_day=2, challenger='digital self-service offer'),
    'S5_HIGH_RISK_CONTACTABLE': dict(kind='model', action='Senior collectors from day 1; daily PTP control within the call caps',
                                     channels='senior operator', operator_from_day=1, challenger='restructuring offer on day 5'),
    'S6_HIGH_RISK_UNREACHABLE': dict(kind='model', action='Skip-trace, early field visit, car check',
                                     channels='skip-trace, field', operator_from_day=1, challenger=None),
}
SEGMENT_ORDER = [k for k, v in SEGMENT_TREATMENT.items() if v['kind'] != 'modifier']
HARD_RULES = [k for k, v in SEGMENT_TREATMENT.items() if v['kind'] == 'hard']
MODIFIER_RULES = [k for k, v in SEGMENT_TREATMENT.items() if v['kind'] == 'modifier']
ALL_RULES = sorted(HARD_RULES + MODIFIER_RULES)
RULE_TEXT = {
    'R01_DECEASED': 'FLG_DEATH = 1', 'R02_BANKRUPTCY': 'active bankruptcy exclusion / PKB bankruptcy status',
    'R03_BANKRUPTCY_ATTEMPT': 'refused out-of-court bankruptcy <= 365d', 'R04_SOCIAL_EXCLUSION': 'active СУСН/УВЕК/упрощенка/БизБиргемиз list',
    'R05_ENFORCEMENT': f"open {'/'.join(CONFIG['ENFORCEMENT_OPERTYPES'])} orders <= {CONFIG['ENFORCEMENT_MAX_AGE_DAYS']}d old",
    'R06_ABROAD_SKIPTRACE': 'CRM abroad note / abroad logins and not reached / unreachable',
    'R07_FPD_FRAUD_CHECK': f"1st/2nd instalment unpaid {CONFIG['FPD_MIN_DAYS']}+ days", 'R08_GAMBLING': 'PKB «Лудоман»',
    'R09_COLLATERAL_ALERT': 'PKB collateral change / assignment',
}
_cc = CONFIG['COMPLIANCE']
CONTACT_CAPS = (f"calls: weekdays {_cc['CALLS']['weekday'][0]:02d}-{_cc['CALLS']['weekday'][1]:02d}h max "
                f"{_cc['CALLS']['weekday'][2]}/day, weekends {_cc['CALLS']['weekend'][0]:02d}-{_cc['CALLS']['weekend'][1]:02d}h max "
                f"{_cc['CALLS']['weekend'][2]}/day; visits weekdays only, max {_cc['VISITS'][0]}/day and {_cc['VISITS'][1]}/week; "
                'identifiable bank numbers only; no debt disclosure to third parties')


def rule_flags(R):
    """Overlay rules (hard + modifiers) on a feature frame; a missing block -> the rule cannot fire."""
    pkb_court = col(R, 'pkb_court_all').gt(0)
    abroad_logins = (col(R, 'geo_share_abroad_30d').ge(0.5)
                     | (col(R, 'pr_last_event_abroad').eq(1) & col(R, 'geo_share_abroad_90d').gt(0)))
    not_reached_90d = ~col(R, 'cl_days_since_contact').le(90)
    unreachable = col(R, 'geo_logins_90d').eq(0) & col(R, 'cl_contact_rate_365d').eq(0) & col(R, 'cl_n_calls_365d').ge(3)
    f = {
        'R01_DECEASED': col(R, 'pr_flag_FLG_DEATH').eq(1),
        'R02_BANKRUPTCY': (col(R, 'exc_active_bankruptcy').eq(1) | col(R, 'pkb_bankruptcy_all').gt(0)
                           | (pkb_court & CONFIG['R02_PKB_COURT'])),
        'R03_BANKRUPTCY_ATTEMPT': col(R, 'bk_flag_365d').eq(1),
        'R04_SOCIAL_EXCLUSION': col(R, 'exc_active_social_list').eq(1),
        'R05_ENFORCEMENT': col(R, 'en_n_open_rule').gt(0),
        'R06_ABROAD_SKIPTRACE': col(R, 'cl_abroad_kw').eq(1) | (abroad_logins & not_reached_90d) | unreachable,
        'R07_FPD_FRAUD_CHECK': col(R, 'dh_fpd_flag').eq(1) & col(R, 'dh_days_in_arrears').ge(CONFIG['FPD_MIN_DAYS']),
        'R08_GAMBLING': col(R, 'pkb_ludoman_all').gt(0),
        'R09_COLLATERAL_ALERT': col(R, 'pkb_collateral_change_all').gt(0) | col(R, 'pkb_assignment_sold_all').gt(0),
    }
    return pd.DataFrame(f, index=R.index)[ALL_RULES].fillna(False).astype(bool)


def capacity_axis(R):
    """1 = can pay (income / deposits cover the monthly payment), 0 = cannot, NaN = unknown."""
    cr, dc = col(R, 'capacity_ratio'), col(R, 'depo_cover')
    yes = cr.ge(CONFIG['CAPACITY_RATIO_MIN']) | dc.ge(CONFIG['DEPO_COVER_MIN'])
    known = cr.notna() | dc.notna()
    return pd.Series(np.where(yes, 1.0, np.where(known, 0.0, np.nan)), index=R.index)


def engaged_axis(R):
    """Reachable / willing: app logins, MAU flags, sessions or a good past contact rate."""
    return (col(R, 'geo_logins_30d').gt(0) | col(R, 'pr_flag_MAU_FIREBASE').eq(1) | col(R, 'pr_flag_MAU_AUTORIZATION').eq(1)
            | col(R, 'pr_CNT_SESSIONS').gt(0) | col(R, 'cl_contact_rate_365d').ge(CONFIG['CONTACT_RATE_ENGAGED']))


def segment_frame(R, A, pdv, grade):
    """Segments: hard rules (priority order, first match) on the rule frame R; the risk x capacity x engagement
    matrix S1-S6 on the axis frame A for everyone else; modifiers and the other rule hits as additional actions."""
    A = A.reindex(R.index)
    F = rule_flags(R)
    cap, eng = capacity_axis(A), engaged_axis(A).to_numpy()
    g = pd.to_numeric(pd.Series(grade, index=R.index).str[1:], errors='coerce').to_numpy()
    c1, c0 = cap.eq(1).to_numpy(), cap.eq(0).to_numpy()
    seg_model = np.select([(g <= 2) & eng & c1, g <= 2, (g >= 3) & c0 & eng, (g >= 4) & eng, g >= 4],
                          ['S1_SELF_CURE', 'S2_LOW_RISK_REMIND', 'S4_HARDSHIP', 'S5_HIGH_RISK_CONTACTABLE',
                           'S6_HIGH_RISK_UNREACHABLE'], default='S3_STANDARD')
    hard = F[HARD_RULES].to_numpy()
    any_hard = hard.any(axis=1)
    first = np.asarray(HARD_RULES)[hard.argmax(axis=1)]
    segment = np.where(any_hard, first, seg_model)
    cap_txt = np.where(c1, 'yes', np.where(c0, 'no', 'unknown'))
    model_reason = [f'{gr}, PD {p:.1%}, capacity {c}, engaged {"yes" if e else "no"}'
                    for gr, p, c, e in zip(grade, pdv, cap_txt, eng)]
    # every rule hit that is not the segment itself -> its action is kept as an additional action
    extra = np.full(len(R), '', dtype=object)
    for r in ALL_RULES:
        hit = F[r].to_numpy() & (segment != r)
        extra = np.where(hit, extra + np.where(extra == '', '', ' | ') + f"[{r[:3]}] {SEGMENT_TREATMENT[r]['action']}", extra)
    names = np.asarray(F.columns)
    arr = F.to_numpy()
    pkb_only_r02 = (segment == 'R02_BANKRUPTCY') & ~col(R, 'exc_active_bankruptcy').eq(1).to_numpy()
    unvalidated = np.isin(segment, ['R08_GAMBLING', 'R09_COLLATERAL_ALERT']) | pkb_only_r02
    return pd.DataFrame({
        'segment': segment,
        'segment_reason': np.where(any_hard, [RULE_TEXT[r] for r in first], model_reason),
        'model_segment': seg_model,
        'rules_hit': [';'.join(names[row]) for row in arr],
        'modifiers': [';'.join(r for r, h in zip(MODIFIER_RULES, row) if h) for row in F[MODIFIER_RULES].to_numpy()],
        'additional_actions': extra,
        'capacity': cap_txt, 'engaged': eng.astype('int8'),
        'legal_review_flag': (col(R, 'pkb_court_all').gt(0) & (not CONFIG['R02_PKB_COURT'])).to_numpy().astype('int8'),
        'exclusion_review_flag': col(R, 'exc_active_other').eq(1).to_numpy().astype('int8'),
        'validation': np.where(unvalidated, 'unvalidated (PKB overlay, no labelled history)', 'backtested'),
    }, index=R.index).astype({c: 'str' for c in ('segment', 'segment_reason', 'model_segment', 'rules_hit', 'modifiers',
                                                 'additional_actions', 'capacity', 'validation')})


# history (every checkpoint) and the stock on the cut: everything as of obs_date
SEG_COLS = ['segment', 'segment_reason', 'model_segment', 'rules_hit', 'modifiers', 'additional_actions', 'capacity',
            'engaged', 'legal_review_flag', 'exclusion_review_flag', 'validation']
SEG_H = segment_frame(X, X, OB['pd'].to_numpy(), OB['risk_grade'].to_numpy())
OB = OB.drop(columns=[c for c in SEG_COLS if c in OB.columns]).join(SEG_H)
# current stock: rules with the LATEST info (score_date overlay); capacity / engagement with the same definitions
# as the backtest (model features on the cut); PD and grade on the current day in arrears
SEG_S = segment_frame(XNOW, X, OB.loc[STOCK_IDS, 'pd'].to_numpy(), OB.loc[STOCK_IDS, 'risk_grade'].to_numpy())
if len(SEG_S):
    OB.loc[STOCK_IDS, SEG_COLS] = SEG_S[SEG_COLS]
print('rule hit rates (labelled history at obs | stock now):')
display(pd.DataFrame({'history': rule_flags(X.loc[OB.index[OB['labeled']]]).mean(),
                      'stock_now': rule_flags(XNOW).mean() if len(XNOW) else np.nan,
                      'kind': [SEGMENT_TREATMENT[r]['kind'] for r in ALL_RULES]}))

In [ ]:
_T = SEGMENT_TREATMENT
_first_pd = OB.loc[FIRST].set_index('episode_id')['pd']
S = OB.loc[STOCK_IDS].join(contracts.set_index('contract_number')[['process_ref', 'iin']], on='contract_number')
_ofd = pd.to_numeric(S['segment'].map({k: v['operator_from_day'] for k, v in _T.items()}), errors='coerce')
_next = S['segment'].map({k: v.get('next_stage') for k, v in _T.items()})
_due = _ofd.notna() & (S['days_in_arrears'].astype('float64') >= _ofd.fillna(np.inf))
_chal = S['segment'].map({k: v['challenger'] for k, v in _T.items()})
_has_cc = S['contract_number'].notna() & _chal.notna()
S = S.assign(
    pd_at_entry=S['episode_id'].map(_first_pd),
    treatment=S['segment'].map({k: v['action'] for k, v in _T.items()}),
    channels=S['segment'].map({k: v['channels'] for k, v in _T.items()}),
    operator_from_day=_ofd, playbook_day=S['days_in_arrears'], operator_due_now=_due.astype('int8'),
    treatment_now=np.where(_due & _next.notna(), _next, S['segment'].map({k: v['action'] for k, v in _T.items()})),
    contact_caps=np.where(S['segment'].map({k: v['channels'] for k, v in _T.items()}).str.contains(
        'call|operator|agent|IVR|field|skip', case=False, na=False) | _due, CONTACT_CAPS, ''),
    score_date=SCORE_DATE, data_cut=DATA_CUT, run_mode=RUN_MODE,
    notify_due_date=(S['entry_date'] + pd.Timedelta(days=_cc['NOTIFY_DAYS'])).astype('datetime64[us]'),
    restructuring_window_end=(S['entry_date'] + pd.Timedelta(days=_cc['RESTRUCT_APPLY_DAYS'])).astype('datetime64[us]'),
    capacity_ratio=col(X, 'capacity_ratio').reindex(S.index), depo_cover=col(X, 'depo_cover').reindex(S.index),
    debitable_funds=col(X, 'debitable_funds').reindex(S.index),
    logins_30d=col(XNOW, 'geo_logins_30d').reindex(S.index),
    contact_rate_365d=col(XNOW, 'cl_contact_rate_365d').reindex(S.index),
    enforcement_open=col(XNOW, 'en_n_open_rule').reindex(S.index),
    no_credit_offer=((col(XNOW, 'pkb_stop_credit_all').gt(0) | col(XNOW, 'pkb_ludoman_all').gt(0)).reindex(S.index)
                     .fillna(False) | (S['days_in_arrears'] > _cc['NO_NEW_LOAN_DPD'])).astype('int8'),
    third_party_contact_flag=col(XNOW, 'cl_n_third_party_365d').gt(0).reindex(S.index).fillna(False).astype('int8'),
    arm=np.where(~_has_cc, 'none', np.where(hash_share(S['contract_number']) < CONFIG['CHALLENGER_SHARE'], 'challenger', 'champion')),
    top_drivers=top_drivers(XE.loc[STOCK_IDS]))
_mod_txt = S['modifiers'].map(lambda m: ' + '.join(_T[r]['action'] for r in m.split(';') if r in _T) if m else '')
S['treatment_now'] = np.where(_mod_txt != '', S['treatment_now'] + ' || MODIFIER (overrides the above where they '
                              'conflict, e.g. no auto-debit retry): ' + _mod_txt, S['treatment_now'])
S['notify_days_left'] = (S['notify_due_date'] - SCORE_DATE).dt.days
S['notify_overdue'] = (S['notify_days_left'] < 0).astype('int8')
S['challenger_treatment'] = np.where(S['arm'] == 'challenger', _chal, None)

SEG_CURRENT = (S.groupby('segment').agg(n=('pd', 'size'), avg_pd=('pd', 'mean'), avg_pd_at_entry=('pd_at_entry', 'mean'),
                                        median_days_in_arrears=('days_in_arrears', 'median'),
                                        operator_due_now=('operator_due_now', 'sum'),
                                        challenger=('arm', lambda a: int((a == 'challenger').sum())),
                                        with_modifiers=('modifiers', lambda m: int((m != '').sum())),
                                        unvalidated=('validation', lambda v: int(v.str.startswith('unvalidated').sum())))
                .reindex([s for s in SEGMENT_ORDER if s in set(S['segment'])]))
SEG_CURRENT['share'] = SEG_CURRENT['n'] / max(len(S), 1)
SEG_CURRENT['treatment'] = SEG_CURRENT.index.map({k: v['action'] for k, v in _T.items()})
show(SEG_CURRENT, f'Current stock ({len(S)} contracts in DPD {CONFIG["SOFT_DPD_MIN"]}-{CONFIG["SOFT_DPD_MAX"]} on {DATA_CUT:%Y-%m-%d}; '
                  'PD on the current day in arrears)')
print(f"stock: mean PD now {S['pd'].mean():.1%} vs at entry {S['pd_at_entry'].mean():.1%}; operator due now for "
      f"{int(S['operator_due_now'].sum())} contracts; with modifiers {int((S['modifiers'] != '').sum())}; "
      f"legal review {int(S['legal_review_flag'].sum())}; unvalidated (PKB) {int(S['validation'].str.startswith('unvalidated').sum())}")
show(S[['contract_number', 'days_in_arrears', 'pd', 'pd_at_entry', 'risk_grade', 'segment', 'modifiers', 'arm',
        'notify_due_date', 'top_drivers']].sort_values('pd', ascending=False).head(10),
     'Highest-PD contracts in the stock (IINs are in the output files)')

### 9.3 Backtest on labelled history
The segments are applied to past observations as of their `obs_date`, at every checkpoint (the same way the stock is scored on its current day). Bad rates are the outcomes under the treatment of the time (see the contamination caveat above). A good segmentation shows a clear spread: S1/S2 should have a low bad rate and high self-cure, S5/S6 a high bad rate.
- `bad90_ci_low/high` is a Wilson 95% interval; `SEGMENT_TESTS` tests neighbouring model segments (two-proportion z-test). A split whose difference is not significant is not yet supported by the data (merge the two segments or wait for more history).
- `RULE_SANITY` shows every rule's bad rate against the portfolio and warns when a hard, high-touch rule (field, skip-trace, fraud, senior agent) is *less* risky than the low-risk S2 segment.
- The axis check validates the engagement and capacity splits inside each risk grade.

In [ ]:
def seg_table(df, by='segment'):
    t = df.groupby(by).agg(n=('bad', 'size'), n_bad=('bad', 'sum'), avg_pd=('pd', 'mean'), bad90=('bad', 'mean'),
                           self_cure=('self_cure30', 'mean'), avg_max_dpd=('max_dpd', 'mean'),
                           avg_days_in_arrears=('days_in_arrears', 'mean'))
    t['bad90_ci_low'], t['bad90_ci_high'] = wilson(t['n_bad'], t['n'])
    t['roll31_60d'] = df[df['labeled_roll']].groupby(by)['roll31_60d'].mean()
    t['share'] = t['n'] / t['n'].sum()
    order = [s for s in SEGMENT_ORDER if s in t.index] if by == 'segment' else list(t.index)
    return t.reindex(order)[['n', 'share', 'avg_pd', 'bad90', 'bad90_ci_low', 'bad90_ci_high', 'self_cure',
                             'roll31_60d', 'avg_max_dpd', 'avg_days_in_arrears']]


_LAB = OB[OB['labeled']]
BACKTEST = pd.concat({'all_labeled': seg_table(_LAB), 'oot': seg_table(OB.loc[IDS['oot']]),
                      f'all_labeled_day{CP0}': seg_table(_LAB[_LAB['checkpoint'] == CP0])}, names=['sample'])
show(BACKTEST, 'Segment backtest, all checkpoints (bad90 = actual 90+ within 6 months, Wilson 95% CI)')
BACKTEST_BY_CP = _LAB.pivot_table(index='segment', columns='checkpoint', values='bad', aggfunc=['mean', 'size'], observed=True)
BACKTEST_BY_CP.columns = [f"{'bad90' if a == 'mean' else 'n'} day {k}" for a, k in BACKTEST_BY_CP.columns]
BACKTEST_BY_CP = BACKTEST_BY_CP.reindex([s for s in SEGMENT_ORDER if s in BACKTEST_BY_CP.index])
show(BACKTEST_BY_CP, 'Segment bad rate by checkpoint (labelled)')

# neighbouring model segments: is the split supported?
_ms = _LAB[_LAB['segment'].str.startswith('S')].groupby('segment')['bad'].agg(['sum', 'size'])
_rows = []
for _a, _b in [('S1_SELF_CURE', 'S2_LOW_RISK_REMIND'), ('S2_LOW_RISK_REMIND', 'S3_STANDARD'), ('S3_STANDARD', 'S4_HARDSHIP'),
               ('S4_HARDSHIP', 'S5_HIGH_RISK_CONTACTABLE'), ('S5_HIGH_RISK_CONTACTABLE', 'S6_HIGH_RISK_UNREACHABLE')]:
    if _a in _ms.index and _b in _ms.index:
        (_ka, _na), (_kb, _nb) = _ms.loc[_a].to_numpy(), _ms.loc[_b].to_numpy()
        _p = prop_test(_ka, _na, _kb, _nb)
        _rows.append(dict(pair=f'{_a[:2]} vs {_b[:2]}', n_a=int(_na), bad90_a=_ka / _na, n_b=int(_nb), bad90_b=_kb / _nb,
                          p_value=_p, supported=bool(_p < 0.05) if np.isfinite(_p) else False))
SEGMENT_TESTS = pd.DataFrame(_rows)
show(SEGMENT_TESTS, 'Neighbouring model segments: two-proportion z-test on labelled observations without a hard rule')
if len(SEGMENT_TESTS) and not SEGMENT_TESTS['supported'].all():
    print('NOTE: splits without a significant difference: '
          + ', '.join(SEGMENT_TESTS.loc[~SEGMENT_TESTS['supported'], 'pair'])
          + ' -> treat them as one risk level until more history accumulates (the treatments may still differ by design).')

# rule sanity: every rule (hit regardless of priority) against the portfolio and against S2
_RF = rule_flags(X.loc[_LAB.index])
_s2 = _LAB.loc[_LAB['model_segment'] == 'S2_LOW_RISK_REMIND', 'bad'].mean()
_rows = []
for _r in ALL_RULES:
    _h = _RF[_r].to_numpy()
    _k, _n = int(_LAB['bad'].to_numpy()[_h].sum()), int(_h.sum())
    _lo, _hi = wilson(_k, _n) if _n else (np.nan, np.nan)
    _rows.append(dict(rule=_r, kind=SEGMENT_TREATMENT[_r]['kind'], n_hits=_n, hit_rate=_n / max(len(_LAB), 1),
                      bad90=_k / _n if _n else np.nan, ci_low=float(_lo), ci_high=float(_hi),
                      bad90_rest=_LAB['bad'].to_numpy()[~_h].mean() if (~_h).any() else np.nan))
RULE_SANITY = pd.DataFrame(_rows).set_index('rule')
_high_touch = ['R03_BANKRUPTCY_ATTEMPT', 'R06_ABROAD_SKIPTRACE', 'R07_FPD_FRAUD_CHECK', 'R09_COLLATERAL_ALERT']
RULE_SANITY['check'] = np.select(
    [RULE_SANITY['n_hits'] == 0, RULE_SANITY.index.isin(_high_touch) & (RULE_SANITY['ci_high'] < _s2)],
    ['no labelled hits: unvalidated', f'WARN: less risky than S2 ({_s2:.1%}) - review the trigger'], default='ok')
show(RULE_SANITY, 'Rule sanity on labelled observations (Wilson 95% CI; bad90_rest = observations without the rule)')
for _r, _c in RULE_SANITY['check'].items():
    if _c.startswith('WARN'):
        print(f'WARNING {_r}: {_c}')
RULE_X_GRADE = pd.concat({_r: _LAB.assign(hit=_RF[_r].to_numpy()).pivot_table(
    index='risk_grade', columns='hit', values='bad', aggfunc='mean', observed=True).reindex(GRADES)
    for _r in ALL_RULES if _RF[_r].any()}, names=['rule'])
if len(RULE_X_GRADE):
    RULE_X_GRADE.columns = [f'bad90 | hit={c}' for c in RULE_X_GRADE.columns]
    show(RULE_X_GRADE.loc[[r for r in ('R05_ENFORCEMENT', 'R04_SOCIAL_EXCLUSION', 'R07_FPD_FRAUD_CHECK')
                           if r in RULE_X_GRADE.index.get_level_values(0)]],
         'Rule x grade: bad rate inside each grade with / without the rule (why R04 / R05 are modifiers)')

_ax = _LAB[_LAB['rules_hit'] == '']
AXIS_CHECK = _ax.pivot_table(index='risk_grade', columns='engaged', values='bad', aggfunc='mean', observed=True)
AXIS_CHECK.columns = [f'bad90 | engaged={c}' for c in AXIS_CHECK.columns]
AXIS_CHECK['bad90 | capacity=yes'] = _ax[_ax['capacity'] == 'yes'].groupby('risk_grade')['bad'].mean()
AXIS_CHECK['bad90 | capacity=no'] = _ax[_ax['capacity'] == 'no'].groupby('risk_grade')['bad'].mean()
show(AXIS_CHECK.reindex(GRADES), 'Axis check: bad rate by grade x engagement / capacity (labelled, no rule hit)')

### 9.4 Behavioural clusters (profiling)
About 15 behavioural features go through median imputation and a normal `QuantileTransformer`, then KMeans with k = 4..8 chosen by silhouette on a sample of at most 20k. Each cluster is profiled and auto-named after its three largest z-score deviations. The cross-tab with the segments shows how the risk segments map onto behaviour. This is descriptive, not a second model.

In [ ]:
_CL_SPEC = {  # cluster feature -> (column in X, transform)
    'pd': (None, None), 'prior_episodes': ('dh_n_prior_episodes', None), 'log_income': ('pr_income_proxy', 'log'),
    'log_depo': ('pr_DEPO_SUM', 'log'), 'pkb_overdue': ('pr_SUM_PKB_OVRD_ACT', 'log'), 'logins_30d': ('geo_logins_30d', None),
    'enforcement_open': ('en_n_open', None), 'mob': ('ct_mob_at_obs', None), 'log_amount': ('ct_log_amount', None),
    'age': ('pr_AGE', None), 'digital_ratio': ('pr_DIGITAL_RATIO', None), 'contact_rate': ('cl_contact_rate_365d', None),
    'rejections_90d': ('rej_n_90d', None), 'capacity_ratio': ('capacity_ratio', 'log'), 'calls_365d': ('cl_n_calls_365d', None),
}
_cl = {}
for _k, (_c, _tf) in _CL_SPEC.items():
    if _c is None:
        _cl[_k] = OB['pd']
    elif _c in X.columns:
        _v = X[_c].astype('float64')
        _cl[_k] = np.log1p(_v.clip(lower=0)) if _tf == 'log' else _v
CL = pd.DataFrame(_cl, index=OB.index)
_miss = CL.isna().mean()
CL = CL.loc[:, _miss <= 0.6]
print(f"cluster features ({CL.shape[1]}): {list(CL.columns)}; dropped (>60% missing): {list(_miss[_miss > 0.6].index)}")

try:                                   # keep KMeans' OpenMP threads within N_JOBS on a shared server
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None
CLUSTER_PROFILE = None
if CL.shape[1] >= 2 and len(CL) >= 50:
    _Z = make_pipeline(SimpleImputer(strategy='median'),
                       QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(CL)),
                                           random_state=CONFIG['RANDOM_STATE'])).fit_transform(CL)
    _rng = np.random.default_rng(CONFIG['RANDOM_STATE'])
    _smp = _rng.choice(len(_Z), size=min(len(_Z), CONFIG['CLUSTER_SAMPLE']), replace=False)
    _sil = {}
    with (threadpool_limits(limits=CONFIG['N_JOBS']) if threadpool_limits else nullcontext()):
        for _k in range(CONFIG['CLUSTER_K_RANGE'][0], CONFIG['CLUSTER_K_RANGE'][1] + 1):
            _km = KMeans(n_clusters=_k, n_init=10, random_state=CONFIG['RANDOM_STATE']).fit(_Z[_smp])
            _sil[_k] = silhouette_score(_Z[_smp], _km.labels_, sample_size=min(len(_smp), 5000),
                                        random_state=CONFIG['RANDOM_STATE'])
        K_BEST = max(_sil, key=_sil.get)
        KM = KMeans(n_clusters=K_BEST, n_init=10, random_state=CONFIG['RANDOM_STATE']).fit(_Z[_smp])
        _lab_cl = KM.predict(_Z)
    _zc = pd.DataFrame(_Z, index=OB.index, columns=CL.columns).groupby(_lab_cl).mean()
    _zc = (_zc - _Z.mean(axis=0)) / _Z.std(axis=0)
    _names = {i: ' / '.join(f"{'high' if _zc.loc[i, f] > 0 else 'low'} {f}"
                            for f in _zc.loc[i].abs().sort_values(ascending=False).index[:3]) for i in _zc.index}
    OB['cluster'] = pd.Series([f'C{i}: {_names[i]}' for i in _lab_cl], index=OB.index, dtype='str')
    print('silhouette by k: ' + ', '.join(f'{k}: {v:.3f}' for k, v in _sil.items()) + f' -> k = {K_BEST}')
    _labd = OB[OB['labeled']]
    CLUSTER_PROFILE = pd.concat([
        OB.groupby('cluster').size().rename('n'),
        (OB.groupby('cluster').size() / len(OB)).rename('share'),
        _labd.groupby('cluster')['bad'].mean().rename('bad90'),
        _labd.groupby('cluster')['self_cure30'].mean().rename('self_cure'),
        CL.groupby(OB['cluster']).median().add_prefix('median_'),
    ], axis=1)
    show(CLUSTER_PROFILE, 'Behavioural clusters (n over all observations; bad90 / self-cure on labelled ones)')
    show(pd.crosstab(OB['cluster'], OB['segment']).reindex(columns=[s for s in SEGMENT_ORDER if s in set(OB['segment'])]),
         'Clusters x segments (all observations)')
    S['cluster'] = OB.loc[S.index, 'cluster']
else:
    OB['cluster'] = pd.Series(np.nan, index=OB.index, dtype='str')
    S['cluster'] = pd.Series(np.nan, index=S.index, dtype='str')
    print('clustering skipped (too few features / observations)')

### 9.5 Dealer FPD table and 9.6 monthly monitoring
- **Dealer table.** First-payment default is a lagging dealer-fraud signal (vendor data: more than 70% of early defaults show origination fraud). Here FPD means the **1st or 2nd instalment went unpaid to 30+ DPD** (instalment number from the opening month), so technical lates of a few days do not count. The table shows the FPD rate per `partner_name`, with a Wilson 95% confidence interval, for dealers with at least 30 contracts seasoned 3+ months, plus the bad90 rate of their labelled episodes.
- **Monitoring** (on new entries, i.e. the first checkpoint). Each month: score PSI against train (0.10 = investigate, 0.25 = act), the mature-cohort Gini against the valid Gini (a relative drop of 10% = amber, 20% = red), the roll31 early proxy, and the grade mix.

In [ ]:
_fe = OB.loc[FIRST].assign(instalment_no=X.loc[FIRST, 'dh_instalment_no'] if 'dh_instalment_no' in X.columns else np.nan)
_fpd_contracts = set(_fe.loc[_fe['instalment_no'].le(2) & _fe['max_dpd'].ge(30), 'contract_number'])
_dl = contracts[contracts['open_date'] <= DATA_CUT - pd.DateOffset(months=3)]
_dl = _dl.assign(dealer=as_cat(col(_dl, 'partner_name'), upper=False).fillna('<missing>'),
                 fpd=_dl['contract_number'].isin(_fpd_contracts).astype(int))
DEALERS = _dl.groupby('dealer').agg(n_contracts=('fpd', 'size'), n_fpd=('fpd', 'sum'))
DEALERS['fpd_rate'] = DEALERS['n_fpd'] / DEALERS['n_contracts']
DEALERS['fpd_ci_low'], DEALERS['fpd_ci_high'] = wilson(DEALERS['n_fpd'], DEALERS['n_contracts'])
_le = _fe[_fe['labeled']].join(_dl.set_index('contract_number')['dealer'], on='contract_number')
_b = _le.groupby('dealer').agg(n_labeled_eps=('bad', 'size'), n_bad=('bad', 'sum'))
DEALERS = DEALERS.join(_b)
DEALERS['bad90_rate'] = DEALERS['n_bad'] / DEALERS['n_labeled_eps']
DEALERS['bad90_ci_low'], DEALERS['bad90_ci_high'] = wilson(DEALERS['n_bad'], DEALERS['n_labeled_eps'])
DEALERS = DEALERS[DEALERS['n_contracts'] >= 30].sort_values('fpd_rate', ascending=False)
_port_fpd = _dl['fpd'].mean()
DEALERS['flag'] = np.where(DEALERS['fpd_ci_low'] > _port_fpd, 'FPD above portfolio (CI)', '')
show(DEALERS, f'Dealers: FPD 30+ rate (portfolio {_port_fpd:.1%}) and bad90 of labelled episodes (n >= 30 contracts)')

In [ ]:
_ref_gini = GINI.get((MAIN, 'valid'), np.nan)          # out-of-sample reference Gini of the main model
_base = OB.loc[IDS_FIRST['train'], 'pd']
_rows = []
for _m, _d in OB.loc[FIRST].groupby('obs_month'):
    _r = dict(obs_month=str(_m), n=len(_d), mean_pd=_d['pd'].mean(), psi_vs_train=psi(_base, _d['pd']),
              share_G4_G5=_d['risk_grade'].isin(['G4', 'G5']).mean())
    _l = _d[_d['labeled']]
    _r['labeled'] = len(_l)
    _r['bad_rate'] = _l['bad'].mean() if len(_l) else np.nan
    _r['gini'] = perf(_l['bad'], _l['pd'])['gini'] if _l['bad'].sum() >= 20 and (1 - _l['bad']).sum() >= 20 else np.nan
    _lr = _d[_d['labeled_roll']]
    _r['roll31_rate'] = _lr['roll31_60d'].mean() if len(_lr) else np.nan
    _rows.append(_r)
MONITORING = pd.DataFrame(_rows).set_index('obs_month')
MONITORING['psi_status'] = np.select([MONITORING['n'] < 100, MONITORING['psi_vs_train'] >= 0.25,
                                      MONITORING['psi_vs_train'] >= 0.10], ['low n', 'ACT', 'INVESTIGATE'], default='ok')
_drop_rel = 1 - MONITORING['gini'] / _ref_gini
MONITORING['gini_status'] = np.select([_drop_rel > 0.20, _drop_rel > 0.10, MONITORING['gini'].notna()],
                                      ['RED', 'AMBER', 'ok'], default='n/a')
show(MONITORING.tail(12), f'Monthly monitoring of new entries, last 12 months (reference: valid Gini {_ref_gini:.3f}, '
                          'train PD baseline; Gini needs >= 20 bads and goods)')

## 10. Charts
There are five charts, saved to `out/charts/*.png`. Each has one y-axis, a legend only where there are two series, and value labels only where they help.

In [ ]:
PAL = dict(s1='#2a78d6', s2='#eb6834', s3='#1baf7a', ink='#0b0b0b', ink2='#52514e', grid='#e4e3df', surface='#ffffff')
_RC_BEFORE = dict(plt.rcParams.copy())        # your kernel's matplotlib settings are restored at the end of this cell
plt.rcParams.update({
    'figure.facecolor': PAL['surface'], 'axes.facecolor': PAL['surface'], 'savefig.facecolor': PAL['surface'],
    'axes.edgecolor': PAL['grid'], 'axes.linewidth': 0.8, 'axes.labelcolor': PAL['ink2'], 'axes.titlecolor': PAL['ink'],
    'axes.titlesize': 11, 'axes.titlelocation': 'left', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': PAL['grid'], 'grid.linewidth': 0.6,
    'xtick.color': PAL['ink2'], 'ytick.color': PAL['ink2'], 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'text.color': PAL['ink'], 'font.size': 9, 'legend.frameon': False, 'legend.fontsize': 8.5, 'figure.dpi': 100,
})
CHARTS = {}


def _save(fig, name):
    fig.tight_layout()
    path = OUT / 'charts' / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    CHARTS[name] = path
    plt.show()
    plt.close(fig)


def _label_bars(ax, bars, fmt, horizontal=False, extra=None):
    for i, b in enumerate(bars):
        v = b.get_width() if horizontal else b.get_height()
        if not np.isfinite(v):
            continue
        xy = (v, b.get_y() + b.get_height() / 2) if horizontal else (b.get_x() + b.get_width() / 2, v)
        ax.annotate(fmt(v) + (extra[i] if extra is not None else ''), xy, xytext=(3, 0) if horizontal else (0, 2), textcoords='offset points',
                    ha='left' if horizontal else 'center', va='center' if horizontal else 'bottom',
                    color=PAL['ink2'], fontsize=8)


# (1) actual bad rate by risk grade: train vs OOT
fig, ax = plt.subplots(figsize=(7, 3.6))
_x = np.arange(len(GRADES))
for _i, (_s, _c, _lbl) in enumerate((('train', PAL['s1'], 'train (out-of-fold PD)'), ('oot', PAL['s2'], 'out-of-time'))):
    _v = GRADE_TABLES[_s]['bad_rate'].to_numpy(dtype='float64')
    _bars = ax.bar(_x + (_i - 0.5) * 0.36, _v, width=0.34, color=_c, label=_lbl)
    _label_bars(ax, _bars, lambda v: f'{v:.0%}')
ax.set_xticks(_x, GRADES)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis='x', visible=False)
ax.set_title('Actual 90+ DPD rate within 6 months, by risk grade')
ax.legend(loc='upper left')
_save(fig, '1_bad_rate_by_grade')

# (2) segment size - current stock (falls back to history if the stock is empty)
_sz = S['segment'].value_counts() if len(S) else OB['segment'].value_counts()
_sz = _sz.reindex([s for s in SEGMENT_ORDER if s in _sz.index])[::-1]
fig, ax = plt.subplots(figsize=(7, 0.32 * len(_sz) + 1.2))
_bars = ax.barh(_sz.index, _sz.to_numpy(), height=0.6, color=PAL['s1'])
_label_bars(ax, _bars, lambda v: f'{v:,.0f}', horizontal=True)
ax.grid(axis='y', visible=False)
ax.set_title(f"Segment size: {'current stock' if len(S) else 'all episodes'} (contracts)")
_save(fig, '2_segment_size')

# (3) bad rate by segment - labelled history
_bt = BACKTEST.loc['all_labeled'][::-1]
_br = _bt['bad90']
fig, ax = plt.subplots(figsize=(7, 0.32 * len(_br) + 1.2))
_bars = ax.barh(_br.index, _br.to_numpy(dtype='float64'), height=0.6, color=PAL['s1'])
_label_bars(ax, _bars, lambda v: f'{v:.0%}', horizontal=True, extra=[f'  n={n:,.0f}' for n in _bt['n']])
ax.set_xlim(0, min(1.0, float(np.nanmax(_br.to_numpy(dtype='float64')))) * 1.18 + 0.02)
ax.set_xticks([t for t in ax.get_xticks() if 0 <= t <= 1.0 + 1e-9])
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis='y', visible=False)
ax.set_title('Actual 90+ DPD rate by segment (labelled history)')
_save(fig, '3_bad_rate_by_segment')

# (4) top-20 feature importance
_imp = IMP_TABLE['gain_share'].head(20)[::-1]
fig, ax = plt.subplots(figsize=(7, 0.28 * len(_imp) + 1.2))
ax.barh(_imp.index, _imp.to_numpy(), height=0.6, color=PAL['s1'])
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis='y', visible=False)
ax.set_title(f'Top-20 features: share of total gain ({MODEL_KIND})')
_save(fig, '4_feature_importance')

# (5) monthly entries and bad rate - two small multiples, one axis each
_mt = MONTHLY.copy()
_mt.index = _mt.index.to_timestamp()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.8), sharex=True)
ax1.bar(_mt.index, _mt['entries'], width=20, color=PAL['s1'])
ax1.grid(axis='x', visible=False)
ax1.set_title('Entries into delinquency per month')
_lb = _mt['bad_rate'].dropna()
ax2.plot(_lb.index, _lb.to_numpy(), color=PAL['s1'], linewidth=2, solid_capstyle='round')
ax2.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax2.set_ylim(bottom=0)
ax2.set_title('Bad rate of labelled entries (90+ DPD within 6 months)')
_save(fig, '5_monthly_entries_bad_rate')
dict.update(plt.rcParams, _RC_BEFORE)

## 11. Outputs
All files go to `OUT_DIR` (`out/`; DEMO runs `out_demo/`; git ignores both because the files contain IINs):
- `segments_current_stock.parquet` + `.xlsx`: one row per stock contract, with `run_mode`, PD on the current day in arrears and at entry, grade, segment and reason, today's treatment (`treatment_now`, `operator_due_now`), modifiers and additional actions, challenger arm, contact caps, compliance dates and flags, and top SHAP drivers in business terms. The workbook has a `legend` sheet with the meaning of every column.
- `segments_history.parquet`: labelled observations (every checkpoint) with PD, grade, segment, cluster and the outcomes.
- `segment_report.xlsx`: data_quality, block_coverage, model_metrics, metrics_by_checkpoint, grades, grades_by_checkpoint, segments_backtest, backtest_by_checkpoint, segment_tests, rule_sanity, rule_x_grade, segments_current, clusters, feature_importance, psi_dropped, psi_stock, monthly, checkpoints, dealers, monitoring, treatments, pkb_rule_map, load_log, block_failures.
- `model/`: the LightGBM text model, the pickled bundle (model, LR, features, category levels, cutpoints, config), and features / config / cutpoints JSON.
- `charts/*.png`, `run_summary.json`.

In [ ]:
STOCK_COLUMNS = {   # column -> meaning (also written as the 'legend' sheet of the stock workbook)
    'run_mode': 'real or demo (demo = synthetic people, never use operationally)', 'data_cut': 'last act_date in ovrd',
    'score_date': 'date the segment applies to (cut + 1 day)', 'contract_number': 'contract', 'process_ref': 'process ref',
    'iin': 'client IIN', 'entry_date': 'first overdue day of the current episode', 'dpd_now': 'DPD on the cut (ovrd)',
    'dpd_asof': 'act_date of dpd_now', 'days_in_arrears': 'days since entry_date (= scoring checkpoint)',
    'pd': 'PD (90+ DPD within 6 months) on the current day in arrears',
    'pd_at_entry': 'PD of the same episode on its first overdue day', 'risk_grade': 'G1 (lowest) .. G5 (highest PD)',
    'segment': 'hard rule R0x or model segment S1-S6', 'segment_reason': 'why this segment',
    'treatment': 'segment playbook', 'treatment_now': 'what to do today: the next stage once the digital-only window is '
                                                      'over, plus R04/R05 modifier actions, which win where they conflict',
    'playbook_day': 'day of the playbook = days in arrears', 'operator_from_day': 'first operator contact day of the segment',
    'operator_due_now': '1 = operator contact is due now', 'channels': 'channels of the segment',
    'contact_caps': 'statutory call / visit limits (Rules No. 136)', 'arm': 'champion / challenger / none (no challenger)',
    'challenger_treatment': 'treatment of the challenger arm', 'modifiers': 'R04 / R05 modifiers that apply',
    'additional_actions': 'actions of every other rule hit (kept on top of the segment)',
    'rules_hit': 'every rule that fired', 'model_segment': 'S1-S6 the contract would get without hard rules',
    'validation': 'backtested, or unvalidated (PKB overlay, no labelled history)',
    'legal_review_flag': '1 = PKB court status (possibly another creditor) - legal review, contact continues',
    'exclusion_review_flag': '1 = on an exclusion list with an unrecognised title - review',
    'capacity': 'yes / no / unknown (capacity_ratio >= 1.5 or depo_cover >= 1)', 'engaged': '1 = reachable / active',
    'capacity_ratio': 'capacity income / monthly payment', 'depo_cover': 'deposits / monthly payment',
    'debitable_funds': 'KZT that may legally be debited (50% / 2xПМ rule)', 'logins_30d': 'Homebank logins, last 30 days',
    'contact_rate_365d': 'share of calls with talk time, last 365 days', 'enforcement_open': 'open ИР3/ИР4 orders (rule window)',
    'no_credit_offer': '1 = do not offer credit (Стоп-Кредит / Лудоман / > 30 days in arrears)',
    'third_party_contact_flag': '1 = CRM shows contact with third parties (compliance review)',
    'notify_due_date': 'statutory notice deadline (entry + 10 days)', 'notify_days_left': 'days left until the notice deadline',
    'notify_overdue': '1 = notice deadline passed: check that the notice went out',
    'restructuring_window_end': 'end of the 30-day change-of-terms application window',
    'cluster': 'behavioural cluster (profiling)', 'top_drivers': 'features pushing the PD up the most (TreeSHAP, log-odds)',
}
STOCK_COLS = list(STOCK_COLUMNS)
STOCK_OUT = S.reset_index()[['episode_id'] + [c for c in STOCK_COLS if c in S.columns]].sort_values('pd', ascending=False)
STOCK_LEGEND = pd.DataFrame({'column': list(STOCK_COLUMNS), 'meaning': list(STOCK_COLUMNS.values())})
HIST_COLS = ['contract_number', 'entry_date', 'checkpoint', 'days_in_arrears', 'obs_date', 'obs_month', 'sample', 'pd',
             'pd_lr', 'risk_grade', 'segment', 'model_segment', 'rules_hit', 'modifiers', 'capacity', 'engaged', 'cluster',
             'bad', 'self_cure30', 'roll31_60d', 'labeled_roll', 'max_dpd', 'reaged']
HIST_OUT = (OB[OB['labeled']].reset_index()[['obs_id', 'episode_id'] + [c for c in HIST_COLS if c in OB.columns]]
            .merge(contracts[['contract_number', 'iin', 'process_ref']], on='contract_number', how='left')
            .assign(obs_month=lambda d: d['obs_month'].astype('str')))

STOCK_OUT.to_parquet(OUT / 'segments_current_stock.parquet', index=False)
HIST_OUT.to_parquet(OUT / 'segments_history.parquet', index=False)


def _xl(df, index=True):
    """Excel-safe copy: index as columns, periods / categories as text."""
    d = df.reset_index() if index else df.copy()
    for c in d.columns:
        if isinstance(d[c].dtype, (pd.PeriodDtype, pd.CategoricalDtype)):
            d[c] = d[c].astype('str')
    return d


def write_excel(path, sheets):
    try:
        with pd.ExcelWriter(path, engine='openpyxl') as xw:
            for name, (df, idx) in sheets.items():
                if df is not None:
                    _xl(df, idx).to_excel(xw, sheet_name=name[:31], index=False)
        log(f'written {path}')
    except Exception as e:      # Excel is a convenience copy; parquet files are the primary output
        print(f'WARN  Excel export {path} failed ({type(e).__name__}: {e})')


write_excel(OUT / 'segments_current_stock.xlsx', {('current_stock' if RUN_MODE == 'real' else 'current_stock_DEMO'): (STOCK_OUT, False),
                                                   'legend': (STOCK_LEGEND, False)})
TREATMENTS = pd.DataFrame(SEGMENT_TREATMENT).T.rename_axis('segment')
write_excel(OUT / 'segment_report.xlsx', {
    'data_quality': (DQ_REPORT, True), 'block_coverage': (BLOCK_COVERAGE, True), 'model_metrics': (METRICS, False),
    'metrics_by_checkpoint': (METRICS_CP, False), 'grades': (GRADES_TBL, True), 'grades_by_checkpoint': (GRADE_BY_CP, True),
    'segments_backtest': (BACKTEST, True), 'backtest_by_checkpoint': (BACKTEST_BY_CP, True),
    'segment_tests': (SEGMENT_TESTS, False), 'rule_sanity': (RULE_SANITY, True), 'rule_x_grade': (RULE_X_GRADE, True),
    'segments_current': (SEG_CURRENT, True), 'clusters': (CLUSTER_PROFILE, True), 'feature_importance': (IMP_TABLE, True),
    'psi_dropped': (DROPPED, False), 'psi_stock': (STOCK_PSI, True), 'monthly': (MONTHLY, True),
    'checkpoints': (CHECKPOINT_TABLE, True), 'dealers': (DEALERS, True), 'monitoring': (MONITORING, True),
    'treatments': (TREATMENTS, True), 'pkb_rule_map': (PKB_RULE_MAP, False), 'load_log': (pd.DataFrame(LOAD_LOG), False),
    'block_failures': (pd.DataFrame(BLOCK_FAILURES, columns=['block', 'run', 'error']), False),
})

_gini = METRICS.set_index(['model', 'sample'])['gini']
_fl = OB.loc[FIRST]
_gcp = (METRICS_CP.set_index(['model', 'sample', 'checkpoint'])['gini'] if len(METRICS_CP) else pd.Series(dtype='float64'))
RUN_SUMMARY = dict(
    run_mode=RUN_MODE, data_cut=str(DATA_CUT.date()), score_date=str(SCORE_DATE.date()), out_dir=str(OUT),
    runtime_seconds=round(time.perf_counter() - T0, 1), model=MODEL_KIND, checkpoint_days=CHECKPOINTS,
    episodes=int(len(_fl)), episodes_labeled=int(_fl['labeled'].sum()),
    bad_rate_labeled_entry=round(float(_fl.loc[_fl['labeled'], 'bad'].mean()), 4),
    reaged_episodes=int(_fl['reaged'].sum()), observations=int((~OB['is_stock']).sum()), labeled=int(OB['labeled'].sum()),
    n_features=len(FEATURES), model_blocks=MODEL_BLOCKS_USED,
    gini={f'{m}/{s}': round(float(v), 4) for (m, s), v in _gini.items()},
    gini_by_checkpoint={f'{m}/{s}/day{k}': round(float(v), 4) for (m, s, k), v in _gcp.items()},
    score_psi_train_oot=round(SCORE_PSI, 4), stock_drift=STOCK_DRIFT,
    grade_monotone=MONOTONE, pd_cutpoints=[round(float(c), 4) for c in CUTS],
    grades_oot=GRADE_TABLES['oot'][['n', 'avg_pd', 'bad_rate']].round(4).reset_index().to_dict('records'),
    segments_stock=S['segment'].value_counts().to_dict(),
    stock_operator_due_now=int(S['operator_due_now'].sum()) if len(S) else 0,
    stock_unvalidated=int(S['validation'].str.startswith('unvalidated').sum()) if len(S) else 0,
    segments_backtest=BACKTEST.loc['all_labeled', ['n', 'bad90', 'bad90_ci_low', 'bad90_ci_high', 'self_cure']]
                      .round(4).reset_index().to_dict('records'),
    rule_warnings={r: c for r, c in RULE_SANITY['check'].items() if c != 'ok'},
    asof_test_pass=ASOF_OK, block_failures=BLOCK_FAILURES,
    stock_size=int(len(S)), dropped_tables=CONFIG['DROP_TABLES'],
    outputs=sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()),
)
_ = (OUT / 'run_summary.json').write_text(json.dumps(RUN_SUMMARY, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
show(pd.DataFrame([(str(p.relative_to(OUT)), f'{p.stat().st_size / 1024:,.0f} KB') for p in sorted(OUT.rglob('*')) if p.is_file()],
                  columns=['file', 'size']).set_index('file'), f'Files in {OUT.resolve()}')

## 12. Next steps and enrichment
- **Add a source.** Load it under a table name, add its columns and keys to `TABLE_COLUMNS` / `TABLE_KEYS`, and write a preparation step that normalises keys and parses dates. Then write `fb_new(obs, ctx)` using `events_before(...)`, `n_within(...)` and `finish(...)`, and register it in `BLOCKS` with `role='auto'`. The `MODEL_BLOCKS='auto'` rule decides whether its history is long enough for the model.
- **PKB monitoring.** It has been accumulating since 2026-09-07. Run `sql/hes_pkb_monitoring_agg.sql` monthly for a series of snapshot dates (month-ends) and store each snapshot. Once 6 months of labelled entries have snapshots before their `obs_date`, switch the block from `overlay` to `auto` and give it an as-of join on the snapshot date; then R08, R09 and the PKB part of R02 can be backtested and lose their `unvalidated` mark. The same applies to geo, vnpt and vneshpt as their history grows.
- **dm allow-list.** Review the dm columns printed in section 3 and put the application-time ones into `CONFIG['DM_ALLOW_COLUMNS']`.
- **Instalment.** Replace the annuity proxy (amount, earliest `end_date`, `INSTALMENT_RATE`) with the contractual payment from the schedule or the application (`amount_duration`, rate) once available; then the capacity ratios can enter the model.
- **Monitoring** (section 9.6): run it monthly. Score PSI above 0.10 means investigate and above 0.25 means act. Also track per-feature CSI for the top-20 features. Gini on matured cohorts (6 months) is amber at a 10% relative drop and red at 20%. Watch roll31_60d as the early proxy, grade monotonicity and the promise-kept rate.
- **Champion-challenger.** Each segment routes 15% of contracts by contract hash (`arm`) to the challenger treatment in `SEGMENT_TREATMENT`. Run 2-3 cycles and compare cure, roll and cost; promote per segment. The first test is S1 with no operator until day 10. This is also what breaks the treatment contamination in the history.
- **Retrain** quarterly, or when PSI or Gini goes red. Keep the time-split OOT. Re-derive the grade cutpoints and re-check monotonicity.
- **Data asks.**
  - `contr` including closed, written-off and sold contracts, to remove survivorship bias.
  - The real ovrd semantics for the last month (is `act_date` the cut date for every row?).
  - The IIN and timestamp column names in `dds.hes_pkb_monitoring_event`.
  - Daily DPD instead of monthly maxima: exact cure dates, partial payments and checkpoints on any day.

## 13. Appendix: Vertica SQL for the PKB monitoring aggregate
This is the same text as `sql/hes_pkb_monitoring_agg.sql`. It produces long format, one row per (iin, rule), with 7/30/60-day and total counts as of `:snapshot_date`. **The IIN and event-timestamp column names are placeholders** (`*__PLACEHOLDER`); replace them before running.

The aggregate must exist **before section 3** loads the data, otherwise R08, R09, the PKB part of R02 and the «Стоп-Кредит» flag are off for this run. Either run the SQL beforehand and save `DATA_DIR/hes_pkb_agg.parquet` (or keep a `hes_pkb_agg` DataFrame in the kernel), or set `CONFIG['RUN_PKB_SQL'] = True` in your Vertica kernel: section 3 then runs this SQL through your `query()` helper first. The cell below does the same by hand (guarded by `RUN_PKB_SQL = False`); after it, **re-run the notebook from section 3**.

In [ ]:
PKB_SQL = r"""
/* =====================================================================================================
   hes_pkb_monitoring_agg.sql  (Vertica)

   PKB (Первое кредитное бюро) monitoring events -> per client (IIN) x rule aggregates, LONG format.

   Source : dds.hes_pkb_monitoring_event   (~110M rows, ~1.9M IINs, events since 2026-09-07)
   Output : one row per (iin, rule_key, rule_name):
              iin            12-char IIN, leading zeros kept
              rule_key       e.g. 'CustomRule.CMC'
              rule_name      e.g. 'Мониторинг статуса «Лудоман»' (the notebook maps names -> codes)
              cnt_7d         events in [snapshot_date - 7 days,  snapshot_date)
              cnt_30d        events in [snapshot_date - 30 days, snapshot_date)
              cnt_60d        events in [snapshot_date - 60 days, snapshot_date)
              cnt_all        all events before snapshot_date (scan bounded to 400 days)
              first_ts       first event time
              last_ts        last event time
              snapshot_date  the parameter (the aggregate is "as of" this date, no look-ahead)
   Use    : save the result as parquet  ->  CONFIG['PKB_AGG_FILE'] (default DATA_DIR/hes_pkb_agg.parquet).
            The notebook block fb_pkb_monitoring uses it for OVERLAY rules only (history is too short to
            train on); once 6+ months have accumulated, build monthly snapshots and move it into the model.

   Parameter
     :snapshot_date   DATE literal, e.g. '2026-11-18'. Only events strictly BEFORE this date are counted.
                      Python: sql.replace(':snapshot_date', "'2026-11-18'")
                      vsql  : \set snapshot_date '''2026-11-18'''

   >>> PLACEHOLDERS: the IIN and event-timestamp column names were not visible in the screenshots. <<<
     iin_col__PLACEHOLDER       -> the client IIN column        (check: SELECT * FROM dds.hes_pkb_monitoring_event LIMIT 5;)
     event_ts_col__PLACEHOLDER  -> the event date-time column   (the one spanning 2026-09-07 .. today)
   Replace every occurrence (search for "__PLACEHOLDER") before running.
   ===================================================================================================== */
WITH params AS (
    SELECT CAST(:snapshot_date AS TIMESTAMP) AS snap_ts
),
ev AS (
    SELECT
        -- PLACEHOLDER column: client IIN -> digits only, left-padded to 12 characters
        LPAD(REGEXP_REPLACE(CAST(e.iin_col__PLACEHOLDER AS VARCHAR(32)), '[^0-9]', ''), 12, '0') AS iin,
        -- PLACEHOLDER column: event timestamp
        CAST(e.event_ts_col__PLACEHOLDER AS TIMESTAMP)                                         AS event_ts,
        e.rule_key,
        e.rule_name
    FROM dds.hes_pkb_monitoring_event e
    CROSS JOIN params p
    WHERE e.event_ts_col__PLACEHOLDER <  p.snap_ts                         -- PLACEHOLDER column: strictly before the snapshot
      AND e.event_ts_col__PLACEHOLDER >= p.snap_ts - INTERVAL '400 days'   -- PLACEHOLDER column: bounded scan
      AND e.rule_name NOT LIKE 'Отписка%'                                  -- technical rule (unsubscribe), not a client event
      -- Optional: keep only the auto-loan portfolio to shrink the output
      -- AND e.iin_col__PLACEHOLDER IN (SELECT iin FROM <your_portfolio_table>)
)
SELECT
    ev.iin,
    ev.rule_key,
    ev.rule_name,
    SUM(CASE WHEN ev.event_ts >= p.snap_ts - INTERVAL '7 days'  THEN 1 ELSE 0 END) AS cnt_7d,
    SUM(CASE WHEN ev.event_ts >= p.snap_ts - INTERVAL '30 days' THEN 1 ELSE 0 END) AS cnt_30d,
    SUM(CASE WHEN ev.event_ts >= p.snap_ts - INTERVAL '60 days' THEN 1 ELSE 0 END) AS cnt_60d,
    COUNT(*)                                                                      AS cnt_all,
    MIN(ev.event_ts)                                                              AS first_ts,
    MAX(ev.event_ts)                                                              AS last_ts,
    CAST(p.snap_ts AS DATE)                                                       AS snapshot_date
FROM ev
CROSS JOIN params p
GROUP BY ev.iin, ev.rule_key, ev.rule_name, p.snap_ts
;
"""
print(PKB_SQL[:600] + "\n...")

In [ ]:
RUN_PKB_SQL = False     # set True in your Vertica-connected kernel (needs your query() helper)
if RUN_PKB_SQL:
    _snap = pd.Timestamp.today().normalize()
    _sql = PKB_SQL.replace(':snapshot_date', f"'{_snap:%Y-%m-%d}'")
    assert '__PLACEHOLDER' not in _sql, 'replace the placeholder column names in the SQL first'
    hes_pkb_agg = query(_sql)                                 # noqa: F821 - your existing helper -> DataFrame
    _path = nb_path(CONFIG['DATA_DIR']) / CONFIG['PKB_AGG_FILE']
    _path.parent.mkdir(parents=True, exist_ok=True)
    hes_pkb_agg.to_parquet(_path, index=False)
    print(f'saved {hes_pkb_agg.shape} -> {_path}; kept as hes_pkb_agg in this kernel -> re-run from section 3')
else:
    print('RUN_PKB_SQL = False -> skipped (set True in the Vertica kernel after replacing the placeholders)')

In [ ]:
print(json.dumps({k: RUN_SUMMARY[k] for k in ('run_mode', 'data_cut', 'episodes', 'observations', 'labeled',
                                               'bad_rate_labeled_entry', 'model', 'gini', 'grade_monotone', 'stock_size',
                                               'segments_stock', 'block_failures')},
                 ensure_ascii=False, indent=1, default=str))
if BLOCK_FAILURES:
    print(f'WARNING {len(BLOCK_FAILURES)} feature block run(s) FAILED - their features and rules are missing: '
          + ', '.join(f"{f['block']} ({f['run']})" for f in BLOCK_FAILURES), file=sys.stderr)
for _k, _v in _PD_OPTIONS_BEFORE.items():     # restore your kernel's pandas display options
    pd.set_option(_k, _v)
log('done')